# [1.4.2] SAE Circuits (연습 문제)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/22_[1.4.2]_SAE_Circuits)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part42_sae_circuits/1.4.2_SAE_Circuits_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part42_sae_circuits/1.4.2_SAE_Circuits_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널로 보내주시고, 본 장의 학습 내용에 관한 질문은 전용 채널에서 해주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어서 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 이동하는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-13-2.png" width="350">
<br>

# 소개

이 연습 문제들에서는 **SAE를 이용한 circuit**을 탐구합니다. 이는 transformer의 서로 다른 layer에 있는 SAE latent들의 집합으로, 서로 통신하며 모델의 특정 동작을 end-to-end 방식으로 설명합니다. 먼저 latent-to-latent, token-to-latent, 그리고 latent-to-logit gradient를 계산하는 것부터 시작하며, 이를 통해 서로 다른 layer의 latent들이 어떻게 연결되어 있는지에 대한 선형 proxy를 얻을 수 있습니다. 그 다음으로는 **transcoder**로 넘어가겠습니다. transcoder는 SAE의 변형으로, 단순히 activation을 재구성하는 것이 아니라 모델 layer의 연산 자체를 재구성하도록 학습하며, circuit 분석에 상당한 이점을 제공합니다.

이 연습 문제들을 수행하기 위해서는 어느 정도의 선수 지식이 필요합니다. 특히 다음 내용을 이해하고 계신다면 매우 도움이 될 것입니다:

- **superposition**이 무엇인지, 그리고 **sparse autoencoder** 구조가 무엇인지 (이 주제들에 대한 복습이 필요하시다면, 연습 문제 **1.5.4 Toy Models of SAEs & Superposition**을 참고하십시오)
- **SAELens** 라이브러리를 사용하여 TransformerLens 모델과 함께 SAE를 로드하고 실행하는 방법 (연습 문제 **1.3.3 Interpretability with SAEs**의 첫 번째 섹션에서 다룹니다)

연습 문제 세트 1.3.3의 가장 관련성 높은 배경 지식(주로 모델 및 SAE 로드, 그리고 이를 이용한 forward pass 실행)을 빠르게 훑어볼 수 있도록 시작 부분에 짧은 섹션을 포함했습니다. 따라서 이 연습 문제를 시작하기 전에 해당 세트를 모두 완료하실 필요는 없습니다.

용어에 대한 한 가지 참고 사항입니다: 저희는 주로 **feature**를 베이스 모델이 학습한 기본 데이터 분포의 특성으로, **SAE latent**(또는 단순히 "latent")를 SAE 내의 방향으로 정의하는 관례를 따를 것입니다. 이는 "feature"라는 용어의 과부하를 피하고, "SAE feature"가 데이터의 실제 feature와 반드시 일치한다는 암묵적인 가정을 피하기 위함입니다. 다만, SAE latent가 데이터의 특정 해석 가능한 feature와 매우 명확하게 일치하는 경우에는 이 용어 구분을 완화하여 사용하겠습니다.

## 읽기 자료

- [Towards Monosemanticity: Decomposing Language Models With Dictionary Learning](https://transformer-circuits.pub/2023/monosemantic-features/index.html) 은 SAE를 사용하여 mechanistic interpretability 분야에서 첫 번째 큰 진전을 이룬 것으로 평가받습니다. 이들은 1-layer 모델에서 SAE를 훈련시키고, 다수의 해석 가능한 feature들을 추출했습니다.
- [Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html) 은 SAE 연구를 더 큰 모델로 어떻게 확장할 수 있는지 보여줍니다.
- [Transcoders Find Interpretable LLM Feature Circuits](https://arxiv.org/abs/2406.11944) (Dunefsky et al., 2024)는 transcoder를 소개합니다. 이는 단순히 activation을 재구성하는 대신 MLP 입력을 MLP 출력으로 매핑하는 법을 배우는 SAE의 변형으로, circuit 분석에 매우 적합합니다. 이는 본 실습 세트의 섹션 2에서 사용되는 주요 기술입니다. 최소한 초록과 섹션 2("Methods")를 읽어보시기 바랍니다.
- [Circuit Tracing: Revealing Computational Graphs in Language Models](https://transformer-circuits.pub/2025/attribution-graphs/methods.html) (Anthropic, 2025)는 transcoder를 사용하여 모델 계산을 해석 가능한 계산 그래프로 분해하는 attribution graph 프레임워크를 소개합니다. 본 실습 세트의 섹션 3에서는 이를 처음부터 구현합니다. 여러분이 구현하게 될 기술적 세부 사항은 "Methods" 섹션을 읽어보시기 바랍니다.
- [On the Biology of a Large Language Model](https://transformer-circuits.pub/2025/attribution-graphs/biology.html) 는 attribution graph를 적용하여 Claude의 다양한 현상을 연구한 동반 논문입니다. 동기 부여를 확인하고 attribution graph가 어떤 종류의 circuit을 밝혀낼 수 있는지 살펴보기 위해 훑어보시기 바랍니다.

## 내용 및 학습 목표

### 1️⃣ Latent Gradients

SAE는 매우 멋지고 흥미로우며, latent를 조절하여 흥미로운 효과를 낼 수 있습니다. 하지만 이것이 우리가 모델이 사용하는 진정한 계산 단위를 정말로 찾아낸 것인지, 아니면 단순히 흥미로운 클러스터링 알고리즘을 발견한 것뿐인지 알 수 있을까요? 정답은 아직 정확히 모른다는 것입니다! 전자를 뒷받침하는 강력한 증거는 **SAE를 이용한 circuit**를 찾는 것입니다. 즉, transformer의 서로 다른 layer에 있는 latent들의 집합이 서로 통신하며, 특정 동작을 end-to-end 방식으로 설명하는 것을 찾는 것입니다. 이 섹션에서는 서로 다른 layer의 latent 간의 gradient를 계산하여 이들이 어떻게 통신하는지에 대한 그림을 그려보겠습니다.

> ##### 학습 목표
>
> - transformer의 서로 다른 layer에 있는 SAE latent 간의 **latent-to-latent gradients**를 계산하는 방법을 배웁니다.
> - **token-to-latent gradients**를 계산하여 어떤 입력 token이 특정 latent activation을 유도하는지 이해합니다.
> - **latent-to-logit gradients**를 계산하여 latent가 모델의 output에 어떻게 영향을 미치는지 이해합니다.
> - 이러한 gradient 기반 방법들을 사용하여 attention SAE에서 circuit(예: induction circuits)를 찾습니다.

### 2️⃣ Transcoders

**Transcoder**는 단순히 한 지점의 activation을 재구성하는 것이 아니라, 모델 layer의 계산(예: MLP input에서 MLP output으로의 sparse mapping)을 재구성하도록 학습하는 SAE의 변형입니다. Transcoder는 MLP layer의 함수를 sparse하고 해석 가능한 단위로 분해하기 때문에 circuit 분석에 상당한 이점을 제공합니다. 이 섹션에서는 transcoder를 로드하여 사용하고, de-embedding과 같은 기술을 통해 그 특성을 연구하며, 가중치 기반 분석만으로 transcoder latent를 역공학하는 blind case study를 진행하겠습니다.

> ##### 학습 목표
>
> - transcoder의 개념과 표준 SAE와의 차이점을 이해합니다.
> - transcoder latent를 해석하기 위한 기술인 **pullbacks**, **de-embeddings**, **extended embeddings**를 배웁니다.
> - **blind case study**를 통해 (activation 예시 없이) 오직 circuit 수준의 분석만으로 transcoder latent를 해석해 봅니다.

### 3️⃣ Attribution graphs

Attribution graph는 섹션 1️⃣의 gradient 기반 방법들을 확장하여, transcoder latent를 통해 transformer의 end-to-end 계산을 이해하기 위한 전체 프레임워크를 제공합니다. 이 섹션에서는 Gemma 3-1B IT와 GemmaScope 2 transcoder를 사용하여 attribution graph 파이프라인 전체를 처음부터 구현합니다. 여기에는 non-linearity를 동결하여 모델을 선형화하고, 모든 노드 타입에 대해 reading/writing vector 추상화를 구축하며, batched backward pass를 통해 edge weight를 계산하고, influence 기반의 Neumann series propagation을 사용하여 그래프를 pruning하는 과정이 포함됩니다.

> ##### 학습 목표
>
> - local replacement model을 이해합니다: attention pattern과 LayerNorm scale을 동결하고 MLP를 linear skip connection으로 대체했을 때 왜 residual stream이 선형이 되는지 배웁니다.
> - reading/writing vector 추상화를 이해합니다: token embedding, transcoder latent, MLP error, logit direction이 모두 residual stream과 어떻게 상호작용하는지 배웁니다.
> - 핵심 attribution 알고리즘을 구현합니다: salient logit 선택, 그래프 노드 구축, gradient injection을 통한 edge weight 계산을 구현합니다.
> - nilpotent adjacency matrix에 Neumann series를 사용하여, 노드 및 엣지 influence 임계값 기반의 그래프 pruning을 구현합니다.
> - Anthropic의 공개 연구와 동일한 대시보드 템플릿을 사용하여 대화형 attribution graph 시각화를 구축합니다.

### 4️⃣ Exploring circuits & interventions

이제 attribution graph 알고리즘을 처음부터 구축했으므로, `circuit-tracer` 라이브러리를 사용하여 실제 circuit를 탐색하고 feature intervention을 수행하겠습니다. Dallas/Austin two-hop factual recall circuit를 연구하고, zero ablation을 통해 인과 구조를 테스트하며, prompt 간에 feature를 교체하고, feature intervention을 통해 텍스트를 생성해 보겠습니다.

> ##### 학습 목표
>
> - 미리 계산된 attribution graph와 그 supernode들을 로드하고 검사합니다.
> - zero ablation 실험을 수행하여 그래프가 제시하는 인과적 주장을 테스트합니다.
> - cross-prompt feature swapping을 수행하여 compositional circuit 구조를 입증합니다.
> - feature intervention을 이용한 open-ended generation을 수행합니다.

## A note on memory usage

In these exercises, we'll be loading some pretty large models into memory (e.g. Gemma 2-2B and its SAEs, as well as a host of other models in later sections of the material). It's useful to have functions which can help profile memory usage for you, so that if you encounter OOM errors you can try and clear out unnecessary models. For example, we've found that with the right memory handling (i.e. deleting models and objects when you're not using them any more) it should be possible to run all the exercises in this material on a Colab Pro notebook, and all the exercises minus the handful involving Gemma on a free Colab notebook.

<details>
<summary>See this dropdown for some functions which you might find helpful, and how to use them.</summary>

First, we can run some code to inspect our current memory usage. Here's an example of running this code on a Colab Pro notebook.

```python
import part42_sae_circuits.utils as utils

# Profile memory usage, and delete gemma models if we've loaded them in
namespace = globals().copy() | locals()
utils.profile_pytorch_memory(namespace=namespace, filter_device="cuda:0")
```

<pre style="font-family: Consolas; font-size: 14px">Allocated = 35.88 GB
Total = 39.56 GB
Free = 3.68 GB
┌──────────────────────┬────────────────────────┬──────────┬─────────────┐
│ Name                 │ Object                 │ Device   │   Size (GB) │
├──────────────────────┼────────────────────────┼──────────┼─────────────┤
│ gemma_2_2b           │ HookedSAETransformer   │ cuda:0   │       11.94 │
│ gpt2                 │ HookedSAETransformer   │ cuda:0   │        0.61 │
│ gemma_2_2b_sae       │ SAE                    │ cuda:0   │        0.28 │
│ sae_resid_dirs       │ Tensor (4, 24576, 768) │ cuda:0   │        0.28 │
│ gpt2_sae             │ SAE                    │ cuda:0   │        0.14 │
│ logits               │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ logits_with_ablation │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ clean_logits         │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ _                    │ Tensor (16, 128, 768)  │ cuda:0   │        0.01 │
│ clean_sae_acts_post  │ Tensor (4, 15, 24576)  │ cuda:0   │        0.01 │
└──────────────────────┴────────────────────────┴──────────┴─────────────┘</pre>

From this, we see that we've allocated a lot of memory for the the Gemma model, so let's delete it. We'll also run some code to move any remaining objects on the GPU which are larger than 100MB to the CPU, and print the memory status again.

```python
del gemma_2_2b
del gemma_2_2b_sae

THRESHOLD = 0.1  # GB
for obj in gc.get_objects():
    try:
        if isinstance(obj, t.nn.Module) and utils.get_tensors_size(obj) / 1024**3 > THRESHOLD:
            if hasattr(obj, "cuda"):
                obj.cpu()
            if hasattr(obj, "reset"):
                obj.reset()
    except Exception:
        pass

# Move our gpt2 model & SAEs back to GPU (we'll need them for the exercises we're about to do)
gpt2.to(device)
gpt2_saes = {layer: sae.to(device) for layer, sae in gpt2_saes.items()}

utils.print_memory_status()
```

<pre style="font-family: Consolas; font-size: 14px">Allocated = 14.90 GB
Reserved = 39.56 GB
Free = 24.66</pre>

Mission success! We've managed to free up a lot of memory. Note that the code which moves all objects collected by the garbage collector to the CPU is often necessary to free up the memory. We can't just delete the objects directly because PyTorch can still sometimes keep references to them (i.e. their tensors) in memory. In fact, if you add code to the for loop above to print out `obj.shape` when `obj` is a tensor, you'll see that a lot of those tensors are actually Gemma model weights, even once you've deleted `gemma_2_2b`.

</details>

## 설정 (읽지 말고 실행만 하세요)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install "openai==1.56.1" einops datasets jaxtyping "sae-lens>=4.0.0,<5.0.0" openai tabulate umap-learn hdbscan eindex-callum git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python git+https://github.com/callummcdougall/sae_vis.git@callum/v3 transformer_lens==2.17.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import gc
import os
import sys
from collections import Counter, namedtuple
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Callable, TypeAlias

import einops
import numpy as np
import plotly.express as px
import torch as t
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from IPython.display import IFrame, display
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from sae_lens import SAE, ActivationsStore, HookedSAETransformer
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from tabulate import tabulate
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache, HookedTransformer
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, test_prompt, to_numpy

dtype = t.float32  # t.bfloat16
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
device = str(device)  # SAELens expects device as string; this is easier!


def _get_hook_layer(sae: SAE) -> int:
    """Extract the layer number from an SAE's hook name (e.g. 'blocks.7.hook_resid_pre' → 7)."""
    return int(sae.cfg.metadata.hook_name.split(".")[1])


# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part42_sae_circuits"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part42_sae_circuits.tests as tests
import part42_sae_circuits.utils as utils

MAIN = __name__ == "__main__"

In [ ]:
# For displaying sae-vis inline
if IN_COLAB:
    import http.server
    import socketserver
    import threading

    from google.colab import output

    PORT = 8000

    def display_vis_inline(filename: Path, height: int = 850):
        """
        Displays the HTML files in Colab. Uses global `PORT` variable defined in prev cell, so that each
        vis has a unique port without having to define a port within the function.
        """
        global PORT

        def serve(directory):
            os.chdir(directory)
            handler = http.server.SimpleHTTPRequestHandler
            with socketserver.TCPServer(("", PORT), handler) as httpd:
                print(f"Serving files from {directory} on port {PORT}")
                httpd.serve_forever()

        thread = threading.Thread(target=serve, args=("/content",))
        thread.start()

        filename = str(filename).split("/content")[-1]

        output.serve_kernel_port_as_iframe(PORT, path=filename, height=height, cache_in_notebook=True)

        PORT += 1

## 관련 배경지식 빠르게 살펴보기

이 섹션에서는 나머지 실습을 위해 필요한 핵심 SAELens 개념들을 다룹니다. **이미 실습 세트 1.3.3 'Interpretability with SAEs'를 완료하셨다면, 이 섹션을 건너뛰고 바로 1️⃣ 섹션으로 이동하셔도 좋습니다.**

### `SAE.from_pretrained`로 SAE 로드하기

[SAELens](https://github.com/jbloomAus/SAELens)은 연구자들이 sparse autoencoder를 훈련하고 분석하는 것을 돕기 위해 설계된 라이브러리입니다. 이를 sparse autoencoder를 위한 TransformerLens의 대응물이라고 생각하시면 됩니다 (또한 곧 살펴보겠지만, TransformerLens 모델과 매우 잘 통합됩니다).

SAE를 로드하려면 `SAE.from_pretrained`를 사용합니다. 이 함수는 SAE 객체를 직접 반환합니다:

```python
gpt2_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device=str(device),
)
```

`get_pretrained_saes_directory()`를 통해 SAELens에서 사용 가능한 SAE 릴리스를 확인할 수 있습니다. 각 릴리스에는 여러 개의 SAE가 포함되어 있습니다 (예: 동일한 base model의 서로 다른 layer에서 훈련된 경우).

Base model은 `HookedSAETransformer` 클래스를 사용하여 로드하며, 이는 TransformerLens의 `HookedTransformer` 클래스를 기반으로 조정되었습니다:

```python
gpt2 = HookedSAETransformer.from_pretrained("gpt2-small", device=device)
```

### SAE 실행 및 activation 캐싱

TransformerLens 모델에서 forward pass를 수행할 때, hook 함수를 추가하는 것과 매우 유사한 방식으로 SAE를 추가할 수 있습니다. 이를 위한 몇 가지 서로 다른 방법이 있습니다:

- **`model.run_with_saes(tokens, saes=[list_of_saes])`**는 `model.run_with_hooks`와 유사하게 작동하며, SAE가 연결된 상태로 단일 forward pass를 수행합니다 (그 후 SAE를 리셋합니다).
- **`logits, cache = model.run_with_cache_with_saes(tokens, saes=[sae])`**는 `model.run_with_cache`와 유사하게 작동하며, SAE activation을 포함한 모든 중간 activation을 캐싱합니다.
- **`with model.saes(saes=[sae]):`**는 일시적으로 SAE를 연결하는 context manager입니다.
- **`model.add_sae(sae)` / `model.reset_saes()`**는 수동으로 SAE를 추가하고 제거합니다.

캐시에서 SAE activation에 접근하려면, HookedTransformer `hook_name`와 SAE hook 이름을 마침표(.)로 연결한 이름을 사용합니다. 가장 중요한 이름들은 다음과 같습니다:

```python
# Post-activation latent values (shape [batch, seq, d_sae]) - this is the one you'll use most
cache[f"{sae.cfg.metadata.hook_name}.hook_sae_acts_post"]

# Pre-activation latent values (before the activation function)
cache[f"{sae.cfg.metadata.hook_name}.hook_sae_acts_pre"]

# The SAE's reconstruction of the original activations
cache[f"{sae.cfg.metadata.hook_name}.hook_sae_recons"]

# The final SAE output (either reconstruction, or reconstruction + error term)
cache[f"{sae.cfg.metadata.hook_name}.hook_sae_output"]
```

다음은 프롬프트의 마지막 token에서 가장 높게 activation된 latent들을 추출하는 전체 예시입니다:

```python
_, cache = gpt2.run_with_cache_with_saes(
    prompt,
    saes=[gpt2_sae],
    stop_at_layer=_get_hook_layer(gpt2_sae) + 1,  # no need to compute past the SAE layer
)
sae_acts_post = cache[f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][0, -1, :]
```

### `use_error_term` 파라미터

파라미터 `sae.use_error_term`은 forward pass 동안 모델의 activation을 SAE reconstruction으로 실제로 대체할지 여부를 결정합니다:

- **`use_error_term=False`** (기본값): SAE의 출력이 transformer의 activation을 대체합니다. 이는 downstream 계산에서 원래의 activation 대신 SAE reconstruction을 사용함을 의미합니다.
- **`use_error_term=True`**: SAE는 모든 내부 상태(latent activation 등)를 동일한 방식으로 계산하지만, transformer의 activation은 그대로 유지됩니다. 이는 모델에 실제로 개입하지 않고 SAE activation을 캐싱하고 싶을 때 유용합니다.

```python
# Cache SAE activations WITHOUT intervening on the model
gpt2_sae.use_error_term = True
logits, cache = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])
# logits are identical to running the base model without SAEs, but we still get SAE activations in the cache

# Cache SAE activations AND replace model activations with SAE reconstructions
gpt2_sae.use_error_term = False
logits_recon, cache_recon = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])
# logits_recon will differ from the base model's logits due to SAE reconstruction error
```

이 파라미터는 이번 세트의 gradient 연습 문제에서 매우 중요합니다. 일반적으로 깨끗한 forward pass로부터 "true" latent activation을 얻기 위해 `use_error_term=True`를 사용하고, Jacobian을 계산할 때는 `use_error_term=False`을 사용합니다 (Jacobian 함수가 실제로 SAE decoder와 encoder를 통과하도록 하기 위함입니다).

### `ActivationsStore` 사용하기

`ActivationsStore` 클래스는 많은 양의 데이터를 직접 로드하는 대신 사용할 수 있는 편리한 대안입니다. 이 클래스는 주어진 데이터셋에서 데이터를 스트리밍합니다. `from_sae` classmethod의 경우, 해당 데이터셋은 SAE의 config에 의해 제공됩니다 (이는 SAE의 원래 학습 데이터셋과 동일합니다):

```python
gpt2_act_store = ActivationsStore.from_sae(
    model=gpt2,
    sae=gpt2_sae,
    streaming=True,
    store_batch_size_prompts=16,
    n_batches_in_buffer=32,
    device=str(device),
)

# Get a batch of tokens
tokens = gpt2_act_store.get_batch_tokens()
assert tokens.shape == (gpt2_act_store.store_batch_size_prompts, gpt2_act_store.context_size)
```

### Neuronpedia & `display_dashboard`

[Neuronpedia](https://neuronpedia.org)은 interpretability 연구를 위한 오픈 플랫폼입니다. 이곳은 특정 SAE latent가 무엇을 나타내는지 빠르게 이해할 수 있도록 돕는 **SAE dashboards**를 제공하며, 여기에는 max activating examples, top logits, activation density plots, 그리고 LLM이 생성한 설명과 같은 구성 요소들이 포함되어 있습니다.

우리는 다음과 같은 helper function을 사용하여 이러한 dashboards를 인라인으로 표시할 수 있으며, 이 함수를 이번 실습의 여러 곳에서 사용할 예정입니다:

In [ ]:
def display_dashboard(
    sae_release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    latent_idx=0,
    width=800,
    height=600,
) -> None:
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

    print(url)
    display(IFrame(url, width=width, height=height))

# 1️⃣ Latent Gradients

> ##### 학습 목표
>
> - transformer의 서로 다른 레이어에 있는 SAE latent들 사이의 **latent-to-latent gradients**를 계산하는 방법을 배웁니다.
> - 어떤 입력 token이 특정 latent activation을 유도하는지 이해하기 위해 **token-to-latent gradients**를 계산합니다.
> - latent가 모델의 출력에 어떻게 영향을 미치는지 이해하기 위해 **latent-to-logit gradients**를 계산합니다.
> - 이러한 gradient 기반 방법들을 사용하여 attention SAE 내의 circuit(예: induction circuits)을 찾습니다.

## 서론

이전의 SAE 연구(자료 1.3.3)에서는 개별 latent를 이해하는 데 집중했습니다. 여기서는 매우 중요한 주제인 **SAE latent의 circuit**에 대해 다룹니다. Circuit 분석은 이미 언어 모델 interpretability 분야에서 어느 정도 성공을 거두었습니다 (예: induction circuit에 대한 Anthropic의 연구 또는 Indirect Object Identification 논문 참조). 하지만 circuit 분석을 더 발전시키려는 많은 시도들이 난관에 부딪혔습니다. 모델 내의 대부분의 연결은 sparse하지 않으며, 서로 다른 컴포넌트와 residual stream subspace 사이의 복잡한 cross-talk를 모두 분리해내는 것이 매우 어렵기 때문입니다. Circuit은 더 나은 방향을 제시합니다. 개별 latent가 일반적으로 sparse할 뿐만 아니라, 이들이 **sparsely connected** 되어 있을 것이라고 기대할 수 있기 때문입니다. 즉, 특정 latent는 아마도 소수의 다른 latent에만 downstream 효과를 줄 것입니다.

실제로 이것이 사실로 밝혀진다면, 이는 SAE로 찾은 latent가 단순히 흥미로운 clustering 알고리즘의 결과가 아니라, 모델이 사용하는 **계산의 기본 단위(fundamental units of computation)**라는 강력한 증거가 됩니다. 물론 우리는 이미 이에 대한 몇 가지 증거를 가지고 있습니다 (예: latent steering의 효과, 그리고 기본적인 컴포넌트만 보았을 때는 명확하지 않았던 모델의 중요한 정보들을 latent가 이미 드러냈다는 사실). 하지만 명확한 latent circuit을 찾아내는 것은 훨씬 더 강력한 증거가 될 것입니다.

## Latent Gradients

우리는 latent 연결에서 기대할 수 있는 sparsity의 종류와 latent circuit 분석이 어려울 수 있는 여러 이유를 보여주는 실습으로 시작하겠습니다. 우리는 두 개의 서로 다른 layer에 속한 SAE들의 모든 활성화된 latent 쌍 사이의 gradient를 반환하는 `latent_to_latent_gradients` 함수를 구현할 것입니다 (우리는 `gpt2-small-res-jb` 릴리스의 두 SAE를 사용할 것입니다). 이러한 gradient를 계산하는 것은 생각보다 복잡하기 때문에, 이 실습은 몇 가지 단계로 나누어 진행됩니다.

latent gradient란 정확히 무엇일까요? 임의의 입력과 서로 다른 layer에 있는 임의의 2개 latent에 대해, 첫 번째 latent에 대한 두 번째 latent activation의 미분을 계산할 수 있습니다. 이는 편미분 행렬, 즉 $J_{ij} = \frac{\partial f_i}{\partial x_j}$의 형태를 띠며, 초기 layer의 latent가 이후 layer의 latent에 어떻게 기여하는지에 대한 선형 proxy 역할을 할 수 있습니다. 이를 계산하기 위한 pseudocode는 다음과 같습니다:

```python
# Computed with no gradients, and not patching in SAE reconstructions...
layer_1_latents, layer_2_latents = model.run_with_cache_with_saes(...)

def latent_acts_to_later_latent_acts(layer_1_latents):
    layer_1_resid_acts_recon = SAE_1_decoder(layer_1_latents)
    layer_2_resid_acts_recon = model.blocks[layer_1: layer_2].forward(layer_1_resid_acts_recon)
    layer_2_latents_recon = SAE_2_encoder(layer_2_resid_acts_recon)
    return layer_2_latents_recon

latent_latent_gradients = t.func.jacrev(latent_acts_to_later_latent_acts)(layer_1_latents)
```

여기서 `jacrev`는 "Jacobian reverse-mode differentiation"의 약어입니다. 이는 tensor -> tensor 함수 `f(x) = y`를 입력으로 받아 Jacobian 함수, 즉 `g[i, j] = d(f[x]_i) / d(x_j)`을 만족하는 `g`을 반환하는 PyTorch 함수입니다.

데이터 분포 전반에 걸쳐 latent들이 서로 어떻게 통신하는지 파악하고 싶다면, 대규모 prompt 세트에 대해 이 결과들의 평균을 낼 수 있습니다. 하지만 지금은 메모리 문제를 피하고 결과를 더 쉽게 시각화하기 위해 상대적으로 작은 prompt 세트를 사용할 것입니다.

먼저, 아직 하지 않으셨다면 모델과 SAE를 로드해 보겠습니다:

In [ ]:
gpt2 = HookedSAETransformer.from_pretrained("gpt2-small", device=device, dtype=dtype)

gpt2_saes = {
    layer: SAE.from_pretrained(
        release="gpt2-small-res-jb",
        sae_id=f"blocks.{layer}.hook_resid_pre",
        device=device,
        dtype=dtype,
    )
    for layer in tqdm(range(gpt2.cfg.n_layers))
}

이제 실습을 시작할 수 있습니다!

참고 - 이어지는 3가지 실습은 모두 다소 복잡하며, Jacobian의 사용과 같은 부분은 상당히 까다로울 수 있습니다. 그러한 이유로, 직접 시도하기보다는 솔루션을 읽으며 코드가 무엇을 수행하는지 이해하는 것이 더 효율적일 수 있습니다. 한 가지 방법은 이 3가지 실습의 솔루션을 보고 latent-to-latent gradient가 어떻게 작동하는지 이해한 다음, (다음 3가지 실습 이후에 나오는) `token_to_latent_gradients` 함수를 직접 구현해 보는 것입니다.

### 연습 문제 (1/3) - `SparseTensor` 클래스 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise (or skip it)
> ```

먼저, sparse tensor(즉, 대부분의 요소가 0인 tensor)를 다루는 것을 돕기 위해 `SparseTensor` 클래스를 생성하겠습니다. 이는 dense tensor(즉, 0을 포함한 모든 값을 포함하는 tensor)로 forward pass를 수행해야 하지만, gradient는 sparse tensor(0이 아닌 값들만)에 대해 계산하고 싶기 때문입니다. 그렇지 않으면 latent의 수가 매우 많아 메모리 문제가 발생할 수 있습니다!

`SparseTensor` 클래스의 `from_dense` 및 `from_sparse` 클래스 메서드를 완성해야 합니다. 테스트 코드가 제공되므로, 이를 통해 이 클래스가 어떻게 동작해야 하는지 이해하는 데 도움이 될 것입니다.

In [ ]:
class SparseTensor:
    """
    Handles 2D tensor data (assumed to be non-negative) in 2 different formats:
        dense:  The full tensor, which contains zeros. Shape is (n1, ..., nk).
        sparse: A tuple of nonzero values with shape (n_nonzero,), nonzero indices with shape
                (n_nonzero, k), and the shape of the dense tensor.
    """

    sparse: tuple[Tensor, Tensor, tuple[int, ...]]
    dense: Tensor

    def __init__(self, sparse: tuple[Tensor, Tensor, tuple[int, ...]], dense: Tensor):
        self.sparse = sparse
        self.dense = dense

    @classmethod
    def from_dense(cls, dense: Tensor) -> "SparseTensor":
        """Creates a SparseTensor from a dense tensor, extracting the positive entries."""
        raise NotImplementedError()

    @classmethod
    def from_sparse(cls, sparse: tuple[Tensor, Tensor, tuple[int, ...]]) -> "SparseTensor":
        """Creates a SparseTensor from a (values, indices, shape) tuple, reconstructing the dense tensor."""
        raise NotImplementedError()

    @property
    def values(self) -> Tensor:
        return self.sparse[0].squeeze()

    @property
    def indices(self) -> Tensor:
        return self.sparse[1].squeeze()

    @property
    def shape(self) -> tuple[int, ...]:
        return self.sparse[2]


# Test `from_dense`
x = t.zeros(10_000)
nonzero_indices = t.randint(0, 10_000, (10,)).sort().values
nonzero_values = t.rand(10)
x[nonzero_indices] = nonzero_values
sparse_tensor = SparseTensor.from_dense(x)
t.testing.assert_close(sparse_tensor.sparse[0], nonzero_values)
t.testing.assert_close(sparse_tensor.sparse[1].squeeze(-1), nonzero_indices)
t.testing.assert_close(sparse_tensor.dense, x)

# Test `from_sparse`
sparse_tensor = SparseTensor.from_sparse((nonzero_values, nonzero_indices.unsqueeze(-1), tuple(x.shape)))
t.testing.assert_close(sparse_tensor.dense, x)

# Verify other properties
t.testing.assert_close(sparse_tensor.values, nonzero_values)
t.testing.assert_close(sparse_tensor.indices, nonzero_indices)

tests.test_sparse_tensor(SparseTensor)

<details><summary>솔루션</summary>

```python
class SparseTensor:
    """
    Handles 2D tensor data (assumed to be non-negative) in 2 different formats:
        dense:  The full tensor, which contains zeros. Shape is (n1, ..., nk).
        sparse: A tuple of nonzero values with shape (n_nonzero,), nonzero indices with shape
                (n_nonzero, k), and the shape of the dense tensor.
    """

    sparse: tuple[Tensor, Tensor, tuple[int, ...]]
    dense: Tensor

    def __init__(self, sparse: tuple[Tensor, Tensor, tuple[int, ...]], dense: Tensor):
        self.sparse = sparse
        self.dense = dense

    @classmethod
    def from_dense(cls, dense: Tensor) -> "SparseTensor":
        """Creates a SparseTensor from a dense tensor, extracting the positive entries."""
        sparse = (dense[dense > 0], t.argwhere(dense > 0), tuple(dense.shape))
        return cls(sparse, dense)

    @classmethod
    def from_sparse(cls, sparse: tuple[Tensor, Tensor, tuple[int, ...]]) -> "SparseTensor":
        """Creates a SparseTensor from a (values, indices, shape) tuple, reconstructing the dense tensor."""
        nonzero_values, nonzero_indices, shape = sparse
        dense = t.zeros(shape, dtype=nonzero_values.dtype, device=nonzero_values.device)
        dense[nonzero_indices.unbind(-1)] = nonzero_values
        return cls(sparse, dense)

    @property
    def values(self) -> Tensor:
        return self.sparse[0].squeeze()

    @property
    def indices(self) -> Tensor:
        return self.sparse[1].squeeze()

    @property
    def shape(self) -> tuple[int, ...]:
        return self.sparse[2]
```
</details>

### 연습 문제 (2/3) - `latent_acts_to_later_latent_acts` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

다음으로, `latent_acts_to_later_latent_acts`를 구현해야 합니다. 이 함수는 모델 앞부분의 latent activation(sparse 형태, 즉 (values, indices, shape) 튜플)을 입력으로 받아, downstream latent activation을 `(sparse_values, (dense_values,))` 튜플로 반환합니다.

왜 `latent_acts_next`의 복사본 2개를 이런 이상한 방식으로 반환하는 것일까요? 그 이유는 우리가 작성한 함수를 `t.func.jacrev(latent_acts_to_later_latent_acts, has_aux=True)`로 감쌀 것이기 때문입니다. `has_aux` 인자를 사용하면 미분되지 않는 tensor 튜플을 반환할 수 있습니다. 다시 말해, 이 함수는 tensor -> (tensor, tuple_of_tensors) 함수 `f(x) = (y, aux)`를 입력받아 `J[i, j] = d(f[x]_i) / d(x_j)`인 함수 `g(x) = (J, aux)`를 반환합니다. 즉, Jacobian과 실제 재구성된 activation을 모두 얻게 되는 것입니다.

<details>
<summary>실제로 어떤 gradient를 계산하고 있는지에 대한 참고 사항</summary>

주의 깊게 읽으신 분들은 우리가 여기서 실제로 하고 있는 일이 나중 latent와 이전 latent 사이의 gradient를 계산하는 것이 아니라, **재구성된 나중 latent**와 이전 latent 사이의 gradient를 계산하는 것이라는 점을 눈치채셨을 것입니다. 다시 말해, 우리가 미분하고 있는 나중 latent는 실제 residual stream이 아니라, 이전 SAE의 residual stream 재구성 결과에 대한 함수입니다. 이는 결과로부터 결론을 도출할 때 다소 위험할 수 있는데, 만약 이전 SAE가 입력을 재구성하는 성능이 좋지 않다면 downstream latent가 upstream activation에 의해 영향을 받는 방식을 놓칠 수 있기 때문입니다. 이를 확인하는 좋은 방법은 (이전 SAE의 재구성 결과로부터 계산된) latent activation과 실제 latent activation을 비교하여 서로 유사한지 확인하는 것입니다.

</details>

Jacobian을 적용하는 것은 세 번째 연습 문제에서 다루겠습니다. 지금은 `latent_acts_to_later_latent_acts`를 채워 넣으시면 됩니다. 이는 기본적으로 이 섹션 시작 부분에서 제공한 `latent_acts_to_later_latent_acts`의 pseudocode와 일치해야 합니다 (tensor를 sparse 형태로 변환하거나 그 반대로 변환하는 과정이 추가됩니다). 도움이 될 만한 구문 안내는 다음과 같습니다:

- 모든 SAE는 input -> latent activation -> reconstructed input으로 매핑하는 `encode` 및 `decode` 메서드를 가지고 있습니다.
- 모든 TransformerLens 모델은 선택적 인자인 `start_at_layer`와 `stop_at_layer`을 갖는 `forward` 메서드를 가지고 있으며, 이 인자들이 제공되면 이전 layer의 함수로서 나중 layer의 activation을 계산합니다.

In [ ]:
def latent_acts_to_later_latent_acts(
    latent_acts_nonzero: Float[Tensor, " nonzero_acts"],
    latent_acts_nonzero_inds: Int[Tensor, "nonzero_acts n_indices"],
    latent_acts_shape: tuple[int, ...],
    sae_from: SAE,
    sae_to: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, tuple[Tensor]]:
    """
    Given some latent activations for a residual stream SAE earlier in the model, computes the
    latent activations of a later SAE. It does this by mapping the latent activations through the
    path SAE decoder -> intermediate model layers -> later SAE encoder.

    This function must input & output sparse information (i.e. nonzero values and their indices)
    rather than dense tensors, because latent activations are sparse but jacrev() doesn't support
    gradients on real sparse tensors.
    """
    # ... YOUR CODE HERE ...

    return latent_acts_next_recon.sparse[0], (latent_acts_next_recon.dense,)

<details><summary>솔루션</summary>

```python
def latent_acts_to_later_latent_acts(
    latent_acts_nonzero: Float[Tensor, " nonzero_acts"],
    latent_acts_nonzero_inds: Int[Tensor, "nonzero_acts n_indices"],
    latent_acts_shape: tuple[int, ...],
    sae_from: SAE,
    sae_to: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, tuple[Tensor]]:
    """
    Given some latent activations for a residual stream SAE earlier in the model, computes the
    latent activations of a later SAE. It does this by mapping the latent activations through the
    path SAE decoder -> intermediate model layers -> later SAE encoder.

    This function must input & output sparse information (i.e. nonzero values and their indices)
    rather than dense tensors, because latent activations are sparse but jacrev() doesn't support
    gradients on real sparse tensors.
    """
    # Convert to dense, map through SAE decoder
    latent_acts = SparseTensor.from_sparse((latent_acts_nonzero, latent_acts_nonzero_inds, latent_acts_shape)).dense
    resid_stream_from = sae_from.decode(latent_acts)

    # Map through model layers
    resid_stream_next = model.forward(
        resid_stream_from,
        start_at_layer=_get_hook_layer(sae_from),
        stop_at_layer=_get_hook_layer(sae_to),
    )

    # Map through SAE encoder, and turn back into SparseTensor
    latent_acts_next_recon = sae_to.encode(resid_stream_next)
    latent_acts_next_recon = SparseTensor.from_dense(latent_acts_next_recon)

    return latent_acts_next_recon.sparse[0], (latent_acts_next_recon.dense,)
```
</details>

### 연습 문제 (3/3) - `latent_to_latent_gradients` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 20-40 minutes on this exercise.
> ```

마지막으로, 전체 `latent_to_latent_gradients` 함수를 구현할 준비가 되었습니다. 이 함수는 다음과 같은 작업을 수행해야 합니다:

- `run_with_cache_with_saes`을 사용하여 두 SAE 모두에 대해 실제 latent activation을 계산합니다 (`sae_from.use_error_term = True`를 설정했는지 확인하십시오. 이전 SAE의 reconstruction으로부터 계산된 것이 아니라, 이후 SAE에 대한 실제 latent activation을 계산해야 하기 때문입니다!)
- 함수 `latent_acts_to_later_latent_acts`를 래핑하여 Jacobian과 이후의 latent activation을 반환하는 함수를 생성합니다 (이 과정이 어떻게 이루어지는지 헷갈린다면 아래 드롭다운의 코드를 확인하십시오),
- 이 함수를 호출하여 Jacobian과 이후의 latent activation을 반환합니다,
- Jacobian과 이전/이후 latent activation을 반환합니다 (후자는 `SparseTensor` 객체로 반환합니다).

<details>
<summary>Jacobian 래퍼 코드</summary>

```python
latent_acts_to_later_latent_acts_and_gradients = t.func.jacrev(
    latent_acts_to_later_latent_acts, argnums=0, has_aux=True
)
```

`argnums=0` 인자는 `jacrev`에게 `latent_acts_to_later_latent_acts`의 첫 번째 인자에 대한 Jacobian을 계산하도록 지시하며, `has_aux=True` 인자는 `latent_acts_to_later_latent_acts`의 보조 출력(즉, 기본 함수의 두 번째 출력인 텐서 튜플)도 함께 반환하도록 지시합니다.

다음과 같이 이 함수를 호출할 수 있습니다:

```python
latent_latent_gradients, (latent_acts_next_recon_dense,) = latent_acts_to_later_latent_acts_and_gradients(
    *latent_acts_prev.sparse, sae_from, sae_to, model
)
```

</details>

<details>
<summary>도움말 - OOM 에러가 발생합니다</summary>

sparsified 버전이 아닌 텐서를 전달할 때 OOM 에러가 자주 발생합니다 (10k개 이상의 요소에 대한 2D 미분 행렬을 계산하는 것은 메모리 소모가 매우 크기 때문입니다!). 에러가 발생했을 때 요청되는 메모리 양을 확인하는 것을 권장합니다. 만약 30GB 이상이라면 거의 확실히 이 실수를 하고 계신 것입니다.

여전히 에러가 발생한다면, 메모리를 점검하고 정리하는 것을 권장합니다. 특히, Gemma와 같은 대형 모델을 GPU에 로드하면 더 이상 필요하지 않은 공간을 차지하게 됩니다. 이를 위해 몇 가지 유틸리티 함수를 제공했습니다 (이 노트북의 맨 처음, 첫 번째 연습 문제 세트 전의 "A note on memory usage" 헤더 아래에 사용 예시가 있습니다).

이 모든 방법으로도 해결되지 않는다면 (즉, 메모리를 정리한 후에도 여전히 에러가 발생한다면), 가상 머신(예: vastai)이나 Colab 노트북을 사용해 보시는 것을 권장합니다.

</details>

함수 아래에 gradient의 히트맵을 생성하고 실행하는 코드를 제공했습니다. 참고로, 플롯의 축은 latent에 대해 `F{layer}.{latent_idx}` 표기법을 사용합니다.

챌린지 - 첫 번째 token이 `" E"` 인 토큰화된 단어들로 구성된 bigram에서 circuit을 형성하는 것으로 보이는 latent 쌍을 찾을 수 있습니까?

In [ ]:
def latent_to_latent_gradients(
    tokens: Float[Tensor, "batch seq"],
    sae_from: SAE,
    sae_to: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, SparseTensor, SparseTensor, SparseTensor]:
    """
    Computes the gradients between all active pairs of latents belonging to two SAEs.

    Returns:
        latent_latent_gradients:    The gradients between all active pairs of latents
        latent_acts_prev:           The latent activations of the first SAE
        latent_acts_next:           The latent activations of the second SAE
        latent_acts_next_recon:     The reconstructed latent activations of the second SAE (i.e.
                                    based on the first SAE's reconstructions)
    """
    # ... YOUR CODE HERE ...

    return (
        latent_latent_gradients,
        latent_acts_prev,
        latent_acts_next,
        latent_acts_next_recon,
    )


tests.test_latent_to_latent_gradients(latent_to_latent_gradients, gpt2, gpt2_saes)

<details><summary>솔루션</summary>

```python
def latent_to_latent_gradients(
    tokens: Float[Tensor, "batch seq"],
    sae_from: SAE,
    sae_to: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, SparseTensor, SparseTensor, SparseTensor]:
    """
    Computes the gradients between all active pairs of latents belonging to two SAEs.

    Returns:
        latent_latent_gradients:    The gradients between all active pairs of latents
        latent_acts_prev:           The latent activations of the first SAE
        latent_acts_next:           The latent activations of the second SAE
        latent_acts_next_recon:     The reconstructed latent activations of the second SAE (i.e.
                                    based on the first SAE's reconstructions)
    """
    acts_prev_name = f"{sae_from.cfg.metadata.hook_name}.hook_sae_acts_post"
    acts_next_name = f"{sae_to.cfg.metadata.hook_name}.hook_sae_acts_post"
    sae_from.use_error_term = True  # so we can get both true latent acts at once

    with t.no_grad():
        # Get the true activations for both SAEs
        _, cache = model.run_with_cache_with_saes(
            tokens,
            names_filter=[acts_prev_name, acts_next_name],
            stop_at_layer=_get_hook_layer(sae_to) + 1,
            saes=[sae_from, sae_to],
            remove_batch_dim=False,
        )
        latent_acts_prev = SparseTensor.from_dense(cache[acts_prev_name])
        latent_acts_next = SparseTensor.from_dense(cache[acts_next_name])

    # Compute jacobian between earlier and later latent activations (and also get the activations
    # of the later SAE which are downstream of the earlier SAE's reconstructions)
    latent_latent_gradients, (latent_acts_next_recon_dense,) = t.func.jacrev(
        latent_acts_to_later_latent_acts, has_aux=True
    )(
        *latent_acts_prev.sparse,
        sae_from,
        sae_to,
        model,
    )

    latent_acts_next_recon = SparseTensor.from_dense(latent_acts_next_recon_dense)

    # Set SAE state back to default
    sae_from.use_error_term = False

    return (
        latent_latent_gradients,
        latent_acts_prev,
        latent_acts_next,
        latent_acts_next_recon,
    )
```
</details>

gradient heatmap을 시각화하기 위해 아래 코드를 실행하기 전, **어떤 결과가 나올지 생각해보십시오**. 이 heatmap은 프롬프트 `"The Eiffel tower is in Paris"`에 대해 layer 0의 SAE latents와 layer 3의 SAE latents 사이의 gradients를 보여줍니다.

- 대부분의 항목이 0일 것이라고 예상하십니까, 아니면 0이 아닐 것이라고 예상하십니까? 그 이유는 무엇입니까?
- 어떤 (source position, destination position) 쌍이 큰 gradient 값을 가질 가능성이 가장 높습니까?
- gradient matrix가 dense할 것이라고 예상하십니까, 아니면 sparse할 것이라고 예상하십니까?

아래 셀을 실행하고 여러분의 예측과 비교해보십시오.

In [ ]:
prompt = "The Eiffel tower is in Paris"
tokens = gpt2.to_tokens(prompt)
str_toks = gpt2.to_str_tokens(prompt)
layer_from = 0
layer_to = 3

# Get latent-to-latent gradients
t.cuda.empty_cache()
with t.enable_grad():
    (
        latent_latent_gradients,
        latent_acts_prev,
        latent_acts_next,
        latent_acts_next_recon,
    ) = latent_to_latent_gradients(tokens, gpt2_saes[layer_from], gpt2_saes[layer_to], gpt2)

# Verify that ~the same latents are active in both, and the MSE loss is small
nonzero_latents = [tuple(x) for x in latent_acts_next.indices.tolist()]
nonzero_latents_recon = [tuple(x) for x in latent_acts_next_recon.indices.tolist()]
alive_in_one_not_both = set(nonzero_latents) ^ set(nonzero_latents_recon)
print(f"# nonzero latents (true): {len(nonzero_latents)}")
print(f"# nonzero latents (reconstructed): {len(nonzero_latents_recon)}")
print(f"# latents alive in one but not both: {len(alive_in_one_not_both)}")

fig = px.imshow(
    to_numpy(latent_latent_gradients.T),
    color_continuous_midpoint=0.0,
    color_continuous_scale="RdBu",
    x=[f"F{layer_to}.{latent}, {str_toks[seq]!r} ({seq})" for (_, seq, latent) in latent_acts_next_recon.indices],
    y=[f"F{layer_from}.{latent}, {str_toks[seq]!r} ({seq})" for (_, seq, latent) in latent_acts_prev.indices],
    labels={"x": f"To layer {layer_to}", "y": f"From layer {layer_from}"},
    title=f'Gradients between SAE latents in layer {layer_from} and SAE latents in layer {layer_to}<br><sup>   Prompt: "{"".join(str_toks)}"</sup>',
    width=1600,
    height=1000,
)
fig.show()

<details>
<summary>몇 가지 관찰 사항</summary>

0이 아닌 gradient의 상당수는 동일한 token에서 활성화되는 token 쌍들에 대한 것입니다. 예를 들어, `(F0.9449, " Paris") -> (F3.385, " Paris")`은 서로 다른 두 layer에 있는 유사한 feature일 가능성이 있어 보입니다:

```python
display_dashboard(sae_id="blocks.0.hook_resid_pre", latent_idx=9449)
display_dashboard(sae_id="blocks.3.hook_resid_pre", latent_idx=385)
```

cross-token gradient는 그리 많지 않습니다. 가장 눈에 띄는 것 중 하나는 `(F0.16911, " E") -> (F3.15266, "iff")`이며, 이는 `" E"`으로 시작하는 단어들을 위한 bigram circuit일 가능성이 있어 보입니다:

```python
display_dashboard(sae_id="blocks.0.hook_resid_pre", latent_idx=16911)
display_dashboard(sae_id="blocks.3.hook_resid_pre", latent_idx=15266)
```

</details>

### 연습 문제 - latent-to-token gradient 구하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 30-40 minutes on this exercise.
> ```

latent-to-latent gradient 구현을 마쳤으므로, 이제 전체 과정을 다시 시도해 보겠습니다. 다만 이번에는 모든 input token과 특정 SAE의 latent 사이의 gradient를 계산합니다.

token은 스칼라 값이 아니기 때문에 token과 latent 사이의 gradient가 무엇을 의미하는지 궁금하실 수 있습니다. 정답은 모델의 embedding에 특정 scale factor `s` (즉, 시퀀스의 각 token에 대한 서로 다른 scale factor 값들의 벡터)를 곱하고, `s = [1, 1, ..., 1]`에서 평가된 이 값들에 대한 SAE latent의 gradient `s`를 구하는 것입니다. 실제로 우리 모델에서는 이러한 embedding 벡터 스케일링이 일어나지 않으므로 매우 원칙적인 방법은 아니지만, **어떤 token이 어떤 latent에 가장 중요한지**를 파악하는 편리한 방법입니다.

챌린지 - 이전 연습 문제에서 첫 번째 token이 `" E"`인 tokenized 단어들로 구성된 bigram 상에서 circuit을 형성하는 것처럼 보였던 latent 쌍을 가져오십시오. 이 plot에서 해당 circuit을 다시 찾을 수 있습니까?

In [ ]:
def tokens_to_latent_acts(
    token_scales: Float[Tensor, "batch seq"],
    tokens: Int[Tensor, "batch seq"],
    sae: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, tuple[Tensor]]:
    """
    Given scale factors for model's embeddings (i.e. scale factors applied after we compute the sum
    of positional and token embeddings), returns the SAE's latents.

    Returns:
        latent_acts_sparse: The SAE's latents in sparse form (i.e. the tensor of values)
        latent_acts_dense:  The SAE's latents in dense tensor, in a length-1 tuple
    """
    # ... YOUR CODE HERE ...

    return sae_latents.sparse[0], (sae_latents.dense,)


def token_to_latent_gradients(
    tokens: Float[Tensor, "batch seq"],
    sae: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, SparseTensor]:
    """
    Computes the gradients between an SAE's latents and all input tokens.

    Returns:
        token_latent_grads: The gradients between input tokens and SAE latents
        latent_acts:        The SAE's latent activations
    """
    # ... YOUR CODE HERE ...

    return (token_latent_grads, latent_acts)


tests.test_token_to_latent_gradients(token_to_latent_gradients, gpt2, gpt2_saes)

sae_layer = 3
token_latent_grads, latent_acts = token_to_latent_gradients(tokens, sae=gpt2_saes[sae_layer], model=gpt2)

fig = px.imshow(
    to_numpy(token_latent_grads[0]),
    color_continuous_midpoint=0.0,
    color_continuous_scale="RdBu",
    x=[f"F{sae_layer}.{latent:05}, {str_toks[seq]!r} ({seq})" for (_, seq, latent) in latent_acts.indices],
    y=[f"{str_toks[i]!r} ({i})" for i in range(len(str_toks))],
    labels={"x": f"To layer {sae_layer}", "y": "From tokens"},
    title=f'Gradients between input tokens and SAE latents in layer {sae_layer}<br><sup>   Prompt: "{"".join(str_toks)}"</sup>',
    width=1900,
    height=450,
)
fig.show()

<details>
<summary>몇 가지 관찰 사항</summary>

이전 연습에서 우리는 `(F0.16911, " E") -> (F3.15266, "iff")` 사이의 gradient를 확인했으며, 이는 `" E"`로 시작하는 단어들을 위한 bigram circuit을 형성하고 있는 것으로 보입니다.

이 plot에서 우리는 `" E"` token과 feature `F3.15266` 사이의 gradient를 볼 수 있으며, 이는 앞선 내용에 기반해 예상할 수 있는 결과입니다.

</details>


<details><summary>솔루션</summary>

```python
def tokens_to_latent_acts(
    token_scales: Float[Tensor, "batch seq"],
    tokens: Int[Tensor, "batch seq"],
    sae: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, tuple[Tensor]]:
    """
    Given scale factors for model's embeddings (i.e. scale factors applied after we compute the sum
    of positional and token embeddings), returns the SAE's latents.

    Returns:
        latent_acts_sparse: The SAE's latents in sparse form (i.e. the tensor of values)
        latent_acts_dense:  The SAE's latents in dense tensor, in a length-1 tuple
    """
    resid_after_embed = model(tokens, stop_at_layer=0)
    resid_after_embed = einops.einsum(resid_after_embed, token_scales, "... seq d_model, ... seq -> ... seq d_model")
    resid_before_sae = model(resid_after_embed, start_at_layer=0, stop_at_layer=_get_hook_layer(sae))

    sae_latents = sae.encode(resid_before_sae)
    sae_latents = SparseTensor.from_dense(sae_latents)

    return sae_latents.sparse[0], (sae_latents.dense,)


def token_to_latent_gradients(
    tokens: Float[Tensor, "batch seq"],
    sae: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, SparseTensor]:
    """
    Computes the gradients between an SAE's latents and all input tokens.

    Returns:
        token_latent_grads: The gradients between input tokens and SAE latents
        latent_acts:        The SAE's latent activations
    """
    # Find the gradients from token positions to latents
    token_scales = t.ones(tokens.shape, device=model.cfg.device, requires_grad=True)
    token_latent_grads, (latent_acts_dense,) = t.func.jacrev(tokens_to_latent_acts, has_aux=True)(
        token_scales, tokens, sae, model
    )

    token_latent_grads = einops.rearrange(token_latent_grads, "d_sae_nonzero batch seq -> batch seq d_sae_nonzero")

    latent_acts = SparseTensor.from_dense(latent_acts_dense)

    return (token_latent_grads, latent_acts)
```
</details>

### 연습 문제 - latent-to-logit gradient 구하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 30-45 minutes on this exercise.
> ```

마지막으로, latent-to-logit gradient를 계산해 보겠습니다. 이 연습 문제는 첫 번째 문제(즉, latent-to-latent gradient)와 매우 유사하지만, 이번에 `jacrev`을 통해 전달하는 함수는 SAE activation을 이후 SAE의 latent가 아닌 logit으로 매핑합니다. gradient를 계산할 상위 logit의 개수만을 지정할 수 있도록 `k` 인자를 제공했다는 점에 유의하십시오 (그렇지 않으면 매우 큰 gradient 행렬을 계산하게 되어 OOM 에러가 발생할 수 있기 때문입니다).

애초에 왜 latent-to-logit gradient를 계산하려고 하는 것일까요? 한 가지 분명한 답은 이것이 우리의 end-to-end circuit 그림을 완성해주기 때문입니다 (이제 token -> latent -> 다른 latent -> logit의 흐름을 갖게 되었습니다). 또 다른 답을 드리자면, latent가 이중적인 성격을 가지고 있다고 생각할 수 있습니다. 입력을 되돌아볼 때 latent는 **representation**이지만, logit을 향해 앞으로 나아갈 때는 **action**이 됩니다. 우리는 양방향 모두에서 sparsity가 나타날 것이라고 기대할 수 있습니다. 다시 말해, latent가 입력에 의해 생성된 activation을 희소하게 표현해야 할 뿐만 아니라, 출력에 영향을 주는 gradient에도 희소하게 영향을 미쳐야 한다는 것입니다. 이 연습 문제를 풀다 보면 알게 되겠지만, 이는 부분적으로 사실입니다. 다만 우리가 보았던 매우 희소한 token-to-latent 또는 latent-to-latent gradient 정도는 아닙니다. 그 이유 중 하나는 representation으로서의 sparsity는 이미 SAE의 loss function(L1 penalty)에 포함되어 있지만, latent gradient의 sparsity를 장려하는 명시적인 penalty 항은 없기 때문입니다. Anthropic은 그들의 [April 2024 update](https://transformer-circuits.pub/2024/april-update/index.html#attr-dl)에서 이것이 어떤 모습일지 제안합니다.

하지만 latent-to-logit gradient의 결과가 이전 두 연습 문제보다 덜 희소함에도 불구하고, 여전히 특정 입력 prompt에 대해 어떤 latent가 중요한지에 대해 많은 것을 알려줄 수 있습니다. 아래 함수들을 완성하고, 직접 latent-to-logit gradient를 실험해 보십시오!

In [ ]:
def latent_acts_to_logits(
    latent_acts_nonzero: Float[Tensor, " nonzero_acts"],
    latent_acts_nonzero_inds: Int[Tensor, "nonzero_acts n_indices"],
    latent_acts_shape: tuple[int, ...],
    sae: SAE,
    model: HookedSAETransformer,
    token_ids: list[int] | None = None,
) -> tuple[Tensor, tuple[Tensor]]:
    """
    Computes the logits as a downstream function of the SAE's reconstructed residual stream. If we
    supply `token_ids`, it means we only compute & return the logits for those specified tokens.
    """
    ...
    return logits_recon[token_ids], (logits_recon,)


def latent_to_logit_gradients(
    tokens: Float[Tensor, "batch seq"],
    sae: SAE,
    model: HookedSAETransformer,
    k: int | None = None,
) -> tuple[Tensor, Tensor, Tensor, list[int] | None, SparseTensor]:
    """
    Computes the gradients between active latents and some top-k set of logits (we
    use k to avoid having to compute the gradients for all tokens).

    Returns:
        latent_logit_gradients:  The gradients between the SAE's active latents & downstream logits
        logits:                  The model's true logits
        logits_recon:            The model's reconstructed logits (i.e. based on SAE reconstruction)
        token_ids:               The tokens we computed the gradients for
        latent_acts:             The SAE's latent activations
    """
    assert tokens.shape[0] == 1, "Only supports batch size 1 for now"

    ...

    return (
        latent_logit_gradients,
        logits,
        logits_recon,
        token_ids,
        latent_acts,
    )

In [ ]:
tests.test_latent_to_logit_gradients(latent_to_logit_gradients, gpt2, gpt2_saes)

layer = 9
prompt = "The Eiffel tower is in the city of"
answer = " Paris"

tokens = gpt2.to_tokens(prompt, prepend_bos=True)
str_toks = gpt2.to_str_tokens(prompt, prepend_bos=True)
k = 25

# Test the model on this prompt, with & without SAEs
test_prompt(prompt, answer, gpt2)

# How about the reconstruction? More or less; it's rank 20 so still decent
gpt2_saes[layer].use_error_term = False
with gpt2.saes(saes=[gpt2_saes[layer]]):
    test_prompt(prompt, answer, gpt2)

latent_logit_grads, logits, logits_recon, token_ids, latent_acts = latent_to_logit_gradients(
    tokens, sae=gpt2_saes[layer], model=gpt2, k=k
)

# sort by most positive in " Paris" direction
sorted_indices = latent_logit_grads[0].argsort(descending=True)
latent_logit_grads = latent_logit_grads[:, sorted_indices]

fig = px.imshow(
    to_numpy(latent_logit_grads),
    color_continuous_midpoint=0.0,
    color_continuous_scale="RdBu",
    x=[
        f"{str_toks[seq]!r} ({seq}), latent {latent:05}" for (_, seq, latent) in latent_acts.indices[sorted_indices]
    ],
    y=[f"{tok!r} ({gpt2.to_single_str_token(tok)})" for tok in token_ids],
    labels={"x": f"Features in layer {layer}", "y": "Logits"},
    title=f'Gradients between SAE latents in layer {layer} and final logits (only showing top {k} logits)<br><sup>   Prompt: "{"".join(str_toks)}"</sup>',
    width=1900,
    height=800,
    aspect="auto",
)
fig.show()

<details>
<summary>몇 가지 관찰 결과</summary>

feature `F9.22250`이 다른 상위 예측들보다 `" Paris"` token을 훨씬 더 많이 boosting하는 것을 볼 수 있습니다. 조사 결과, 이 feature는 주로 프랑스어 텍스트에서 활성화되며, 이는 타당한 결과입니다!

또한 독일과 관련된 단어들(예: Berlin, Hamberg, Cologne, Zurich)을 강하게 boosting하는 것으로 보이는 `F9.5879`도 확인할 수 있습니다. 여기서도 유사한 패턴이 나타나는데, 해당 feature는 주로 독일어 텍스트(또는 더 흔하게는 독일에 대해 이야기하는 영어 텍스트)에서 활성화됩니다.

```python
display_dashboard(sae_id="blocks.9.hook_resid_pre", latent_idx=22250)
display_dashboard(sae_id="blocks.9.hook_resid_pre", latent_idx=5879)
```

</details>


<details><summary>솔루션</summary>

```python
def latent_acts_to_logits(
    latent_acts_nonzero: Float[Tensor, " nonzero_acts"],
    latent_acts_nonzero_inds: Int[Tensor, "nonzero_acts n_indices"],
    latent_acts_shape: tuple[int, ...],
    sae: SAE,
    model: HookedSAETransformer,
    token_ids: list[int] | None = None,
) -> tuple[Tensor, tuple[Tensor]]:
    """
    Computes the logits as a downstream function of the SAE's reconstructed residual stream. If we
    supply `token_ids`, it means we only compute & return the logits for those specified tokens.
    """
    # Convert to dense, map through SAE decoder
    latent_acts = SparseTensor.from_sparse((latent_acts_nonzero, latent_acts_nonzero_inds, latent_acts_shape)).dense

    resid = sae.decode(latent_acts)

    # Map through model layers, to the end
    logits_recon = model(resid, start_at_layer=_get_hook_layer(sae))[0, -1]

    return logits_recon[token_ids], (logits_recon,)


def latent_to_logit_gradients(
    tokens: Float[Tensor, "batch seq"],
    sae: SAE,
    model: HookedSAETransformer,
    k: int | None = None,
) -> tuple[Tensor, Tensor, Tensor, list[int] | None, SparseTensor]:
    """
    Computes the gradients between active latents and some top-k set of logits (we
    use k to avoid having to compute the gradients for all tokens).

    Returns:
        latent_logit_gradients:  The gradients between the SAE's active latents & downstream logits
        logits:                  The model's true logits
        logits_recon:            The model's reconstructed logits (i.e. based on SAE reconstruction)
        token_ids:               The tokens we computed the gradients for
        latent_acts:             The SAE's latent activations
    """
    assert tokens.shape[0] == 1, "Only supports batch size 1 for now"

    acts_hook_name = f"{sae.cfg.metadata.hook_name}.hook_sae_acts_post"
    sae.use_error_term = True

    with t.no_grad():
        # Run model up to the position of the first SAE to get those residual stream activations
        logits, cache = model.run_with_cache_with_saes(
            tokens,
            names_filter=[acts_hook_name],
            saes=[sae],
            remove_batch_dim=False,
        )
        latent_acts = cache[acts_hook_name]
        latent_acts = SparseTensor.from_dense(latent_acts)

        logits = logits[0, -1]

    # Get the tokens we'll actually compute gradients for
    token_ids = None if k is None else logits.topk(k=k).indices.tolist()

    # Compute jacobian between latent acts and logits
    latent_logit_gradients, (logits_recon,) = t.func.jacrev(latent_acts_to_logits, has_aux=True)(
        *latent_acts.sparse, sae, model, token_ids
    )

    sae.use_error_term = False

    return (
        latent_logit_gradients,
        logits,
        logits_recon,
        token_ids,
        latent_acts,
    )
```
</details>

### 연습 문제 (선택 사항) - attention SAE에서 induction circuit 찾기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 30-60 minutes on this exercise.
> ```

지난 3가지 연습 세트에서 residual stream latent를 연구했던 것과 거의 동일한 방식으로 MLP 또는 attention latent를 연구할 수 있습니다. 예를 들어, MLP layer로 학습된 이전 SAE에 대해 logit 또는 이후의 SAE의 gradient를 계산하고 싶다면, MLP layer의 activation을 해당 이전 SAE의 decoded activation으로 교체한 다음, 이 이전 SAE의 activation에 대한 downstream 값들의 Jacobian을 계산하면 됩니다. 이때 `model.run_with_hooks`과 같은 기능을 사용하면 모델의 forward pass의 모든 단계를 수동으로 수행하지 않고도 이러한 교체를 수행할 수 있습니다.

attention SAE에 작동하는 `latent_to_latent_gradients_attn` 함수의 버전을 작성해 보십시오 (docstring은 아래에 있습니다). 이 함수를 사용하여 서로에 대해 0이 아닌 gradient를 가지며, 함께 induction circuit을 형성하는 previous token latent와 induction latent를 찾을 수 있습니까?

<details>
<summary>힌트 - 찾아봐야 할 곳</summary>

먼저 무작위 token 시퀀스를 생성하고, `circuitvis`을 사용하여 attention pattern을 시각화하는 것부터 시작하십시오:

```python
import circuitsvis as cv

seq_len = 10
tokens = t.randint(0, model.cfg.d_vocab, (1, seq_len)).tolist()[0]
tokens = t.tensor([model.tokenizer.bos_token_id] + tokens + tokens)

_, cache = model.run_with_cache(tokens)

prev_token_heads = [(4, 11)]
induction_heads = [(5, 1), (5, 5), (5, 8)]
all_heads = prev_token_heads + induction_heads

html = cv.attention.attention_patterns(
    tokens=model.to_str_tokens(tokens),
    attention=t.stack([cache["pattern", layer][0, head] for layer, head in all_heads]),
    attention_head_names=[f"{layer}.{head}" for (layer, head) in all_heads],
)
display(html)
```

이를 통해 layer 4에는 명확한 previous token head가 포함되어 있고, layer 5에는 여러 개의 induction head가 포함되어 있음을 알 수 있습니다. 따라서 layer 4와 layer 5 사이의 latent-to-latent gradient를 살펴보는 것이 좋습니다.

기억하십시오 - induction circuit은 `AB...AB`과 같은 시퀀스에서 작동하며, 여기서 previous token head는 첫 번째 `B`에서 첫 번째 `A`으로 attention을 보내고, 그 다음 induction head는 두 번째 `A`에서 첫 번째 `B`로 attention을 보냅니다. latent-to-latent gradient heatmap에서 induction circuit의 증거를 찾을 때 이 점을 염두에 두십시오.

</details>

이 함수 아래의 코드는 latent-to-latent gradient를 플롯하며, `AB...AB` 패턴에서 layer-4 feature가 첫 번째 `B`에서 활성화되고 layer-5 feature가 두 번째 `A`에서 활성화되는 사례에 검은색 사각형을 추가합니다 (이것이 우리가 induction circuit에서 기대하는 모습입니다). 다시 말해, 함수가 제대로 작동한다면 이 사각형들이 표시된 영역에서 0이 아닌 값들의 패턴이 보여야 합니다.

In [ ]:
def latent_acts_to_later_latent_acts_attn(
    latent_acts_nonzero: Float[Tensor, " nonzero_acts"],
    latent_acts_nonzero_inds: Int[Tensor, "nonzero_acts n_indices"],
    latent_acts_shape: tuple[int, ...],
    sae_from: SAE,
    sae_to: SAE,
    model: HookedSAETransformer,
    resid_pre_clean: Tensor,
) -> tuple[Tensor, Tensor]:
    """
    Returns the latent activations of an attention SAE, computed downstream of an earlier SAE's
    output (whose values are given in sparse form as the first three arguments).

    `resid_pre_clean` is also supplied, i.e. these are the input values to the attention layer in
    which the earlier SAE is applied.
    """
    ...

    return latent_acts_next_recon.sparse[0], (latent_acts_next_recon.dense,)


def latent_to_latent_gradients_attn(
    tokens: Float[Tensor, "batch seq"],
    sae_from: SAE,
    sae_to: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, SparseTensor, SparseTensor, SparseTensor]:
    """
    Computes the gradients between all active pairs of latents belonging to two SAEs. Both SAEs
    are assumed to be attention SAEs, i.e. they take the concatenated z values as input.

    Returns:
        latent_latent_gradients:  The gradients between all active pairs of latents
        latent_acts_prev:          The latent activations of the first SAE
        latent_acts_next:          The latent activations of the second SAE
        latent_acts_next_recon:    The reconstructed latent activations of the second SAE
    """
    ...

    return (
        latent_latent_gradients,
        latent_acts_prev,
        latent_acts_next,
        latent_acts_next_recon,
    )

In [ ]:
attn_saes = {
    layer: SAE.from_pretrained(
        "gpt2-small-hook-z-kk",
        f"blocks.{layer}.hook_z",
        device=device,
        dtype=dtype,
    )
    for layer in range(gpt2.cfg.n_layers)
}

seq_len = 10  # higher seq len / more batches would be more reliable, but this simplifies the plot
tokens = t.randint(0, gpt2.cfg.d_vocab, (1, seq_len)).tolist()[0]
tokens = t.tensor([gpt2.tokenizer.bos_token_id] + tokens + tokens)
str_toks = gpt2.to_str_tokens(tokens)
layer_from = 4
layer_to = 5

# Get latent-to-latent gradients
with t.enable_grad():
    (
        latent_latent_gradients,
        latent_acts_prev,
        latent_acts_next,
        latent_acts_next_recon,
    ) = latent_to_latent_gradients_attn(tokens, attn_saes[layer_from], attn_saes[layer_to], gpt2)

# Verify that ~the same latents are active in both, and the MSE loss is small
nonzero_latents = [tuple(x) for x in latent_acts_next.indices.tolist()]
nonzero_latents_recon = [tuple(x) for x in latent_acts_next_recon.indices.tolist()]
alive_in_one_not_both = set(nonzero_latents) ^ set(nonzero_latents_recon)
print(f"# nonzero latents (true): {len(nonzero_latents)}")
print(f"# nonzero latents (reconstructed): {len(nonzero_latents_recon)}")
print(f"# latents alive in one but not both: {len(alive_in_one_not_both)}")

# Create initial figure
fig = px.imshow(
    to_numpy(latent_latent_gradients.T),
    color_continuous_midpoint=0.0,
    color_continuous_scale="RdBu",
    x=[f"F{layer_to}.{latent}, {str_toks[seq]!r} ({seq})" for (_, seq, latent) in latent_acts_next_recon.indices],
    y=[f"F{layer_from}.{latent}, {str_toks[seq]!r} ({seq})" for (_, seq, latent) in latent_acts_prev.indices],
    labels={"y": f"From layer {layer_from}", "x": f"To layer {layer_to}"},
    title=f'Gradients between SAE latents in layer {layer_from} and SAE latents in layer {layer_to}<br><sup>   Prompt: "{"".join(str_toks)}"</sup>',
    width=1200,
    height=1000,
)
# Add rectangles to it, to cover the blocks where the layer 4 & 5 positions correspond to what we
# expect for the induction circuit
for first_B_posn in range(2, seq_len + 2):
    second_A_posn = first_B_posn + seq_len - 1
    x0 = (latent_acts_next_recon.indices[:, 1] < second_A_posn).sum().item()
    x1 = (latent_acts_next_recon.indices[:, 1] <= second_A_posn).sum().item()
    y0 = (latent_acts_prev.indices[:, 1] < first_B_posn).sum().item()
    y1 = (latent_acts_prev.indices[:, 1] <= first_B_posn).sum().item()
    fig.add_shape(type="rect", x0=x0, y0=y0, x1=x1, y1=y1)

fig.show()

<details><summary>솔루션</summary>

```python
def latent_acts_to_later_latent_acts_attn(
    latent_acts_nonzero: Float[Tensor, " nonzero_acts"],
    latent_acts_nonzero_inds: Int[Tensor, "nonzero_acts n_indices"],
    latent_acts_shape: tuple[int, ...],
    sae_from: SAE,
    sae_to: SAE,
    model: HookedSAETransformer,
    resid_pre_clean: Tensor,
) -> tuple[Tensor, Tensor]:
    """
    Returns the latent activations of an attention SAE, computed downstream of an earlier SAE's
    output (whose values are given in sparse form as the first three arguments).

    `resid_pre_clean` is also supplied, i.e. these are the input values to the attention layer in
    which the earlier SAE is applied.
    """
    # Convert to dense, map through SAE decoder
    latent_acts = SparseTensor.from_sparse((latent_acts_nonzero, latent_acts_nonzero_inds, latent_acts_shape)).dense
    z_recon = sae_from.decode(latent_acts)

    hook_name_z_prev = get_act_name("z", _get_hook_layer(sae_from))
    hook_name_z_next = get_act_name("z", _get_hook_layer(sae_to))

    def hook_set_z_prev(z: Tensor, hook: HookPoint):
        return z_recon

    def hook_store_z_next(z: Tensor, hook: HookPoint):
        hook.ctx["z"] = z

    # fwd pass: replace earlier z with SAE reconstructions, and store later z (no SAEs needed yet)
    model.run_with_hooks(
        resid_pre_clean,
        start_at_layer=_get_hook_layer(sae_from),
        stop_at_layer=_get_hook_layer(sae_to) + 1,
        fwd_hooks=[
            (hook_name_z_prev, hook_set_z_prev),
            (hook_name_z_next, hook_store_z_next),
        ],
    )
    z = model.hook_dict[hook_name_z_next].ctx.pop("z")
    latent_acts_next_recon = SparseTensor.from_dense(sae_to.encode(z))

    return latent_acts_next_recon.sparse[0], (latent_acts_next_recon.dense,)


def latent_to_latent_gradients_attn(
    tokens: Float[Tensor, "batch seq"],
    sae_from: SAE,
    sae_to: SAE,
    model: HookedSAETransformer,
) -> tuple[Tensor, SparseTensor, SparseTensor, SparseTensor]:
    """
    Computes the gradients between all active pairs of latents belonging to two SAEs. Both SAEs
    are assumed to be attention SAEs, i.e. they take the concatenated z values as input.

    Returns:
        latent_latent_gradients:  The gradients between all active pairs of latents
        latent_acts_prev:          The latent activations of the first SAE
        latent_acts_next:          The latent activations of the second SAE
        latent_acts_next_recon:    The reconstructed latent activations of the second SAE
    """
    resid_pre_name = get_act_name("resid_pre", _get_hook_layer(sae_from))
    acts_prev_name = f"{sae_from.cfg.metadata.hook_name}.hook_sae_acts_post"
    acts_next_name = f"{sae_to.cfg.metadata.hook_name}.hook_sae_acts_post"
    sae_from.use_error_term = True  # so we can get both true latent acts at once
    sae_to.use_error_term = True  # so we can get both true latent acts at once

    with t.no_grad():
        # Get the true activations for both SAEs
        _, cache = model.run_with_cache_with_saes(
            tokens,
            names_filter=[resid_pre_name, acts_prev_name, acts_next_name],
            stop_at_layer=_get_hook_layer(sae_to) + 1,
            saes=[sae_from, sae_to],
            remove_batch_dim=False,
        )
        latent_acts_prev = SparseTensor.from_dense(cache[acts_prev_name])
        latent_acts_next = SparseTensor.from_dense(cache[acts_next_name])

    # Compute jacobian between earlier and later latent activations (and also get the activations
    # of the later SAE which are downstream of the earlier SAE's reconstructions)
    latent_latent_gradients, (latent_acts_next_recon_dense,) = t.func.jacrev(
        latent_acts_to_later_latent_acts_attn, has_aux=True
    )(*latent_acts_prev.sparse, sae_from, sae_to, model, cache[resid_pre_name])

    latent_acts_next_recon = SparseTensor.from_dense(latent_acts_next_recon_dense)

    # Set SAE state back to default
    sae_from.use_error_term = False
    sae_to.use_error_term = False

    return (
        latent_latent_gradients,
        latent_acts_prev,
        latent_acts_next,
        latent_acts_next_recon,
    )
```
</details>

참고 - 이 데이터에 대한 SAE의 재구성 성능은 이 하위 섹션의 이전 연습 문제들보다 낮게 나타날 것입니다 (활성화된 feature들의 교집합 기준으로 측정할 경우). 제 추측으로는, induction 시퀀스가 SAE가 학습한 분포에 대해 더 OOD이기 때문입니다 (말 그대로 무작위 token들이기 때문입니다). 또한, 더 큰 데이터 batch에 대해 측정하고 전체 token의 일정 비율 이상에서 활성화되는 모든 feature를 추출한다면, 노이즈가 적은 결과를 얻을 수 있을 것입니다.

다음은 layer-5 feature들 중 시퀀스의 후반부 모든 token에서 활성화되는 것들만 필터링하고, layer-4 feature들 중 전반부에서 활성화되는 것들만 살펴보는 코드입니다. 서로 큰 gradient를 가지는 개별 feature 쌍들을 조사해 보십시오. 이들이 각각 previous token feature와 induction feature처럼 보입니까?

In [ ]:
# Filter for layer-5 latents which are active on every token in the second half (which induction
# latents should be!)
acts_on_second_half = latent_acts_next_recon.indices[latent_acts_next_recon.indices[:, 1] >= seq_len + 1]
c = Counter(acts_on_second_half[:, 2].tolist())
top_feats = sorted([feat for feat, count in c.items() if count >= seq_len])
print(f"Layer 5 SAE latents which fired on all tokens in the second half: {top_feats}")
mask_next = (latent_acts_next_recon.indices[:, 2] == t.tensor(top_feats, device=device)[:, None]).any(dim=0) & (
    latent_acts_next_recon.indices[:, 1] >= seq_len + 1
)

# Filter the layer-4 axis to only show activations at sequence positions that we expect to be used
# in induction
mask_prev = (latent_acts_prev.indices[:, 1] >= 1) & (latent_acts_prev.indices[:, 1] <= seq_len)

# Filter the y-axis, just to these
fig = px.imshow(
    to_numpy(latent_latent_gradients[mask_next][:, mask_prev]),
    color_continuous_midpoint=0.0,
    color_continuous_scale="RdBu",
    y=[
        f"{str_toks[seq]!r} ({seq}), #{latent:05}" for (_, seq, latent) in latent_acts_next_recon.indices[mask_next]
    ],
    x=[f"{str_toks[seq]!r} ({seq}), #{latent:05}" for (_, seq, latent) in latent_acts_prev.indices[mask_prev]],
    labels={"x": f"From layer {layer_from}", "y": f"To layer {layer_to}"},
    title=f'Gradients between SAE latents in layer {layer_from} and SAE latents in layer {layer_to}<br><sup>   Prompt: "{"".join(str_toks)}"</sup>',
    width=1800,
    height=500,
)
fig.show()

<details>
<summary>몇 가지 관찰 결과</summary>

엄격하게 확인하지는 않았으며 (코드 정리도 많이 필요합니다!), layer-5에서 가장 두드러진 2개의 feature (35425, 36126)가 확실히 induction feature라는 것을 발견할 수 있었습니다. 또한 첫 번째 feature와 layer 4의 feature들이 강하게 compose된 것 중 3/4은 previous token feature인 것으로 보입니다:

```python
for (layer, latent_idx) in [(5, 35425), (5, 36126), (4, 22975), (4, 21020), (4, 23954)]:
    display_dashboard(
        sae_release="gpt2-small-hook-z-kk",
        sae_id=f"blocks.{layer}.hook_z",
        latent_idx=latent_idx,
    )
```

</details>

# 2️⃣ Transcoders

> ##### 학습 목표
>
> - transcoder의 개념과 표준 SAE와의 차이점을 이해합니다.
> - transcoder latent를 해석하기 위한 기법인 **pullbacks**, **de-embeddings**, 그리고 **extended embeddings**를 학습합니다.
> - **blind case study**를 통해 activation 예시 없이 circuit-level 분석만으로 transcoder latent를 해석하는 과정을 실습합니다.

## 서론

우리가 살펴본 MLP-layer SAE는 activation을 latent 벡터들의 희소 선형 결합(sparse linear combination)으로 표현하려고 시도합니다. 중요한 점은, 이들이 **모델의 단일 지점**에서의 activation에 대해서만 작동한다는 것입니다. 이들은 실제로 MLP layer의 연산을 수행하는 법을 배우는 것이 아니라, 그 연산의 결과를 재구성(reconstruct)하는 법을 배웁니다. 표준 SAE를 사용해서는 superposition 상태인 MLP layer에 대해 가중치 기반 분석을 수행하기가 매우 어렵습니다. 많은 latent들이 neuron basis에서 매우 조밀(dense)하며, 이는 neuron들을 분해하기 어렵다는 것을 의미하기 때문입니다.

이와 대조적으로, **transcoder**는 MLP layer 이전의 activation(즉, 정규화되었을 가능성이 있는 residual stream 값들)을 입력으로 받아, 해당 MLP layer의 post-MLP activation을 다시 latent 벡터들의 희소 선형 결합으로 표현하는 것을 목표로 합니다. transcoder라는 용어가 가장 일반적이지만, 이들은 **input-output SAE**(기본 모델 layer의 입력을 받아 출력을 학습하려고 하기 때문) 또는 **predicting future activations**(명백한 이유로)라고도 불립니다. 엄밀히 말하면 transcoder는 재구성이 아닌 매핑을 학습하는 것이므로 autoencoder가 아니지만, SAE에서 얻은 많은 직관이 transcoder에도 그대로 적용됩니다.

왜 transcoder가 표준 SAE보다 개선된 방식일 수 있을까요? 주로, 모델 layer의 기능에 대해 훨씬 더 명확한 통찰을 제공하기 때문입니다. [Transcoders LessWrong post](https://www.lesswrong.com/posts/YmkjnWtZGLbHRbzrP/transcoders-enable-fine-grained-interpretable-circuit)에서 다음과 같이 설명합니다:

> transcoder의 강점 중 하나는 MLP layer의 기능을 희소하고, 독립적으로 변화하며, 의미 있는 단위들로 분해한다는 점입니다 (이는 superposition이 발견되기 전 neuron들이 원래 의도되었던 모습과 같습니다). 이는 circuit 분석을 상당히 단순화합니다.
>
> ...
>
> 비유를 들자면, 우리가 이해하고 싶은 복잡하게 컴파일된 컴퓨터 프로그램이 있다고 가정해 보겠습니다 (_a la_ [Chris Olah’s analogy](https://transformer-circuits.pub/2022/mech-interp-essay/index.html)). SAE는 프로그램의 여러 위치에 중단점(breakpoint)을 설정하고 변수를 읽을 수 있게 해주는 디버거와 유사합니다. 반면, transcoder는 이 프로그램의 특정 서브루틴을 인간이 해석 가능한 근사치로 교체하는 도구와 유사합니다.

직관적으로는 transcoder가 단순히 출력을 재현하는 것이 아니라 MLP의 연산을 모방하려 하기 때문에, 더 다른(더 복잡한) 종류의 최적화 문제를 해결하는 것처럼 보일 수 있으며, 따라서 표준 SAE에 비해 성능상의 트레이드오프가 있을 것이라고 생각할 수 있습니다. 하지만 여러 증거는 그렇지 않을 수 있음을 시사하며, transcoder가 표준 SAE보다 파레토 개선(pareto improvement)을 제공할 수 있음을 보여줍니다.

먼저 transcoder를 로드하는 것부터 시작하겠습니다. transcoder는 SAE와 정확히 동일한 방식인 `SAE.from_pretrained`를 통해 로드합니다.

우리가 사용할 모델은 GPT-2의 8번째 MLP layer를 재구성하도록 학습되었습니다. 중요한 참고 사항은, MLP layer의 정규화된 입력을 가져와서 `mlp_out`(즉, residual stream에 다시 더해질 값들)를 출력하는 것에 대해 이야기하고 있다는 점입니다. 따라서 pre-MLP 및 post-MLP 값에 대해 언급할 때는 activation function의 전/후가 아니라 이를 의미합니다!

In [ ]:
gpt2 = HookedSAETransformer.from_pretrained("gpt2-small", device=device, dtype=dtype)

hf_repo_id = "callummcdougall/arena-demos-transcoder"
sae_id = "gpt2-small-layer-{layer}-mlp-transcoder-folded-b_dec_out"
gpt2_transcoders = {
    layer: SAE.from_pretrained(release=hf_repo_id, sae_id=sae_id.format(layer=layer), device=device, dtype=dtype)
    for layer in tqdm(range(9))
}

layer = 8
gpt2_transcoder = gpt2_transcoders[layer]
print("Transcoder hooks (same as regular SAE hooks):", gpt2_transcoder.hook_dict.keys())

# Load the sparsity values, and plot them
log_sparsity_path = hf_hub_download(hf_repo_id, f"{sae_id.format(layer=layer)}/log_sparsity.pt")
log_sparsity = t.load(log_sparsity_path, map_location="cpu", weights_only=True)
fig = px.histogram(
    to_numpy(log_sparsity), width=800, template="ggplot2", title="Transcoder latent sparsity"
).update_layout(showlegend=False)
fig.show()


live_latents = np.arange(len(log_sparsity))[to_numpy(log_sparsity > -4)]

# Get the activations store
gpt2_act_store = ActivationsStore.from_sae(
    model=gpt2,
    sae=gpt2_transcoders[layer],
    dataset="NeelNanda/pile-10k",
    streaming=True,
    store_batch_size_prompts=16,
    n_batches_in_buffer=32,
    device=device,
)
tokens = gpt2_act_store.get_batch_tokens()
assert tokens.shape == (gpt2_act_store.store_batch_size_prompts, gpt2_act_store.context_size)

다음으로, 실행 전에 transcoder에 `use_error_term` 설정을 처리하기 위해 `model.run_with_cache_with_saes`을 래핑한 헬퍼 함수를 제공해 드렸습니다. 기본값은 `use_error_term=True`이며, 이는 모델의 activation을 그대로 유지하고 transcoder activation만 캐싱함을 의미합니다.

In [ ]:
def run_with_cache_with_transcoder(
    model: HookedSAETransformer,
    transcoders: list[SAE],
    tokens: Tensor,
    use_error_term: bool = True,  # by default we don't intervene, just compute activations
) -> ActivationCache:
    """
    Runs MLP transcoder(s) on a batch of tokens, using native SAELens v6 transcoder support.

    If use_error_term=True (default), the model's activations are left intact and we just cache
    transcoder activations. If False, the model's MLP outputs are replaced with transcoder outputs.
    """
    prev_use_error_terms = [tc.use_error_term for tc in transcoders]
    for tc in transcoders:
        tc.use_error_term = use_error_term
    try:
        _, cache = model.run_with_cache_with_saes(tokens, saes=transcoders)
    finally:
        for tc, prev in zip(transcoders, prev_use_error_terms):
            tc.use_error_term = prev
    return cache

마지막으로, SAE 대시보드를 복제했던 이전 연습 세트에서 이미 접하셨을 함수들을 제공해 드렸습니다 (아직 이 연습들을 하지 않으셨다면, 강력히 추천합니다!). 유일한 차이점은 transcoder에서 `use_error_term` 설정을 처리하기 위해 `run_with_cache_with_transcoder` (`model.run_with_cache_with_saes`을 감싼 얇은 wrapper)를 사용한다는 점입니다.

In [ ]:
def get_k_largest_indices(
    x: Float[Tensor, "batch seq"], k: int, buffer: int = 0, no_overlap: bool = True
) -> Int[Tensor, "k 2"]:
    if buffer > 0:
        x = x[:, buffer:-buffer]
    indices = x.flatten().argsort(-1, descending=True)
    rows = indices // x.size(1)
    cols = indices % x.size(1) + buffer

    if no_overlap:
        unique_indices = t.empty((0, 2), device=x.device).long()
        while len(unique_indices) < k:
            unique_indices = t.cat((unique_indices, t.tensor([[rows[0], cols[0]]], device=x.device)))
            is_overlapping_mask = (rows == rows[0]) & ((cols - cols[0]).abs() <= buffer)
            rows = rows[~is_overlapping_mask]
            cols = cols[~is_overlapping_mask]
        return unique_indices

    return t.stack((rows, cols), dim=1)[:k]


def index_with_buffer(
    x: Float[Tensor, "batch seq"], indices: Int[Tensor, "k 2"], buffer: int | None = None
) -> Float[Tensor, " k *buffer_x2_plus1"]:
    rows, cols = indices.unbind(dim=-1)
    if buffer is not None:
        rows = einops.repeat(rows, "k -> k buffer", buffer=buffer * 2 + 1)
        cols[cols < buffer] = buffer
        cols[cols > x.size(1) - buffer - 1] = x.size(1) - buffer - 1
        cols = einops.repeat(cols, "k -> k buffer", buffer=buffer * 2 + 1) + t.arange(
            -buffer, buffer + 1, device=x.device
        )
    return x[rows, cols]


def display_top_seqs(data: list[tuple[float, list[str], int]]):
    table = Table("Act", "Sequence", title="Max Activating Examples", show_lines=True)
    for act, str_toks, seq_pos in data:
        formatted_seq = (
            "".join([f"[b u green]{str_tok}[/]" if i == seq_pos else str_tok for i, str_tok in enumerate(str_toks)])
            .replace("�", "")
            .replace("\n", "↵")
        )
        table.add_row(f"{act:.3f}", repr(formatted_seq))
    rprint(table)


def fetch_max_activating_examples(
    model: HookedSAETransformer,
    transcoder: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 100,
    k: int = 10,
    buffer: int = 10,
    display: bool = False,
) -> list[tuple[float, list[str], int]]:
    data = []

    for _ in tqdm(range(total_batches)):
        tokens = act_store.get_batch_tokens()
        cache = run_with_cache_with_transcoder(model, [transcoder], tokens)
        acts = cache[f"{transcoder.cfg.metadata.hook_name}.hook_sae_acts_post"][..., latent_idx]

        k_largest_indices = get_k_largest_indices(acts, k=k, buffer=buffer)
        tokens_with_buffer = index_with_buffer(tokens, k_largest_indices, buffer=buffer)
        str_toks = [model.to_str_tokens(toks) for toks in tokens_with_buffer]
        top_acts = index_with_buffer(acts, k_largest_indices).tolist()
        data.extend(list(zip(top_acts, str_toks, [buffer] * len(str_toks))))

    data = sorted(data, key=lambda x: x[0], reverse=True)[:k]
    if display:
        display_top_seqs(data)
    return data

latent 1을 선택하여, 우리의 결과와 neuronpedia 대시보드를 비교해 보겠습니다 (이 모델이 아직 SAELens에 등록되지 않았음에도 불구하고, neuronpedia 대시보드가 존재한다는 점에 유의하십시오).

In [ ]:
latent_idx = 1
neuronpedia_id = "gpt2-small/8-tres-dc"
url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"
display(IFrame(url, width=800, height=600))

fetch_max_activating_examples(
    gpt2, gpt2_transcoder, gpt2_act_store, latent_idx=latent_idx, total_batches=200, display=True
)

### Pullback & de-embeddings

이 섹션의 시작 부분에 있는 latent-latent gradient 연습 문제에서, 서로 다른 layer에 있는 임의의 두 latent가 어떻게 상호작용하는지 계산하는 것이 상당히 어려울 수 있다는 점을 확인했습니다. 실제로, 우리는 동일한 forward pass에서 모두 활성화된 latent 사이의 gradient만 계산할 수 있었습니다. 이를 해결하기 위해 우리가 시도해 볼 수 있었던 한 가지 방법은 한 latent의 "writing vector"와 다른 latent의 "reading vector"의 dot product를 구하는 것입니다. 예를 들어, SAE가 post-ReLU MLP activation으로 학습되었다고 가정하면, 다음과 같이 계산할 수 있습니다: `W_dec[:, f1] @ W_out[layer1] @ W_in[layer2] @ W_enc[f2, :]` (여기서 `f1`과 `f2`는 각각 이전과 이후의 latent index이고, `layer1`과 `layer2`는 SAE layer이며, `W_in`, `W_out`은 모든 layer의 MLP input 및 output weight matrix입니다). 이 공식의 의미를 살펴보면, `W_dec[:, f1] @ W_out[layer1]` 항은 첫 번째(이전) latent에 의해 residual stream에 더해지는 "writing vector"이며, 이를 `W_in[layer2] @ W_enc[f2, :]`과 dot product 하여 두 번째(이후) latent의 activation을 계산하게 됩니다. 하지만 한 가지 아쉬운 점은 이후 MLP layer의 ReLU 함수를 무시하고 있다는 것입니다 (SAE는 pre-ReLU가 아니라 post-ReLU activation을 재구성한다는 점을 기억하십시오). 이는 사소한 점처럼 보일 수 있지만, 실제로는 연산을 수행하는 layer에서 학습된 표준 SAE의 한계라는 핵심적인 부분과 연결됩니다. **SAE는 모델의 스냅샷을 재구성하고 있지만, layer의 실제 연산 과정에 대한 통찰력을 얻는 데는 도움을 주지 못하고 있습니다**.

transcoder는 여기서 어떻게 도움이 될까요? transcoder는 (비선형성을 포함하여) MLP layer 전체를 감싸고 있기 때문에, "writing vector"와 다운스트림의 "reading vector" 사이의 dot product를 직접 계산하여 특정 latent가 다른 latent를 활성화시키는지 파악할 수 있습니다 (layernorm은 무시합니다). 몇 가지 정의를 내리겠습니다:

- 어떤 이후 latent의 **pullback**은 $p = (W_{dec})^T f_{later}$입니다. 즉, 이후 latent 벡터(reading weight)와 이전 latent들의 모든 decoder weight(writing weight)의 dot product입니다.
- **de-embedding**은 특수한 경우입니다: $d = W_E f_{later}$. 즉, "어떤 이전 transcoder latent가 이후의 특정 latent를 활성화시키는가?"라고 묻는 대신, "어떤 token이 이후의 특정 latent를 최대한으로 활성화시키는가?"라고 묻는 것입니다.

원칙적으로 일반적인 MLP SAE에 대해서도 이 두 양을 모두 계산할 수 있습니다. 하지만 모델의 실제 연산과 정확히 일치하지 않으므로, 이를 통해 강력한 결론을 많이 도출하기는 어렵습니다.

(embeddings -> transcoders -> unembeddings)로 이어지는 circuit 그림을 완성하기 위해, transcoder latent에 대해서도 일반적인 SAE와 정확히 동일한 방식으로 logit lens를 계산할 수 있다는 점을 언급할 가치가 있습니다. 단순히 transcoder decoder 벡터와 unembedding matrix의 dot product를 구하면 됩니다. 이는 기본적으로 일반 SAE의 경우와 정당성 및 해석이 동일하므로, 새로운 용어를 만들 필요 없이 계속해서 logit lens라고 부르겠습니다!

### 연습 문제 - de-embedding 계산하기

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

아래 셀에서 이 latent에 대한 de-embedding을 계산해야 합니다 (즉, 어떤 token들이 이 latent를 가장 강하게 활성화시키는지 확인합니다). 가이드로 logit lens 함수를 사용할 수 있습니다 (이전 연습 문제에서 사용되었던 함수를 제공해 드렸습니다).

In [ ]:
def show_top_logits(
    model: HookedSAETransformer,
    sae: SAE,
    latent_idx: int,
    k: int = 10,
) -> None:
    """Displays the top & bottom logits for a particular latent."""
    logits = sae.W_dec[latent_idx] @ model.W_U

    pos_logits, pos_token_ids = logits.topk(k)
    pos_tokens = model.to_str_tokens(pos_token_ids)
    neg_logits, neg_token_ids = logits.topk(k, largest=False)
    neg_tokens = model.to_str_tokens(neg_token_ids)

    print(
        tabulate(
            zip(map(repr, neg_tokens), neg_logits, map(repr, pos_tokens), pos_logits),
            headers=["Bottom tokens", "Value", "Top tokens", "Value"],
            tablefmt="simple_outline",
            stralign="right",
            numalign="left",
            floatfmt="+.3f",
        )
    )


print(f"Top logits for transcoder latent {latent_idx}:")
show_top_logits(gpt2, gpt2_transcoder, latent_idx=latent_idx)


def show_top_deembeddings(model: HookedSAETransformer, sae: SAE, latent_idx: int, k: int = 10) -> None:
    """Displays the top & bottom de-embeddings for a particular latent."""
    raise NotImplementedError()


print(f"\nTop de-embeddings for transcoder latent {latent_idx}:")
show_top_deembeddings(gpt2, gpt2_transcoder, latent_idx=latent_idx)
tests.test_show_top_deembeddings(show_top_deembeddings, gpt2, gpt2_transcoder)

<details><summary>솔루션</summary>

```python
def show_top_logits(
    model: HookedSAETransformer,
    sae: SAE,
    latent_idx: int,
    k: int = 10,
) -> None:
    """Displays the top & bottom logits for a particular latent."""
    logits = sae.W_dec[latent_idx] @ model.W_U

    pos_logits, pos_token_ids = logits.topk(k)
    pos_tokens = model.to_str_tokens(pos_token_ids)
    neg_logits, neg_token_ids = logits.topk(k, largest=False)
    neg_tokens = model.to_str_tokens(neg_token_ids)

    print(
        tabulate(
            zip(map(repr, neg_tokens), neg_logits, map(repr, pos_tokens), pos_logits),
            headers=["Bottom tokens", "Value", "Top tokens", "Value"],
            tablefmt="simple_outline",
            stralign="right",
            numalign="left",
            floatfmt="+.3f",
        )
    )


def show_top_deembeddings(model: HookedSAETransformer, sae: SAE, latent_idx: int, k: int = 10) -> None:
    """Displays the top & bottom de-embeddings for a particular latent."""
    de_embeddings = model.W_E @ sae.W_enc[:, latent_idx]

    pos_logits, pos_token_ids = de_embeddings.topk(k)
    pos_tokens = model.to_str_tokens(pos_token_ids)
    neg_logits, neg_token_ids = de_embeddings.topk(k, largest=False)
    neg_tokens = model.to_str_tokens(neg_token_ids)

    print(
        tabulate(
            zip(map(repr, neg_tokens), neg_logits, map(repr, pos_tokens), pos_logits),
            headers=["Bottom tokens", "Value", "Top tokens", "Value"],
            tablefmt="simple_outline",
            stralign="right",
            numalign="left",
            floatfmt="+.3f",
        )
    )
```
</details>

이것은... 꽤 실망스럽습니다! 대시보드를 보면 가장 활성화되는 token이 `" goal"`이어야 한다는 것이 매우 분명해 보이는데, 왜 `"liga"`이나 `"jee"` 같은 이상한 단어들이 나오는 것일까요? 물론 `" scored"`이나 `" scoring"`처럼 말이 되는 단어들도 있지만, 전반적으로 이는 우리가 기대한 결과가 아닙니다.

여기서 무슨 일이 일어나고 있는지 추측할 수 있습니까? (다음 연습 문제의 설명을 읽으면 정답이 공개되므로, 계속 읽기 전에 먼저 생각해보시기 바랍니다!)

<details>
<summary>힌트</summary>

IOI ARENA 연습 문제를 풀었거나 (또는 IOI 논문을 읽었다면), 이 개념을 접해보셨을 것입니다. 이는 GPT2-Small의 architecture와 관련이 있습니다.

</details>

<details>
<summary>정답</summary>

GPT2-Small은 **tied embeddings**를 사용합니다. 즉, embedding matrix가 unembedding matrix의 transpose입니다. 이는 direct path가 bigram 빈도를 표현할 수 없음을 의미합니다 (예를 들어, bigram `Barack Obama`에 대해 `Obama Barack`보다 더 높은 logit을 가질 수 없습니다). 따라서 MLP layer가 개입하여 이 대칭성을 깨뜨려야 합니다. 특히 MLP0가 이 역할을 수행하는 것으로 보이며, 이것이 우리가 이를 **extended embedding** (또는 **effective embedding**)이라고 부르는 이유입니다.

그 결과, embedding matrix의 인덱싱된 행들은 모델이 실제로 특정 token의 embedding으로 학습하여 처리하고 있는 대상의 적절한 표현이 아니게 됩니다.

</details>

### 연습 문제 - de-embedding 함수 수정하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

위의 드롭다운에서 논의한 함수의 오류를 수정하기 위해, extended embedding을 계산하는 아래 함수를 완성해야 합니다.

extended embedding을 계산하는 방법은 여러 가지가 있습니다 (예: 때로는 attention layer를 포함하고 항상 self-attend한다고 가정하거나, 때로는 MLP0의 출력만 사용하거나, 때로는 이를 raw embedding에 더하거나, 때로는 더 정확하게 만들기 위해 BOS token을 사용합니다). 이러한 방법들 대부분은 비슷한 품질의 결과를 낼 것입니다 (정확히 어떻게 포함하느냐보다 MLP0를 포함하는 것 자체가 훨씬 더 중요합니다). 하지만 테스트를 위해 다음 방법을 사용해야 합니다:

- embedding matrix를 가져옵니다.
- 여기에 layernorm을 적용합니다 (즉, 각 token의 embedding 벡터가 unit std dev를 갖도록 스케일링합니다).
- 여기에 MLP0를 적용합니다 (즉, 각 token의 정규화된 embedding 벡터에 개별적으로 적용합니다).
- 그 결과를 원래의 embedding matrix에 다시 더합니다.

팁 - layernorm과 MLP를 위한 개별 연산을 직접 작성하는 대신, 각각 `model.blocks[layer].ln2` 또는 `.mlp`의 forward 메서드를 사용할 수 있습니다.

In [ ]:
def create_extended_embedding(model: HookedTransformer) -> Float[Tensor, "d_vocab d_model"]:
    """
    Creates the extended embedding matrix using the model's layer-0 MLP, and the method described
    in the exercise above.

    You should also divide the output by its standard deviation across the `d_model` dimension
    (this is because that's how it'll be used later e.g. when fed into the MLP layer / transcoder).
    """
    raise NotImplementedError()


tests.test_create_extended_embedding(create_extended_embedding, gpt2)

<details><summary>솔루션</summary>

```python
def create_extended_embedding(model: HookedTransformer) -> Float[Tensor, "d_vocab d_model"]:
    """
    Creates the extended embedding matrix using the model's layer-0 MLP, and the method described
    in the exercise above.

    You should also divide the output by its standard deviation across the `d_model` dimension
    (this is because that's how it'll be used later e.g. when fed into the MLP layer / transcoder).
    """
    W_E = model.W_E.clone()[:, None, :]  # shape [batch=d_vocab, seq_len=1, d_model]

    mlp_output = model.blocks[0].mlp(model.blocks[0].ln2(W_E))  # shape [batch=d_vocab, seq_len=1, d_model]

    W_E_ext = (W_E + mlp_output).squeeze()
    return (W_E_ext - W_E_ext.mean(dim=-1, keepdim=True)) / W_E_ext.std(dim=-1, keepdim=True)
```
</details>

해당 테스트들을 통과했다면, 확장된 embedding을 사용하도록 `show_top_deembeddings`을 다시 작성해 보십시오. 결과가 더 좋게 보이나요? (힌트 - 더 좋게 나와야 합니다!)

참고 - 결과의 크기가 놀라울 정도로 크게 느껴지더라도 걱정하지 마십시오. MLP 이전에 normalization 단계가 적용되므로, 실제 activation은 생성될 표의 값들이 시사하는 것보다 더 작을 것입니다.

In [ ]:
def show_top_deembeddings_extended(model: HookedSAETransformer, sae: SAE, latent_idx: int, k: int = 10) -> None:
    """Displays the top & bottom de-embeddings for a particular latent."""
    raise NotImplementedError()


tests.test_show_top_deembeddings_extended(show_top_deembeddings_extended, gpt2, gpt2_transcoder)

print(f"Top de-embeddings (extended) for transcoder latent {latent_idx}:")
show_top_deembeddings_extended(gpt2, gpt2_transcoder, latent_idx=latent_idx)

<details><summary>솔루션</summary>

```python
def show_top_deembeddings_extended(model: HookedSAETransformer, sae: SAE, latent_idx: int, k: int = 10) -> None:
    """Displays the top & bottom de-embeddings for a particular latent."""
    de_embeddings = create_extended_embedding(model) @ sae.W_enc[:, latent_idx]

    pos_logits, pos_token_ids = de_embeddings.topk(k)
    pos_tokens = model.to_str_tokens(pos_token_ids)
    neg_logits, neg_token_ids = de_embeddings.topk(k, largest=False)
    neg_tokens = model.to_str_tokens(neg_token_ids)

    print(
        tabulate(
            zip(map(repr, neg_tokens), neg_logits, map(repr, pos_tokens), pos_logits),
            headers=["Bottom tokens", "Value", "Top tokens", "Value"],
            tablefmt="simple_outline",
            stralign="right",
            numalign="left",
            floatfmt="+.3f",
        )
    )
```
</details>

### Blind case study

이 과정은 이 섹션에서 배운 모든 내용을 실제로 적용해 보기 위해 설계된 개방형 탐색입니다. 이는 도전적인 과제이며 단 하나의 정답이 존재하지 않습니다. 따라서 수렴해야 할 정답이 있는 문제라기보다 연구 연습으로 생각하시기 바랍니다.

[post introducing transcoders](https://www.lesswrong.com/posts/YmkjnWtZGLbHRbzrP/transcoders-enable-fine-grained-interpretable-circuit)의 저자들은 **blind case study**라는 아이디어를 제시합니다. 그들의 포스트를 인용하자면 다음과 같습니다:

> ...우리는 어떤 transcoder에 일부 latent를 가지고 있으며, 이 latent를 활성화시키는 예시들을 보지 않고 이 transcoder latent를 해석하고자 합니다. 우리의 목표는 대신 위에서 설명한 input-independent 및 input-dependent circuit 분석 방법만을 사용하여, 해당 latent가 언제 활성화되는지에 대한 가설을 세우는 것입니다.

**input-independent circuit analysis**란 pullback 및 de-embedding과 같은 것들(즉, 모델과 transcoder의 weight에만 의존하는 함수인 것들)을 의미합니다. **input-dependent**는 구체적으로 **input-dependent influence**를 의미하며, 이는 이전 transcoder로의 pullback과 해당 이전 transcoder의 post-ReLU activation의 요소별 곱(elementwise product)으로 정의됩니다. 다시 말해, 이는 이전 latent들이 발화(fire)할 때 어떤 이전 latent들이 이후의 latent에 영향을 줄 것인지뿐만 아니라, 특정 input에서 실제로 어떤 것들이 이후의 latent에 영향을 *주는지*(즉, 실제로 어떤 것들이 발화했는지를 고려함)를 알려줍니다.

이것의 동기는 무엇일까요? 결국 우리는 latent가 단순히 데이터의 특정 latent에 반응하는 개별 단위로서가 아니라, 복잡한 circuit 내에 나타날 때 이를 이해할 수 있기를 원합니다. 그리고 그 과정의 일부는 다른 latent(또는 input의 특정 token)와의 연결성만을 기반으로 주어진 latent가 무엇을 하고 있는지에 대한 가설을 세울 수 있는 능력을 포함해야 합니다. 상위 활성화 예시들을 직접 살펴보는 것이 분명 도움이 될 수 있지만, 이는 때때로 [misleading](https://www.lesswrong.com/posts/3zBsxeZzd3cvuueMJ/paper-a-is-for-absorption-studying-latent-splitting-and) 일 뿐만 아니라, latent가 *무엇*을 하고 있는지는 알려줄 수 있어도 *왜* 그렇게 하는지에 대한 통찰은 많이 제공하지 못합니다.

규칙을 명확히 하자면 다음과 같습니다:

- 특정 예시 prompt의 특정 token에서 latent의 activation을 확인할 수 없습니다.
- input-dependent 분석을 사용할 수 있습니다. 예를 들어, 특정 input에서 일부 이전 latent가 타겟 latent에 미치는 influence를 분석할 수 있습니다 (단, input은 token이 아닌 token ID 형태로 유지해야 합니다. latent를 활성화시키는 prompt의 실제 내용을 확인하는 것은 부정행위이기 때문입니다).
- input-independent 분석을 사용할 수 있습니다. 예를 들어, latent의 de-embedding이나 logit lens를 사용할 수 있습니다.

우리는 이를 매우 개방형 연습으로 만들었습니다. 위에서 일부 함수를 제공했지만, 분석에 가장 유용하다고 생각되는 것에 따라 다른 함수들은 직접 작성해야 할 수도 있습니다 (예를 들어, pullback을 계산하는 함수는 아직 제공하지 않았습니다). 더 쉬운 연습을 원하신다면 포스트에서 성공적으로 역공학(reverse-engineer)한 latent(예: latent 355, transcoder의 300번째 live latent)를 사용할 수 있으며, 도전을 원하신다면 latent 479(transcoder의 400번째 live latent이며, 저자들이 초기 포스트에서 역공학에 실패한 것)를 시도해 볼 수 있습니다.

조금 더 쉬운 버전을 원하신다면, 가설을 테스트하기 위해 직접 시퀀스를 모델에 입력하는 것을 허용하는 규칙 완화를 적용할 수 있습니다 (단, 대규모 데이터셋에서 상위 활성화 시퀀스를 찾아 이를 디코딩하는 것과 같은 행위는 금지됩니다). 이를 통해 행동 공간에 어느 정도 제한을 둔 상태에서 가설을 테스트할 수 있습니다.

In [ ]:
blind_study_latent = 479

layer = 8
gpt2_transcoder = gpt2_transcoders[layer]

# YOUR CODE HERE!

아래의 드롭다운을 클릭하여 제가 이 연습 문제를 어떻게 풀었는지 확인하시거나, 이 latent에 대한 저자들의 blind case study 해석 과정이 담긴 [this notebook](https://github.com/jacobdunefsky/transcoder_circuits/blob/master/case_study_local_context.ipynb)를 읽어보실 수 있습니다. 다만, 제목에서 문제의 일부가 드러날 수 있으므로 충분히 스스로 시도해 본 후에 노트북을 방문하시기 바랍니다!

<details>
<summary>나의 시도</summary>

저의 접근 방식은 네 단계의 분석 과정을 거치며 가설을 세우고 각 단계에서 이를 정교화하는 것이었습니다.

#### 1단계: De-embeddings 및 logit lens

입력과 무관한 분석부터 시작합니다. 이 latent가 어떤 token으로부터 읽어오고(de-embeddings), 어떤 token에 쓰는지(logit lens) 확인합니다.

```python
# (1) look at de-embedding
print("De-embeddings:")
show_top_deembeddings_extended(gpt2, gpt2_transcoder, latent_idx=blind_study_latent)
print("Logit lens:")
show_top_logits(gpt2, gpt2_transcoder, latent_idx=blind_study_latent)

# Results?
# - de-embedding has quite a few words related to finance or accumulation, e.g. " deficits", " output", " amounts", " amassed" (also "imately" could be the second half of "approximately")
#   - but definitely not as strong evidence as we got for "goal" earlier
# - logit lens shows us this latent firing will boost words like ' costing' and ' estimated'
#   - possible theory: it fires on phrases like "...fines <<costing>>..." or "...amassed <<upwards>> of..."
#   - prediction based on theory: we should see earlier latents firing on money-related words, and being attended to
#   - e.g. "the bank had <<amassed>> upwards of $100m$": maybe "amassed" attends to "bank"
```

#### 2단계: 이전 latent로부터의 영향 (direct path)

다음으로, 이전 transcoder latent로부터 오는 직접적인(attention을 거치지 않는) 영향을 살펴봅니다. 상위 activation 시퀀스들을 수집하고, pullback-weighted activation을 계산하며, 가장 일관되게 영향을 주는 latent들의 de-embeddings를 조사합니다.

```python
# (2) look at influence from earlier latents

# Gather 20 top activating sequences for the target latent
total_batches = 500
k = 20
buffer = 10
data = []  # list of (seq_pos: int, tokens: list[int], top_act: float)
for _ in tqdm(range(total_batches)):
    tokens = gpt2_act_store.get_batch_tokens()
    cache = run_with_cache_with_transcoder(gpt2, [gpt2_transcoder], tokens, use_error_term=True)
    acts = cache[f"{gpt2_transcoder.cfg.metadata.hook_name}.hook_sae_acts_post"][..., blind_study_latent]
    k_largest_indices = get_k_largest_indices(acts, k=k, buffer=buffer)  # [k, 2]
    tokens_in_top_sequences = tokens[k_largest_indices[:, 0]]  # [k, seq_len]
    top_acts = index_with_buffer(acts, k_largest_indices)  # [k,]
    data.extend(list(zip(k_largest_indices[:, 1].tolist(), tokens_in_top_sequences.tolist(), top_acts.tolist())))

data = sorted(data, key=lambda x: x[2], reverse=True)[:k]
tokens = t.tensor([x[1] for x in data])  # each row is a full sequence, containing one of the max activating tokens
top_seqpos = [x[0] for x in data]  # list of sequence positions of the max activating tokens
acts = [x[2] for x in data]  # list of max activating values

# Compute pullback from earlier latents to target latent, then compute influence for these top activating sequences
cache = run_with_cache_with_transcoder(gpt2, list(gpt2_transcoders.values()), tokens, use_error_term=True)
t.cuda.empty_cache()
all_influences = []
for _layer in range(layer):
    acts = cache[f"{gpt2_transcoders[_layer].cfg.metadata.hook_name}.hook_sae_acts_post"]  # shape [k=20, seq_len=128, d_sae=24k]
    acts_at_top_posn = acts[range(k), top_seqpos]  # shape [k=20, d_sae=24k]
    pullback = gpt2_transcoders[_layer].W_dec @ gpt2_transcoder.W_enc[:, blind_study_latent]  # shape [d_sae]
    influence = acts_at_top_posn * pullback  # shape [k=20, d_sae=24k]
    all_influences.append(influence)

# Find the earlier latents which are consistently in the top 10 for influence on target latent, and inspect their de-embeddings
all_influences = t.cat(all_influences, dim=-1)  # shape [k, n_layers*d_sae]
top_latents = all_influences.topk(k=10, dim=-1).indices.flatten()  # shape [k*10]
top_latents_as_tuples = [(i // gpt2_transcoder.cfg.d_sae, i % gpt2_transcoder.cfg.d_sae) for i in top_latents.tolist()]
top5_latents_as_tuples = sorted(Counter(top_latents_as_tuples).items(), key=lambda x: x[1], reverse=True)[:5]
print(
    tabulate(
        top5_latents_as_tuples,
        headers=["Latent", "Count"],
        tablefmt="simple_outline",
    )
)
for (_layer, _idx), count in top5_latents_as_tuples:
    print(f"Latent {_layer}.{_idx} was in the top 5 for {count}/{k} of the top-activating seqs. Top de-embeddings:")
    show_top_deembeddings_extended(gpt2, gpt2_transcoders[_layer], latent_idx=_idx)

# Results?
# - 7.13166 is very interesting: it's in the top way more than any other latent (17/20 vs 10/20 for the second best), and it boosts quantifiers like " approximately", " exceeding", " EQ", " ≥"
# - Since this is the direct path, possibly we'll find our target latent fires on these kinds of words too? Would make sense given its logit lens results
# - Also more generally, the words we're getting as top de-embeddings in these latents all appear in similar contexts, but they're not similar (i.e. substitutable) words, which makes this less likely to be a token-level latent
```

#### 3단계: attention head를 통한 영향

이제 attention을 통한 영향을 살펴봅니다. 어떤 이전 transcoder latent들이 attention head를 통해 타겟 latent에 영향을 주는가 하는 점입니다. 각 head에 대해, 타겟 latent의 reading vector를 OV circuit을 통해 역으로 매핑한 다음, 이를 이전 transcoder들의 attention-weighted writing vector와 내적합니다.

```python
# (3) look at influence coming from attention heads (i.e. embedding -> earlier transcoders -> attention -> target transcoder latent)

# The method here is a bit complicated. We do the following, for each head:
# - (A) Map the target latent's "reading vector" backwards through the attention head, to get a "source token reading vector" (i.e. the vector we'd dot product with the residual stream at the source token to get the latent activation for our target latent at the destination token)
# - (B) For all earlier transcoders, compute their "weighted source token writing vector" (i.e. the vector which they write to the residual stream at each source token, weighted by attention from target position to source position)
# - (C) Take the dot product of these, and find the top early latents for this particular head
top_latents_as_tuples = []
for attn_layer in range(layer + 1):  # we want to include target layer, because attn comes before MLP
    for attn_head in range(gpt2.cfg.n_heads):
        for early_transcoder_layer in range(attn_layer):  # we don't include target layer, because attn comes before MLP
            # Get names
            pattern_name = utils.get_act_name("pattern", attn_layer)
            transcoder_acts_name = f"{gpt2_transcoders[early_transcoder_layer].cfg.metadata.hook_name}.hook_sae_acts_post"

            # (A)
            reading_vector = gpt2_transcoder.W_enc[:, blind_study_latent]  # shape [d_model]
            reading_vector_src = einops.einsum(
                reading_vector,
                gpt2.W_O[attn_layer, attn_head],
                gpt2.W_V[attn_layer, attn_head],
                "d_model_out, d_head d_model_out, d_model_in d_head -> d_model_in",
            )

            # (B)
            writing_vectors = gpt2_transcoders[early_transcoder_layer].W_dec  # shape [d_sae, d_model]
            patterns = cache[pattern_name][range(k), attn_head, top_seqpos]  # shape [k, seq_K]
            early_transcoder_acts = cache[transcoder_acts_name]  # shape [k, seq_K, d_sae]
            pattern_weighted_acts = einops.einsum(patterns, early_transcoder_acts, "k seq_K, k seq_K d_sae -> d_sae")
            # pattern_weighted_acts = (patterns[..., None] * early_transcoder_acts).mean(0).mean(0) # shape [k, d_sae]
            weighted_src_token_writing_vectors = einops.einsum(
                pattern_weighted_acts, writing_vectors, "d_sae, d_sae d_model -> d_sae d_model"
            )

            # (C)
            influences = weighted_src_token_writing_vectors @ reading_vector_src  # shape [d_sae]
            top_latents_as_tuples.extend(
                [
                    {
                        "early_latent": repr(f"{early_transcoder_layer}.{idx.item():05d}"),
                        "attn_head": (attn_layer, attn_head),
                        "influence": value.item(),
                    }
                    # (early_transcoder_layer, attn_layer, attn_head, idx.item(), value.item())
                    for value, idx in zip(*influences.topk(k=10, dim=-1))
                ]
            )

top20_latents_as_tuples = sorted(top_latents_as_tuples, key=lambda x: x["influence"], reverse=True)[:20]
print(
    tabulate(
        [v.values() for v in top20_latents_as_tuples],
        headers=["Early latent", "Attention head", "Influence"],
        tablefmt="simple_outline",
    )
)

# Results?
# - Attribution from layer 7 transcoder:
#   - 2 latents fire in layer 7, and boost our target latent via head L8H5
#   - I'll inspect both of these (prediction = as described above, these latents' de-embeddings will be financial words)
# - Attribution from earlier transcoders:
#   - There are a few transcoder latents in layers 0, 1, 2 which have influence mediated through L7 attention heads (mostly L7H3 and L7H4)
#   - I'll check out both of them, but I'll also check out the de-embedding mapped directly through these heads (ignoring earlier transcoders), because I suspect these early transcoder latents might just be the extended embedding in disguise


def show_top_deembeddings_extended_via_attention_head(
    model: HookedSAETransformer,
    sae: SAE,
    latent_idx: int,
    attn_head: tuple[int, int] | None = None,
    k: int = 10,
    use_extended: bool = True,
) -> None:
    """
    Displays the top k de-embeddings for a particular latent, optionally after that token's embedding is mapped through
    some attention head.
    """
    t.cuda.empty_cache()
    W_E_ext = create_extended_embedding(model) if use_extended else (model.W_E / model.W_E.std(dim=-1, keepdim=True))

    if attn_head is not None:
        W_V = model.W_V[*attn_head]
        W_O = model.W_O[*attn_head]
        W_E_ext = (W_E_ext @ W_V) @ W_O
        W_E_ext = (W_E_ext - W_E_ext.mean(dim=-1, keepdim=True)) / W_E_ext.std(dim=-1, keepdim=True)

    de_embeddings = W_E_ext @ sae.W_enc[:, latent_idx]

    pos_logits, pos_token_ids = de_embeddings.topk(k)
    pos_tokens = model.to_str_tokens(pos_token_ids)

    print(
        tabulate(
            zip(map(repr, pos_tokens), pos_logits),
            headers=["Top tokens", "Value"],
            tablefmt="simple_outline",
            stralign="right",
            numalign="left",
            floatfmt="+.3f",
        )
    )


print("Layer 7 transcoder latents (these influence the target latent via L8H5):")
for _layer, _idx in [(7, 3373), (7, 14110), (7, 10719), (7, 8696)]:
    print(f"{_layer}.{_idx} de-embeddings:")
    show_top_deembeddings_extended_via_attention_head(gpt2, gpt2_transcoders[_layer], latent_idx=_idx)

print("\n" * 3 + "Layer 1-2 transcoder latents (these influence the target latent via L7H3 and L7H4):")
for _layer, _idx in [(2, 21691), (1, 14997)]:
    print(f"{_layer}.{_idx} de-embeddings:")
    show_top_deembeddings_extended_via_attention_head(gpt2, gpt2_transcoders[_layer], latent_idx=_idx)

print("\n" * 3 + "De-embeddings of target latent via L7H3 and L7H4:")
for attn_layer, attn_head in [(7, 3), (7, 4)]:
    print(f"L{attn_layer}H{attn_head} de-embeddings:")
    show_top_deembeddings_extended_via_attention_head(
        gpt2,
        gpt2_transcoder,
        latent_idx=blind_study_latent,
        attn_head=(attn_layer, attn_head),
    )

# Results?
# - Layer 7 transcoder latents:
#   - 14110 & 8696 both seem to fire on financial words, e.g. " revenues" is top word for both and they also both include " GDP" in their top 10
#       - They also both fire on words like "deaths" and "fatalities", which also makes sense given my hypothesis (e.g. this could be sentences like "the number fatalities* is approximately** totalling***" (where * = src token where the layer 7 latent fires, ** = word predicted by target latent)
#   - 10719 very specifically fires on the word "estimated" (or variants), which also makes sense: these kinds of sentences can often have the word "estimated" in them (e.g. "the estimated number of fatalities is 1000")
#   - 3373 fires on "effectively", "constitutes" and "amounted", which are also likely to appear in sentences like this one (recall we've not looked at where attn is coming from - this could be self-attention!)
# - Earlier transcoder latents:
#   - Disappointingly, these don't seem very interpretable (nor when I just look at direct contributions from the attention heads which are meant to be mediating their influence)
```

#### 4단계: 컴포넌트 수준의 attribution

마지막으로, 상위 activation 시퀀스들에 걸쳐 각 컴포넌트(embeddings, attention heads, MLPs)로부터 오는 평균 attribution을 합산합니다. 이를 통해 어떤 경로가 가장 중요한지에 대한 정성적인 그림을 얻을 수 있습니다.

```python
# (4) Final experiment: component-level attribution

# For all these top examples, I want to tally up the contributions from each component (past MLP layers, attention heads, and direct path) and compare them
# This gives me a qualitative sense of which ones matter more

latent_dir = gpt2_transcoder.W_enc[:, blind_study_latent]  # shape [d_model,]

embedding_attribution = cache["embed"][range(k), top_seqpos].mean(0) @ latent_dir

attn_attribution = (
    t.stack(
        [
            einops.einsum(
                cache["z", _layer][range(k), top_seqpos].mean(0),
                gpt2.W_O[_layer],
                "head d_head, head d_head d_model -> head d_model",
            )
            for _layer in range(layer + 1)
        ]
    )
    @ latent_dir
)  # shape [layer+1, n_heads]

mlp_attribution = (
    t.stack([cache["mlp_out", _layer][range(k), top_seqpos].mean(0) for _layer in range(layer)]) @ latent_dir
)

all_attributions = t.zeros((layer + 2, gpt2.cfg.n_heads + 1))
all_attributions[0, 0] = embedding_attribution
all_attributions[1:, :-1] = attn_attribution
all_attributions[1:-1, -1] = mlp_attribution

df = pd.DataFrame(utils.to_numpy(all_attributions))

text = [["W_E", *["" for _ in range(gpt2.cfg.n_heads)]]]
for _layer in range(layer + 1):
    text.append(
        [f"L{_layer}H{_head}" for _head in range(gpt2.cfg.n_heads)] + [f"MLP{_layer}" if _layer < layer else ""]
    )

fig = px.imshow(
    df,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    width=700,
    height=600,
    title="Attribution from different components",
)
fig.data[0].update(text=text, texttemplate="%{text}", textfont={"size": 12})
fig.show()

# Results?
# - Way less impact from W_E than I expected, and even MLP0 (extended embedding) had a pretty small impact, this is evidence away from it being a token-level latent
# - Biggest attributions are from L8H5 and MLP7
#   - L8H5 is the one that attends back to (A) tokens with financial/fatalities context, (B) the word "estimated" and its variants, and (C) other related quantifiers like "effectively" or "amounted"
#   - MLP7 was seen to contain many latents that fired on words which would appear in sentences related to financial estimations (see (2), where we looked at the top 5 contributing latents - they were all in layer 7)
# - Also, the not-very-interpretable results from attention heads 7.3 and 7.4 matter less now, because we can see from this that they aren't very important (although I don't know why they turned up so high before)
```

#### 최종 이론 및 검증

위에서 수집한 모든 증거를 바탕으로 구체적인 가설을 세우고, Neuronpedia 대시보드를 통해 이를 테스트합니다.

```python
# Based on all evidence, this is my final theory:

# - The latent activates primarily on sentences involving estimates of financial quantities (or casualties)
# - For example I expect top activating seqs like:
#     - "The number of fatalities is **approximately** totalling..."
#     - "The bank had **amassed** upwards of $100m..."
#     - "The GDP of the UK **exceeds** $300bn..."
#     - "This tech company is estimated to be **roughly** worth..."
#    where I've highlighted what I guess to be the top activating token, but the surrounding cluster should also be activating
# - Concretely, what causes it to fire? Most important things (in order) are:
#     - (1) Attention head 8.5, which attends back to the output of layer 7 transcoder latents that fire on words which imply we're in sentences discussing financial quantities or fatality estimates (e.g. "fatalities", "bank", "GDP" and "company" in the examples above). Also this head strongly attends back to a layer 7 latent which detects the word "estimated" and its variants, so I expect very strong activations to start after this word appears in a sentence
#     - (2) Layer-7 transcoder latents (directly), for example latent 7.13166 fires on the token "≤" and causes our target latent to fire
#     - (3) Direct path: the latent should fire strongest on words like **approximately** which rank highly in its de-embedding

# Let's display the latent dashboard for both the target latent and the other latents involved in this theory, and see if the theory is correct:

neuronpedia_id = "gpt2-small/8-tres-dc"
url = f"https://neuronpedia.org/{neuronpedia_id}/{blind_study_latent}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"
display(IFrame(url, width=800, height=600))

# Conclusions?
# - Mostly correct:
#   - The top activating sequences are mostly financial estimates
#   - Activations are very large after the word "estimated" (most of the top examples are sentences containing this word)
#   - The latent doesn't seem to be token-level; it fires on a cluster of adjacent words
# - Some areas where the hypothesis was incorrect, or lacking:
#   - I didn't give a hypothesis for when the activations would stop - it seems they stop exactly at the estimated value, and I don't think I would have been able to predict that based on the experiments I ran
#       - Relatedly, I wouldn't have predicted activations staying high even on small connecting words before the estimated value (e.g. "of" in "monthly rent of...", or "as" in "as much as...")
#   - I overestimated the importance of the current word in the sentence (or more generally, I had too rigid a hypothesis for what pattern of sentences would this latent activate on & where it would activate)
#   - I thought there would be more casualty estimates in the top activating sequences, but there weren't. Subsequent testing (see code below) shows that it does indeed fire strongly on non-financial estimates with the right sentence structure, and fatalities fires stronger than the other 2 non-financial example sentences, but the difference is small, so I think this was still an overestimation in my hypothesis)

prompts = {
    "fatalities": """Body counts are a crude measure of the war's impact and more reliable estimates will take time to compile. Since war broke out in the Gaza Strip almost a year ago, the official number of Palestinians killed is estimated to exceed 41,000.""",
    "emissions": """Environmental measurements are an imperfect gauge of climate change impact and more comprehensive studies will take time to finalize. Since the implementation of new global emissions policies almost a year ago, the reduction in global carbon dioxide emissions is estimated to exceed million metric tons.""",
    "visitors": """Visitor counts are a simplistic measure of a national park's popularity and more nuanced analyses will take time to develop. Since the implementation of the new trail system almost a year ago, the number of unique bird species spotted in Yellowstone National Park is estimated to have increased by 47.""",
}
acts_dict = {}
for name, prompt in prompts.items():
    str_tokens = [f"{tok} ({i})" for i, tok in enumerate(gpt2.to_str_tokens(prompt))]
    cache = run_with_cache_with_transcoder(gpt2, [gpt2_transcoder], prompt)
    acts = cache[f"{gpt2_transcoder.cfg.metadata.hook_name}.hook_sae_acts_post"][0, :, blind_study_latent]
    acts_dict[name] = utils.to_numpy(acts).tolist()

min_length = min([len(x) for x in acts_dict.values()])
acts_dict = {k: v[-min_length:] for k, v in acts_dict.items()}

df = pd.DataFrame(acts_dict)
px.line(df, y=prompts.keys(), height=500, width=800).show()
```

</details>

1️⃣ 및 2️⃣ 섹션에서는 latent 수준의 gradient와 transcoder를 연구하기 위해 GPT-2 Small을 사용했습니다. 우리는 latent 쌍 사이의 gradient를 통해 어떤 upstream latent가 특정 downstream latent에 가장 큰 영향을 미치는지 알 수 있으며, transcoder가 MLP 계산의 해석 가능한 분해(각 latent에 대한 명시적인 reading 및 writing 벡터라는 추가적인 이점 포함)를 제공한다는 것을 확인했습니다.

다음 섹션에서는 이 두 가지 아이디어를 결합하여 **attribution graph**를 만들어 보겠습니다. 이를 통해 모델이 특정 출력을 생성하는 방식에 대한 완전한 인과적 그림을 얻을 수 있습니다. 또한, 연구를 위해 더 성능이 뛰어난 모델인 **Gemma 3-1B IT**와 GemmaScope 2 transcoder로 전환하겠습니다.

# 3️⃣ Attribution graphs

> ##### 학습 목표
>
> - local replacement model을 이해합니다: attention pattern과 LayerNorm scale을 고정하고, MLP를 linear skip connection으로 대체했을 때 왜 residual stream이 linear해지는지 학습합니다.
> - reading/writing vector 추상화를 이해합니다: token embedding, transcoder latent, MLP error, logit direction이 residual stream과 어떻게 상호작용하는지 학습합니다.
> - 핵심 attribution 알고리즘을 구현합니다: salient logit 선택, graph node 구축, 그리고 gradient injection을 통한 edge weight 계산을 구현합니다.
> - nilpotent adjacency matrix에 대한 Neumann series를 사용하여, node 및 edge influence thresholding을 통한 graph pruning을 구현합니다.
> - Anthropic의 공개 연구와 동일한 대시보드 템플릿을 사용하여 대화형 attribution graph 시각화를 구축합니다.

1️⃣ 및 2️⃣ 섹션에서 우리는 개별 latent 쌍 사이의 gradient를 계산하는 방법과 transcoder가 MLP 계산을 해석 가능한 latent로 분해하는 방법을 살펴보았습니다. 이제 이러한 아이디어들을 결합하여 **attribution graphs**를 만들어 보겠습니다. 이는 모델이 특정 출력을 생성하는 방식에 대한 엔드-투-엔드 인과적 설명이며, transcoder latent의 관점에서 표현됩니다.

Anthropic의 [Circuit Tracing paper](https://transformer-circuits.pub/2025/attribution-graphs/methods.html)에서 설명하듯이, 목표는 "'replacement model'에서 개별 계산 단계를 추적함으로써 관심 있는 prompt에 대한 모델 계산의 그래프 설명을 생성하는 것"입니다. MLP를 transcoder로 대체함으로써 모든 중간 단계가 해석 가능한 feature로 표현되는 replacement model을 얻게 됩니다. 또한 모델을 linearise(attention pattern과 LayerNorm scale을 고정)하기 때문에, 임의의 feature 쌍 사이의 영향력은 backward-pass attribution을 통해 효율적으로 계산할 수 있는 잘 정의된 선형 양이 됩니다.

attribution graph는 방향성 비순환 그래프(directed acyclic graph)입니다. 입력 노드는 token embedding(prompt의 token당 하나)입니다. 중간 노드는 transcoder latent(해당 feature가 활성화되는 각 위치의 feature)입니다. 출력 노드는 logit direction(최종 위치에서 가장 높게 예측된 token들)입니다. 엣지는 노드 간의 직접적인 인과적 영향력을 나타내며, 모델의 linearised 버전을 통한 backward-pass attribution으로 계산됩니다.

핵심 통찰은 **reading/writing vector abstraction**입니다. 모든 노드는 **writing vector**(residual stream에 더하는 방향)와 **reading vector**(residual stream에서 읽어오는 방향)를 가집니다. 두 노드 사이의 엣지 가중치는 기본적으로 소스 노드의 writing vector가 고정된 중간 레이어들을 통해 매핑된 후, 타겟 노드의 reading vector와 내적(dot product)한 값입니다.

이를 구현하기 위해, 우리는 모델의 비선형성(attention pattern, LayerNorm scale)을 고정하여 모델을 **linearise** 합니다. 이렇게 하면 residual stream이 각 노드의 feature activation에 대한 선형 함수가 되어, batched backward pass를 통해 모든 엣지 가중치를 효율적으로 계산할 수 있습니다. MLP preactivation 비선형성(JumpReLU/ReLU)은 그대로 유지된다는 점에 유의하십시오. 즉, 모델은 raw input에 대해 선형인 것이 아니라, 고정된 feature activation이 주어졌을 때 선형입니다. 그 후, sparse latent를 사용하더라도 특정 prompt에서 활성화되는 latent가 너무 많아 전체 그래프를 해석하기 어렵기 때문에, 가장 영향력 있는 노드와 엣지만 남기도록 그래프를 **prune** 합니다.

이 섹션은 Anthropic의 [Circuit Tracing paper](https://transformer-circuits.pub/2025/attribution-graphs/methods.html)과 그 자매 논문인 [On the Biology of a Large Language Model](https://transformer-circuits.pub/2025/attribution-graphs/biology.html)를 바탕으로 합니다.

## GemmaScope 2

우리는 Google DeepMind의 Gemma 모델 제품군을 위한 두 번째 sparse autoencoder 및 transcoder 세트인 **GemmaScope 2**의 transcoder를 사용할 것입니다.

GemmaScope 2는 기존 GemmaScope 릴리스의 후속작입니다. 이는 (Gemma 2가 아닌) **Gemma 3** 모델 제품군을 다루며, 최대 27B까지 모든 크기의 pre-trained 및 instruction-tuned 모델의 모든 layer에서 학습된 transcoder를 제공합니다.

이 transcoder들은 **Matryoshka loss**로 학습되었으므로, latent들이 자연스러운 계층 구조를 가집니다. 인덱스가 작은 latent들은 더 자주 활성화되고 reconstruction에 더 중요한 경향이 있는 반면, 인덱스가 큰 latent들은 더 좁고 구체적인 개념을 나타냅니다 (그리고 종종 연구하기에 더 흥미롭습니다).

또한 이들은 **affine skip connections** (`W_skip`)와 함께 학습되었습니다. skip connection은 MLP 계산의 선형 성분을 캡처하고, transcoder latent들은 비선형 성분을 캡처합니다. attribution을 위해 모델을 freeze하면, gradient는 전체 비선형 MLP가 아닌 선형 skip connection을 통해 흐르게 됩니다.

GemmaScope 2에 대해 더 자세히 읽어보실 수 있습니다 [here](https://deepmind.google/blog/gemma-scope-2-helping-the-ai-safety-community-deepen-understanding-of-complex-language-model-behavior/).

## 설정: transcoder를 사용하여 Gemma 3-1B IT 로드하기

우리는 SAELens의 **GemmaScope 2 transcoders**와 함께 **Gemma 3-1B IT**를 사용할 것입니다. Gemma 모델은 gated 모델이므로, 먼저 huggingface에서 API 키를 받아 `.env` 파일에 `HF_TOKEN` 로 저장해야 한다는 점에 유의하시기 바랍니다.

gemmascope-2 모델 시리즈가 어떻게 저장되어 있는지 확인하려면 [HuggingFace readme](https://huggingface.co/google/gemma-scope-2-1b-it) 를 읽어보실 수 있습니다. 요약하자면 다음과 같습니다:

- 릴리스 이름은 특정 모델의 모든 layer에 적용된 SAE 릴리스의 경우 `gemma-scope-2-{size}-{type}-{site}-all` 입니다 (그리고 `-all` 를 제거하면 더 다양한 hyperparameter 버전이 포함되어 있지만 일부 layer subset에 대해서만 학습된 SAE 세트를 얻을 수 있습니다).
  - `size` 는 `270m`, `1b`, `4b`, `12b` 또는 `27b` 중 하나입니다.
  - `type` 은 `pt` (pretrained) 또는 `it` (instruction-tuned) 중 하나입니다.
  - `site` 는 `transcoders` 이거나 sae 사이트의 이름, 즉 `resid_post`, `mlp_out` 또는 `attn_out` 입니다 (단순화를 위해 현재 multi-layer 모델은 무시합니다).
- 개별 SAE ID는 `layer_{layer}_width_{width}_l0_{l0}` 형태이며, 여기서:
  - `layer` 은 정수 layer index (zero-indexed)입니다.
  - `width` 은 문자열이며, `-all` 릴리스의 경우 `16k` 또는 `262k` 입니다. 다만 subset 릴리스는 `65k` 뿐만 아니라 resid-post SAE의 경우 `1m` 도 지원합니다.
  - `l0` 은 `small` (약 10-20) 또는 `big` (약 100-150)이며, subset 릴리스에는 약 `30-60` 인 `medium` 이 있습니다.
  - 또한 transcoders는 SAE ID 끝에 `_affine` 가 추가되어 있습니다.

API 키를 준비하셨다면, 이제 모델과 transcoders를 로드해 보겠습니다.

In [ ]:
# Load huggingface token
load_dotenv(dotenv_path=str(exercises_dir / ".env"))
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN, "Please set HF_TOKEN in your chapter1_transformer_interp/exercises/.env file"

# Load gemma model using HF token
gemma = HookedSAETransformer.from_pretrained(
    "google/gemma-3-1b-it",
    device=device,
    dtype=dtype,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
)
gemma.set_use_hook_mlp_in(True)

# Check this version of SAELens has the gemmascope transcoders
assert "gemma-scope-2-1b-it-transcoders-all" in get_pretrained_saes_directory()
n_layers = 26
n_saes_per_layer = 8  # 2 widths, 2 L0s, 2 (affine vs non-affine)
transcoders_map = get_pretrained_saes_directory()["gemma-scope-2-1b-it-transcoders-all"].saes_map
assert len(transcoders_map) == n_layers * n_saes_per_layer

# Load transcoders for all layers (parallel I/O, sequential GPU transfer)
n_layers = gemma.cfg.n_layers
sae_ids = [f"layer_{layer}_width_16k_l0_small_affine" for layer in range(n_layers)]
transcoders: dict[int, SAE] = utils.load_saes_parallel(
    release="gemma-scope-2-1b-it-transcoders-all",
    sae_ids=sae_ids,
    device=device,
    dtype=dtype,
    max_workers=2,
)
# Correct the hook names (temporary until this is fixed)
for layer in range(n_layers):
    transcoders[layer].cfg.metadata.hook_name = f"blocks.{layer}.mlp.hook_in"

이제 prompt를 정의해 보겠습니다. Gemma IT 모델을 위한 chat template 형식을 사용하겠습니다. 특수 token `<start_of_turn>` 및 `<end_of_turn>`에 유의하십시오. attribution graph를 구축할 때 처음 4개의 token (BOS + `<start_of_turn>` + `user` + `\n`)은 마스킹해야 합니다. 이들은 의미 있는 attribution 신호를 전달하지 않는 formatting token이기 때문입니다.

In [ ]:
def format_prompt(user_prompt: str, model_response: str) -> str:
    """Format a prompt for Gemma IT models using the chat template."""
    return f"<start_of_turn>user\n{user_prompt}<end_of_turn>\n<start_of_turn>model\n{model_response}"


START_POSN = 4  # Mask [BOS, <start_of_turn>, user, \n]

# Example: rhyming couplet prompt from Anthropic's attribution graph work
prompt = format_prompt(
    "Write me a short rhyming couplet.",
    "The sun descends, a golden hue,\nAs evening whispers, soft and",  # (...true)
)

# Tokenize and verify
tokens = gemma.to_tokens(prompt)
str_tokens = gemma.to_str_tokens(prompt)
print(f"Prompt has {len(str_tokens)} tokens")
print(f"First {START_POSN} tokens (masked): {str_tokens[:START_POSN]}")
print(f"Remaining tokens: {str_tokens[START_POSN:]}")

## 로컬 대체 모델 (The local replacement model)

attribution graph의 핵심 아이디어는 **모델을 선형화(linearising)하는 것**입니다. transformer의 residual stream은 거의 선형적입니다. 주요 비선형성은 attention pattern (query와 key에 대한 softmax), LayerNorm scale (RMSNorm 정규화 계수), 그리고 MLP activation (비선형 MLP 계산)입니다.

만약 이 세 가지를 forward pass 값으로 고정한다면, residual stream은 각 노드에서의 **feature activation에 대한 선형 함수**가 됩니다. (MLP preactivation의 비선형성은 남아있지만, 이는 노드 값 자체에 흡수됩니다.) 이는 단 한 번의 backward pass를 통해 명확하게 정의된 attribution을 계산할 수 있음을 의미합니다.

특히 MLP의 경우, 전체 비선형 MLP를 **선형 skip connection**으로 대체합니다: `mlp_out ≈ mlp_input @ W_skip`, 여기서 `W_skip`은 transcoder의 affine skip connection weight입니다. 이것이 완벽한 근사는 아니지만, MLP 동작의 선형 성분을 포착하며, transcoder latent가 비선형 성분을 포착합니다.

여기서 중요한 세부 사항은 attention에서 정확히 무엇을 고정하느냐입니다. 우리는 attention pattern (softmax 출력)은 고정하지만, value 계산 (`V @ W_O`)은 미분 가능한 상태로 유지합니다. 이는 매우 중요합니다. 만약 attention 출력 전체를 고정한다면, attention을 통한 모든 position 간의 gradient flow를 잃게 되며, attribution graph는 동일한 position 내의 edge만 가지게 될 것입니다.

TransformerLens에서 선형화가 작동하는 방식은 다음과 같습니다:

- `hook_pattern` (attention pattern matrix / softmax 출력)을 고정합니다. gradient는 여전히 value 벡터를 통해 흐르지만, 가중치 조합은 고정됩니다.
- `hook_scale` (LayerNorm scale factor)를 고정하여 LayerNorm을 선형 연산으로 만듭니다.
- MLP의 경우 "skip connection trick"을 사용합니다: `mlp_out`를 `skip + (mlp_out - skip).detach()`으로 대체하며, 여기서 `skip = ln(resid_mid) @ W_skip`입니다. forward pass 동안에는 올바른 MLP 출력을 내놓지만, backward 동안에는 gradient가 선형 skip connection을 통해 흐르게 됩니다.

### `FreezeHooks` - attention 및 LayerNorm 동결하기

attention pattern과 LayerNorm scale의 동결을 처리하는 `FreezeHooks` 클래스를 제공해 드리겠습니다. 이 코드를 주의 깊게 읽고 어떤 역할을 하는지 정확히 이해하시기 바랍니다.

In [ ]:
class FreezeHooks:
    """
    Installs forward hooks that freeze attention patterns and LayerNorm scales at their
    forward-pass values, making these operations linear.

    Uses TransformerLens-native hooks (not PyTorch register_forward_hook). Two interfaces:

    1. Context manager (for direct use):
        with freeze:
            model(tokens)  # patterns/scales replaced by frozen values

    2. fwd_hooks property (for combining with other hooks via model.hooks()):
        with model.hooks(fwd_hooks=freeze.fwd_hooks + other_hooks, bwd_hooks=...):
            model(tokens).backward()
    """

    def __init__(self, model: HookedSAETransformer):
        self.model = model
        self.frozen_values: dict[str, Tensor] = {}

    def _freeze_hook(self, value: Tensor, hook: HookPoint) -> Tensor:
        """
        TL-native hook: replaces activations with their frozen values.

        Handles batched forward passes (batch_size > 1) even though frozen values were cached
        with batch_size=1. Two cases arise:

        - Regular hooks (hook_pattern, ln.hook_scale): frozen shape is [1, ...], so we repeat
          along dim 0 to get [batch, ...].
        - QK-norm hook_scale: the batch dim is folded into dim 0 as [batch*pos*n_heads, 1],
          so the frozen shape is [pos*n_heads, 1]. We repeat by the batch ratio to get
          [batch*pos*n_heads, 1].

        In both cases the ratio value.shape[0] // frozen.shape[0] gives the right repeat factor.
        """
        frozen = self.frozen_values[hook.name]
        if frozen.shape[0] != value.shape[0] and value.shape[0] % frozen.shape[0] == 0:
            ratio = value.shape[0] // frozen.shape[0]
            frozen = frozen.repeat(ratio, *([1] * (frozen.dim() - 1)))
        return frozen

    @property
    def fwd_hooks(self) -> list[tuple[str, Callable]]:
        """
        Return freeze hooks as a list of (hook_name, hook_fn) pairs.

        Use this with model.hooks() to combine freeze hooks with other hooks in a
        single context manager (Option A), avoiding nesting issues:

            with model.hooks(fwd_hooks=freeze.fwd_hooks + capture_hooks, bwd_hooks=...):
                model(tokens).backward()
        """
        return [(name, self._freeze_hook) for name in self.frozen_values]

    def cache_frozen_values(self, tokens: Tensor) -> ActivationCache:
        """Runs a forward pass, caching values we'll freeze later, plus other useful activations."""
        # We cache attention patterns, LN scales, and residual stream values
        names_filter = lambda name: any(
            s in name for s in ["hook_pattern", "hook_scale", "resid_post", "resid_pre", "mlp.hook_in", "mlp_out"]
        )
        _, cache = self.model.run_with_cache(tokens, names_filter=names_filter)

        # Store frozen values (patterns and scales)
        self.frozen_values = {
            name: cache[name].detach() for name in cache.keys() if "hook_pattern" in name or "hook_scale" in name
        }
        return cache

    def __enter__(self):
        """
        Install freeze hooks using TransformerLens's model.add_hook() (non-permanent).

        Non-permanent hooks are removed by model.reset_hooks(including_permanent=False),
        which is what __exit__ calls. TranscoderReplacementHooks uses permanent hooks, so
        they survive this reset.
        """
        for name in self.frozen_values:
            self.model.add_hook(name, self._freeze_hook)
        return self

    def __exit__(self, *args):
        """Remove all non-permanent hooks (freeze hooks only; permanent tc_hooks survive)."""
        self.model.reset_hooks(including_permanent=False)

### 연습 문제 - `TranscoderReplacementHooks` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 25-30 minutes on this exercise.
> ```

이제 `TranscoderReplacementHooks` 클래스를 구현합니다. 이 클래스는 forward pass 동안에는 올바른 MLP 출력을 유지하면서, backward pass 동안에는 MLP를 transcoder의 linear skip connection으로 대체합니다.

핵심 트릭은 **stop-gradient skip connection**입니다:

```
skip = ln2(resid_mid) @ W_skip
mlp_out_new = skip + (mlp_out_original - skip).detach()
```

**forward pass** 동안에는 `mlp_out_new = skip + mlp_out_original - skip = mlp_out_original` 이므로, 정확히 동일한 출력을 얻게 됩니다.

**backward pass** 동안에는 `.detach()` 가 `(mlp_out_original - skip)` 을 통한 gradient를 차단하므로, gradient는 linear approximation인 `skip = ln2(resid_mid) @ W_skip` 로만 흐르게 됩니다.

또한, 그래프 노드를 구축하는 데 필요하므로 이 클래스는 각 layer에 대해 transcoder activation(어떤 feature가 얼마나 강하게 활성화되었는지)을 계산하고 저장해야 합니다.

`JumpReLUSkipTranscoder` 의 다음 attribute들을 사용해야 합니다:
- `tc.cfg.metadata.hook_name`: transcoder input을 위한 hook 이름 (예: `"blocks.0.mlp.hook_in"`)
- `tc.W_skip`: skip connection weight matrix, shape `(d_model, d_model)`
- `tc.encode(x)`: transcoder feature activation을 반환, shape `(batch, seq, d_enc)`
- `tc.decode(acts)`: feature activation으로부터 transcoder output을 반환, shape `(batch, seq, d_model)`

<details><summary>힌트 - hook 이름</summary>

transcoder input hook은 `blocks.{layer}.mlp.hook_in` (즉, LayerNorm 이후의 MLP input)에 위치합니다. MLP output hook은 `blocks.{layer}.hook_mlp_out` 에 위치합니다.

skip connection 트릭을 적용하려면 MLP output에 hook을 걸어야 합니다.

</details>

<details><summary>힌트 - 구조</summary>

``install`` 메서드는 각 transcoder를 순회하며 다음을 수행해야 합니다:
1. skip connection 트릭을 적용하는 hook을 MLP output에 추가합니다.
2. 나중에 사용하기 위해 (`tc.encode` 로부터) transcoder activation을 저장합니다.

skip connection과 transcoder activation을 모두 계산하려면 LN-normalised MLP input이 필요합니다. 이는 `FreezeHooks` cache에서 가져오거나, forward pass 동안 hook을 통해 얻을 수 있습니다.

</details>

<details><summary>힌트 - hook 함수 패턴</summary>

TL-native hook을 등록하려면 `model.add_hook(output_hook, fn, is_permanent=True)` 을 사용하십시오. hook 함수의 signature는 `hook_fn(mlp_out: Tensor, hook: HookPoint) -> Tensor` 이며, 여기서 `mlp_out` 는 MLP output tensor입니다. hook은 skip connection 트릭을 사용해 수정된 버전을 반환해야 합니다.

`model.reset_hooks(including_permanent=False)` 을 호출하여 non-permanent hook만 제거하는 `FreezeHooks.__exit__` 에서도 hook이 유지되도록 `is_permanent=True` 으로 등록하십시오.

루프 내의 Python closure에 주의하십시오! 각 hook에 대해 올바른 layer와 transcoder를 캡처하려면 factory function(`make_hook(layer, tc)`)을 사용하십시오.

</details>

In [ ]:
from sae_lens import JumpReLUSkipTranscoder, JumpReLUTranscoder

Transcoder: TypeAlias = JumpReLUTranscoder | JumpReLUSkipTranscoder


class TranscoderReplacementHooks:
    """
    Installs hooks that replace MLP backward passes with linear skip connections, while
    computing and storing transcoder feature activations for graph construction.

    The skip connection trick ensures:
    - Forward pass: exact MLP output (unchanged)
    - Backward pass: gradients flow through linear skip connection (W_skip)

    Uses TransformerLens-native hooks registered as PERMANENT (is_permanent=True), so they
    survive FreezeHooks.__exit__ which only resets non-permanent hooks. Call remove() when
    done to clear permanent hooks via model.reset_hooks(including_permanent=True).
    """

    def __init__(
        self,
        model: HookedSAETransformer,
        transcoders: dict[int, "Transcoder"],
        cache: ActivationCache,
    ):
        self.model = model
        self.transcoders = transcoders
        self.cache = cache  # From FreezeHooks.cache_frozen_values
        self.transcoder_acts: dict[int, Tensor] = {}  # layer -> feature activations
        self.transcoder_output: dict[int, Tensor] = {}  # layer -> transcoder reconstruction

        self.current_ln_inputs: dict[int, Tensor] = {}  # updated each forward pass

    # def install(self):
    #     """Install hooks for all transcoders."""
    #     self.current_ln_inputs: dict[int, Tensor] = {}
    #     for layer, tc in self.transcoders.items():
    #         # Get hook names for this layer
    #         input_hook = tc.cfg.metadata.hook_name  # e.g. "blocks.0.mlp.hook_in"
    #         output_hook = tc.cfg.metadata.hook_name_out  # e.g. "blocks.0.hook_mlp_out"
    #
    #         # Compute and store transcoder activations (no gradient needed)
    #         with t.no_grad():
    #             ln_input = self.cache[input_hook]
    #             tc_acts = tc.encode(ln_input)
    #             tc_output = tc.decode(tc_acts)
    #             self.transcoder_acts[layer] = tc_acts.detach()
    #             self.transcoder_output[layer] = tc_output.detach()
    #
    #         # YOUR CODE HERE - register TWO permanent TL-native hooks:
    #         #
    #         # 1. A capture hook at input_hook (mlp.hook_in):
    #         #    Signature: (tensor, hook) -> None
    #         #    Store the live ln_input in self.current_ln_inputs[layer].
    #         #    This MUST use the live tensor, not self.cache[input_hook] - the cached
    #         #    tensor is attached to a stale graph that gets freed after the first backward().
    #         #
    #         # 2. A skip hook at output_hook (hook_mlp_out):
    #         #    Signature: (mlp_out, hook) -> Tensor
    #         #    Apply the skip connection trick using self.current_ln_inputs[layer]:
    #         #       skip = ln_input @ tc.W_skip
    #         #       return skip + (mlp_out - skip).detach()
    #         #
    #         # Register both with is_permanent=True so they survive FreezeHooks context exits.
    #         # Use factory functions to avoid closure issues in the loop.
    #         pass

    def remove(self):
        """Remove all hooks, including permanent ones added by install()."""
        self.model.reset_hooks(including_permanent=True)

In [ ]:
# Test: verify the skip connection trick gives correct forward pass
freeze = FreezeHooks(gemma)
cache = freeze.cache_frozen_values(tokens)

tc_hooks = TranscoderReplacementHooks(gemma, transcoders, cache)
tc_hooks.install()

with freeze:
    logits_with_hooks = gemma(tokens)

tc_hooks.remove()

# Compare with original logits
logits_original = gemma(tokens)

print(f"Max difference in logits: {(logits_with_hooks - logits_original).abs().max().item():.6f}")
print("(Should be ~0 since the skip trick preserves forward pass values)")

<details><summary>솔루션</summary>

```python
from sae_lens import JumpReLUSkipTranscoder, JumpReLUTranscoder

Transcoder: TypeAlias = JumpReLUTranscoder | JumpReLUSkipTranscoder


class TranscoderReplacementHooks:
    """
    Installs hooks that replace MLP backward passes with linear skip connections, while
    computing and storing transcoder feature activations for graph construction.

    The skip connection trick ensures:
    - Forward pass: exact MLP output (unchanged)
    - Backward pass: gradients flow through linear skip connection (W_skip)

    Uses TransformerLens-native hooks registered as PERMANENT (is_permanent=True), so they
    survive FreezeHooks.__exit__ which only resets non-permanent hooks. Call remove() when
    done to clear permanent hooks via model.reset_hooks(including_permanent=True).
    """

    def __init__(
        self,
        model: HookedSAETransformer,
        transcoders: dict[int, "Transcoder"],
        cache: ActivationCache,
    ):
        self.model = model
        self.transcoders = transcoders
        self.cache = cache  # From FreezeHooks.cache_frozen_values
        self.transcoder_acts: dict[int, Tensor] = {}  # layer -> feature activations
        self.transcoder_output: dict[int, Tensor] = {}  # layer -> transcoder reconstruction

        self.current_ln_inputs: dict[int, Tensor] = {}  # updated each forward pass

    def install(self):
        """
        Install hooks for all transcoders.

        Two permanent hooks are registered per layer:
          1. hook_mlp_in - saves the live LN-normalised MLP input each forward pass.
          2. hook_mlp_out - applies the skip connection trick using that saved input.

        Using the LIVE ln_input (not the cached one) is critical: the cached tensor has a
        grad_fn attached to the original run_with_cache graph, which PyTorch frees after the
        first backward(). Subsequent batches would then fail with "trying to backward through
        the graph a second time". Using the live tensor keeps every backward pass within its
        own fresh computation graph.
        """
        for layer, tc in self.transcoders.items():
            # Get the LN-normalised MLP input from the cache
            input_hook = tc.cfg.metadata.hook_name  # e.g. "blocks.0.mlp.hook_in"
            output_hook = tc.cfg.metadata.hook_name_out  # e.g. "blocks.0.hook_mlp_out"

            # Compute transcoder activations (no gradient needed for this)
            with t.no_grad():
                ln_input = self.cache[input_hook]
                tc_acts = tc.encode(ln_input)
                tc_output = tc.decode(tc_acts)
                self.transcoder_acts[layer] = tc_acts.detach()
                self.transcoder_output[layer] = tc_output.detach()

            def make_capture_hook(layer_idx: int):
                """Hook at mlp.hook_in: save the live ln_input for this forward pass."""

                def hook_fn(tensor: Tensor, hook: HookPoint) -> None:
                    self.current_ln_inputs[layer_idx] = tensor

                return hook_fn

            def make_skip_hook(layer_idx: int, tc_ref: "Transcoder"):
                """Hook at hook_mlp_out: apply the skip connection trick."""

                def hook_fn(mlp_out: Tensor, hook: HookPoint) -> Tensor:
                    if hasattr(tc_ref, "W_skip"):
                        # Use the live ln_input saved by the capture hook above.
                        # This keeps the skip's grad_fn inside the current graph, so
                        # backward() works correctly across multiple batch iterations.
                        ln_input = self.current_ln_inputs[layer_idx]
                        skip = ln_input @ tc_ref.W_skip
                    else:
                        skip = t.zeros_like(mlp_out)
                    # Skip connection trick: forward gives mlp_out, backward gives skip grad
                    return skip + (mlp_out - skip).detach()

                return hook_fn

            # Both hooks are PERMANENT so they survive FreezeHooks.__exit__ which only
            # calls reset_hooks(including_permanent=False).
            self.model.add_hook(input_hook, make_capture_hook(layer), is_permanent=True)
            self.model.add_hook(output_hook, make_skip_hook(layer, tc), is_permanent=True)


    def remove(self):
        """Remove all hooks, including permanent ones added by install()."""
        self.model.reset_hooks(including_permanent=True)
```
</details>

### Sanity check: transcoder layer당 평균 L0

attribution graph를 구축하기 전에, transcoder가 token당 적절한 수의 active feature를 생성하고 있는지 확인해 보겠습니다. sparse autoencoder의 **L0**는 token 위치당 0이 아닌 latent의 평균 개수입니다. 만약 L0가 높다면 (예: 50 이상), 이는 graph에서 latent node 수가 예상보다 많게 나타나는 이유가 되며, 아마도 잘못된 transcoder variant가 로드되었음을 의미합니다 (우리는 `l0_big`이 아니라 `l0_small`을 원합니다). 처음 `START_POSN`개의 token(chat-formatting token)은 의미 있는 콘텐츠 위치가 아니므로 제외합니다.

In [ ]:
print("Average L0 per transcoder layer (excluding first 4 tokens):")
for layer in range(len(transcoders)):
    acts = tc_hooks.transcoder_acts[layer][0, START_POSN:, :]  # (seq - START_POSN, d_enc)
    l0_per_pos = (acts > 0).float().sum(dim=-1)  # (seq - START_POSN,)
    avg_l0 = l0_per_pos.mean().item()
    print(f"  Layer {layer}: avg L0 = {avg_l0:.1f}")
    assert avg_l0 <= 50, (
        f"Layer {layer} avg L0 = {avg_l0:.1f} exceeds 50 - check that you loaded the "
        f"'l0_small' transcoder variant, not 'l0_big'"
    )
print("L0 sanity check passed!")

## attribution graph 구축하기

이제 모델을 선형화할 수 있으므로, attribution graph의 **노드(nodes)**와 **엣지(edges)**를 정의해야 합니다.

### 주요 logit 선택

출력 노드는 **logit directions**, 즉 가장 높게 예측된 token들에 대한 unembedding 벡터들입니다. vocabulary의 모든 token을 사용하는 것은 낭비가 되므로, 모델의 확률 질량 대부분을 차지하는 주요 token들만 선택합니다.

구체적으로, 예측 확률 기준 상위 k개의 token을 선택하며(이 token들이 전체 확률의 일정 임계값 이상을 차지하도록 k를 정하거나, 3-5와 같이 고정된 k를 사용합니다), 이들의 **demeaned** unembedding 벡터를 reading directions로 사용합니다.

왜 demean 과정을 거칠까요? logit lens에 따르면, 마지막 위치의 residual stream은 unembedding 행렬 `W_U` 와 곱해져 logit을 생성합니다. 하지만 residual stream에 더해진 상수 벡터는 모든 logit에 동일하게 영향을 미치며 softmax 출력값을 변화시키지 않습니다. 따라서 각 선택된 열에서 `W_U` 열들의 평균을 빼줌으로써, 해당 token의 logit을 평균과 다르게 만드는 요소에 집중합니다.

### 연습 문제 - `compute_salient_logits` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

가장 높은 확률로 예측된 token들을 선택하고, 이들의 demeaned unembedding 벡터(출력 노드의 reading 벡터로 사용됩니다)를 반환하는 함수를 구현하십시오.

In [ ]:
def compute_salient_logits(
    model: HookedSAETransformer,
    logits: Float[Tensor, "batch seq d_vocab"],
    n_output_nodes: int = 3,
) -> tuple[Float[Tensor, "n_output d_model"], list[tuple[str, float]]]:
    """
    Select the top predicted tokens and return their demeaned W_U columns.

    Args:
        model: The transformer model.
        logits: Full logit tensor from model forward pass.
        n_output_nodes: Number of top tokens to select.

    Returns:
        reading_vecs: Demeaned unembedding vectors for top tokens, shape (n_output, d_model).
        top_token_info: List of (token_string, probability) tuples for the selected tokens.
    """
    raise NotImplementedError()

In [ ]:
reading_vecs, top_token_info = compute_salient_logits(gemma, logits_original)
print("Top predicted tokens:")
for tok_str, prob in top_token_info:
    print(f"  {tok_str!r}: p={prob:.4f}")
print(f"Reading vectors shape: {reading_vecs.shape}")

<details><summary>솔루션</summary>

```python
def compute_salient_logits(
    model: HookedSAETransformer,
    logits: Float[Tensor, "batch seq d_vocab"],
    n_output_nodes: int = 3,
) -> tuple[Float[Tensor, "n_output d_model"], list[tuple[str, float]]]:
    """
    Select the top predicted tokens and return their demeaned W_U columns.

    Args:
        model: The transformer model.
        logits: Full logit tensor from model forward pass.
        n_output_nodes: Number of top tokens to select.

    Returns:
        reading_vecs: Demeaned unembedding vectors for top tokens, shape (n_output, d_model).
        top_token_info: List of (token_string, probability) tuples for the selected tokens.
    """
    # Get probabilities for the last sequence position
    final_logits = logits[0, -1]  # (d_vocab,)
    probs = t.softmax(final_logits, dim=-1)

    # Get top-k tokens
    top_probs, top_tokens = probs.topk(n_output_nodes)

    # Get the unembedding matrix and select columns for top tokens
    W_U = model.W_U  # (d_model, d_vocab)
    selected_cols = W_U[:, top_tokens].T  # (n_output, d_model)

    # Demean: subtract the mean unembedding vector
    W_U_mean = W_U.mean(dim=-1)  # (d_model,)
    reading_vecs = selected_cols - W_U_mean.unsqueeze(0)

    # Get token strings
    top_token_info = [(model.tokenizer.decode(tok.item()), prob.item()) for tok, prob in zip(top_tokens, top_probs)]

    return reading_vecs, top_token_info
```
</details>

### 읽기/쓰기 벡터 추상화

이제 각 노드 타입에 대한 **읽기 및 쓰기 벡터(reading and writing vectors)**를 이해해야 합니다. 이것이 attribution graph의 개념적 핵심입니다.

모든 latent는 두 가지 방식으로 residual stream과 상호작용합니다. latent가 활성화될 때, residual stream에 `activation * W_dec[latent]`를 *씁니다(writes)*. 그리고 latent는 residual stream이 자신의 encoder 방향과 얼마나 일치하는지에 따라 활성화되므로, `W_enc.T[latent]`를 통해 *읽습니다(reads)*.

노드 타입별 읽기 및 쓰기 벡터의 구성은 다음과 같습니다:

| 노드 타입 | 쓰기 벡터 | 읽기 벡터 |
|-----------|---------------|----------------|
| Token embedding | token embedding 벡터 | Zero vector (embedding은 입력입니다) |
| Transcoder latent | `activation * W_dec[latent]` | `W_enc.T[latent]` |
| MLP error | `mlp_output - transcoder_reconstruction` | Zero vector (error 항은 읽지 않습니다) |
| Output logit | Zero vector (logit은 출력입니다) | Demeaned `W_U[:, token]` |

노드 A에서 노드 B로의 edge weight를 계산하려면, A의 쓰기 벡터를 가져와 A와 B 사이의 고정된 중간 레이어(attention + skip connections)를 통해 매핑한 후, B의 읽기 벡터와 내적(dot product)을 수행합니다.

자동(gradient 기반) 방법에서는 이러한 매핑을 명시적으로 수행할 필요가 없습니다. 대신, B의 읽기 벡터를 gradient seed로 주입하고 고정된 모델을 통해 backward pass를 실행하며, 이 과정에서 매핑이 암시적으로 계산됩니다. 그러면 edge weight는 결과로 나온 gradient와 A의 쓰기 벡터의 내적이 됩니다.

우리는 세 단계에 걸쳐 그래프 노드를 구축할 것이며, 각 단계는 개별 연습 문제로 구성됩니다. 먼저 embedding 노드를 생성하고, 그다음 레이어별 feature 및 error 노드를 생성하며, 마지막으로 logit 노드를 생성합니다. 이 세 단계를 모두 마친 후, 이를 하나의 전체 `GraphNodes` 객체로 결합합니다.

노드 순서는 다음과 같아야 합니다: [input embeddings] [layer 0 features] [layer 0 error] [layer 1 features] ... [layer N-1 error] [output logits]. 이 순서가 중요한 이유는 인접 행렬(adjacency matrix)을 **하삼각 행렬(lower triangular)**로 만들기 때문입니다(정보가 이전 레이어에서 이후 레이어로만 흐릅니다). 이는 행렬이 **멱영(nilpotent)**임을 의미합니다. 우리는 나중에 효율적인 influence 계산을 위해 이 성질을 이용할 것입니다.

In [ ]:
class NodeType(Enum):
    EMBEDDING = "embedding"
    LATENT = "latent"
    MLP_ERROR = "mlp_error"
    LOGIT = "logit"


@dataclass
class NodeInfo:
    """Metadata about a single node in the attribution graph."""

    node_type: NodeType
    layer: int | str  # "E" for embeddings, layer_idx for features, "L" for logits
    ctx_idx: int  # sequence position
    feature: int  # feature index (token ID for embeds, feature ID for latents, token ID for logits)
    activation: float = 0.0  # feature activation (for latent nodes)
    token_prob: float = 0.0  # probability (for logit nodes)
    str_token: str = ""  # string representation (for embed/logit nodes)
    label: str = ""  # human-readable label

    @property
    def node_id(self) -> str:
        return f"{self.layer}_{self.feature}_{self.ctx_idx}"


@dataclass
class GraphNodes:
    """Container for all nodes in an attribution graph, with their reading/writing vectors."""

    nodes: list[NodeInfo] = field(default_factory=list)
    writing_vecs: Tensor | None = None  # (n_nodes, d_model) or None
    reading_vecs: Tensor | None = None  # (n_nodes, d_model) or None
    node_range_dict: dict = field(default_factory=dict)  # layer -> (start_idx, end_idx)
    seq_len: int = 0
    n_layers: int = 0

### 연습 문제 (1/3) - `build_embedding_nodes` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> ```

attribution graph를 구축하는 첫 번째 단계는 입력 embedding에 대해 token 위치당 하나의 노드를 생성하는 것입니다. 각 embedding 노드는 `EMBEDDING` 타입을 가지며, 자신의 embedding 벡터를 residual stream에 쓰고, 읽기 벡터는 0입니다 (embedding은 아무것도 읽지 않기 때문입니다).

embedding 벡터를 가져오기 위해 `cache["blocks.0.hook_resid_pre"]`을 사용해야 합니다.

In [ ]:
def build_embedding_nodes(
    model: HookedSAETransformer,
    cache: ActivationCache,
    tokens: Int[Tensor, "1 seq"],
) -> tuple[list[NodeInfo], list[Tensor], list[Tensor]]:
    """
    Build embedding nodes (one per token position) with their writing and reading vectors.

    Args:
        model: The transformer model.
        cache: Activation cache from forward pass.
        tokens: Input token IDs, shape (1, seq).

    Returns:
        Tuple of (nodes, writing_vecs, reading_vecs) where each list has length seq_len.
        Writing vecs are the token embeddings, reading vecs are zeros.
    """
    raise NotImplementedError()


embed_nodes, embed_writing, embed_reading = build_embedding_nodes(model=gemma, cache=cache, tokens=tokens)
seq_len = tokens.shape[1]

assert len(embed_nodes) == seq_len, f"Expected {seq_len} embedding nodes, got {len(embed_nodes)}"
assert all(n.node_type == NodeType.EMBEDDING for n in embed_nodes), "All embedding nodes should have type EMBEDDING"
for i, wv in enumerate(embed_writing):
    assert wv.shape == (gemma.cfg.d_model,), f"Writing vec {i} has wrong shape: {wv.shape}"
    assert wv.abs().sum() > 0, f"Writing vec {i} is all zeros but should be a token embedding"
for i, rv in enumerate(embed_reading):
    assert rv.shape == (gemma.cfg.d_model,), f"Reading vec {i} has wrong shape: {rv.shape}"
    t.testing.assert_close(rv, t.zeros_like(rv), msg=f"Reading vec {i} should be all zeros")

print("All build_embedding_nodes tests passed!")

<details><summary>솔루션</summary>

```python
def build_embedding_nodes(
    model: HookedSAETransformer,
    cache: ActivationCache,
    tokens: Int[Tensor, "1 seq"],
) -> tuple[list[NodeInfo], list[Tensor], list[Tensor]]:
    """
    Build embedding nodes (one per token position) with their writing and reading vectors.

    Args:
        model: The transformer model.
        cache: Activation cache from forward pass.
        tokens: Input token IDs, shape (1, seq).

    Returns:
        Tuple of (nodes, writing_vecs, reading_vecs) where each list has length seq_len.
        Writing vecs are the token embeddings, reading vecs are zeros.
    """
    seq_len = tokens.shape[1]
    d_model = model.cfg.d_model
    str_tokens = [model.tokenizer.decode(t_id.item()) for t_id in tokens[0]]

    nodes: list[NodeInfo] = []
    writing_vecs: list[Tensor] = []
    reading_vecs: list[Tensor] = []

    embed_vecs = cache["blocks.0.hook_resid_pre"][0]  # (seq, d_model)
    for pos in range(seq_len):
        nodes.append(
            NodeInfo(
                node_type=NodeType.EMBEDDING,
                layer="E",
                ctx_idx=pos,
                feature=tokens[0, pos].item(),
                str_token=str_tokens[pos],
            )
        )
        writing_vecs.append(embed_vecs[pos])
        reading_vecs.append(t.zeros(d_model, device=embed_vecs.device))

    return nodes, writing_vecs, reading_vecs
```
</details>

### 연습 문제 (2/3) - `build_intermediate_nodes` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> ```

다음으로 중간 노드들을 구축합니다. 각 layer와 각 position(`start_posn` 이후부터)에 대해, activation 기준 상위 `top_k` 개의 transcoder latent를 선택하고 각각에 대해 LATENT 노드를 생성합니다. 또한 재구성 오차(reconstruction error)를 캡처하기 위해 position 및 layer당 하나의 MLP_ERROR 노드를 생성합니다.

외부 루프 구조(layer/position 반복 및 활성 feature 인덱스 추출)는 이미 제공되었습니다. 여러분은 실제 노드와 벡터를 생성하는 내부 루프 본문을 채워야 합니다:

- 각 활성 feature에 대해: writing vector `activation * W_dec[feature_idx]` (activation으로 스케일링된 decoder 방향)와 reading vector `W_enc.T[feature_idx]` (encoder 방향)를 가진 LATENT 노드를 생성합니다.
- 특정 position의 모든 feature를 처리한 후: writing vector `mlp_output[pos] - tc_output[pos]`와 reading vector zeros를 가진 하나의 MLP_ERROR 노드를 생성합니다.

<details><summary>힌트 - MLP error 계산하기</summary>

`tc.cfg.metadata.hook_name_out` (예: `"blocks.0.hook_mlp_out"`)를 사용하여 cache에서 MLP output을 가져올 수 있습니다. 그러면 error는 `mlp_output[pos] - tc_output[pos]` 입니다.

</details>

In [ ]:
def build_intermediate_nodes(
    transcoders: dict[int, "Transcoder"],
    cache: ActivationCache,
    tc_hooks: TranscoderReplacementHooks,
    tokens: Int[Tensor, "1 seq"],
    d_model: int,
    start_posn: int = 4,
    top_k: int = 5,
) -> tuple[list[NodeInfo], list[Tensor], list[Tensor], dict]:
    """
    Build intermediate (LATENT + MLP_ERROR) nodes for all layers.

    Args:
        transcoders: Dict mapping layer -> transcoder.
        cache: Activation cache from forward pass.
        tc_hooks: TranscoderReplacementHooks with computed activations.
        tokens: Input token IDs, shape (1, seq).
        d_model: Model hidden dimension.
        start_posn: First position to include (masks chat formatting tokens).
        top_k: Number of top-activating features to include per position per layer.

    Returns:
        Tuple of (nodes, writing_vecs, reading_vecs, node_range_dict) where node_range_dict
        maps each layer index to (start_idx, end_idx) within the returned lists.
    """
    seq_len = tokens.shape[1]
    n_layers = len(transcoders)
    device = cache["blocks.0.hook_resid_pre"].device

    nodes: list[NodeInfo] = []
    writing_vecs: list[Tensor] = []
    reading_vecs: list[Tensor] = []
    node_range_dict = {}

    for layer in range(n_layers):
        tc = transcoders[layer]
        layer_start = len(nodes)

        tc_acts = tc_hooks.transcoder_acts[layer][0]  # (seq, d_enc)
        tc_output = tc_hooks.transcoder_output[layer][0]  # (seq, d_model)

        # Get MLP output from cache for error computation
        mlp_output = cache[tc.cfg.metadata.hook_name_out][0]  # (seq, d_model)

        W_dec = tc.W_dec.detach()  # (d_enc, d_model)
        W_enc_T = tc.W_enc.detach().T  # (d_enc, d_model)

        for pos in range(start_posn, seq_len):
            acts = tc_acts[pos]  # (d_enc,)
            # Select top-k features by activation
            k = min(top_k, (acts > 0).sum().item())  # don't include zero-activation features
            if k > 0:
                _, active_indices = acts.topk(k)
            else:
                active_indices = t.where(acts > 0)[0]  # fallback: empty if no active features

            # TODO: For each active feature, create a LATENT node with the correct writing
            # vector (activation * W_dec[feature]) and reading vector (W_enc.T[feature]).
            # After all features at this position, create one MLP_ERROR node with writing
            # vector = mlp_output[pos] - tc_output[pos] and reading vector = zeros.
            pass

        node_range_dict[layer] = (layer_start, len(nodes))

    return nodes, writing_vecs, reading_vecs, node_range_dict


inter_nodes, inter_writing, inter_reading, inter_range_dict = build_intermediate_nodes(
    transcoders=transcoders,
    cache=cache,
    tc_hooks=tc_hooks,
    tokens=tokens,
    d_model=gemma.cfg.d_model,
    start_posn=START_POSN,
)

n_layers = len(transcoders)
seq_len = tokens.shape[1]
active_positions = seq_len - START_POSN

# All LATENT nodes should have positive activations
latent_nodes = [n for n in inter_nodes if n.node_type == NodeType.LATENT]
assert len(latent_nodes) > 0, "Expected at least some LATENT nodes"
assert all(n.activation > 0 for n in latent_nodes), "All LATENT nodes should have positive activations"

# MLP_ERROR nodes: one per position per layer
error_nodes = [n for n in inter_nodes if n.node_type == NodeType.MLP_ERROR]
assert len(error_nodes) == active_positions * n_layers, (
    f"Expected {active_positions * n_layers} MLP_ERROR nodes, got {len(error_nodes)}"
)

# Writing vectors for LATENT nodes should be non-zero
latent_indices = [i for i, n in enumerate(inter_nodes) if n.node_type == NodeType.LATENT]
for idx in latent_indices[:5]:  # check first 5
    assert inter_writing[idx].abs().sum() > 0, "LATENT writing vectors should be non-zero"

# Reading vectors for MLP_ERROR nodes should be zeros
error_indices = [i for i, n in enumerate(inter_nodes) if n.node_type == NodeType.MLP_ERROR]
for idx in error_indices[:5]:  # check first 5
    t.testing.assert_close(inter_reading[idx], t.zeros_like(inter_reading[idx]))

# node_range_dict should have entries for each layer
assert set(inter_range_dict.keys()) == set(range(n_layers)), (
    f"node_range_dict should have entries for layers 0..{n_layers - 1}, got keys {set(inter_range_dict.keys())}"
)

print("All build_intermediate_nodes tests passed!")

<details><summary>솔루션</summary>

```python
def build_intermediate_nodes(
    transcoders: dict[int, "Transcoder"],
    cache: ActivationCache,
    tc_hooks: TranscoderReplacementHooks,
    tokens: Int[Tensor, "1 seq"],
    d_model: int,
    start_posn: int = 4,
    top_k: int = 5,
) -> tuple[list[NodeInfo], list[Tensor], list[Tensor], dict]:
    """
    Build intermediate (LATENT + MLP_ERROR) nodes for all layers.

    Args:
        transcoders: Dict mapping layer -> transcoder.
        cache: Activation cache from forward pass.
        tc_hooks: TranscoderReplacementHooks with computed activations.
        tokens: Input token IDs, shape (1, seq).
        d_model: Model hidden dimension.
        start_posn: First position to include (masks chat formatting tokens).
        top_k: Number of top-activating features to include per position per layer.

    Returns:
        Tuple of (nodes, writing_vecs, reading_vecs, node_range_dict) where node_range_dict
        maps each layer index to (start_idx, end_idx) within the returned lists.
    """
    seq_len = tokens.shape[1]
    n_layers = len(transcoders)
    device = cache["blocks.0.hook_resid_pre"].device

    nodes: list[NodeInfo] = []
    writing_vecs: list[Tensor] = []
    reading_vecs: list[Tensor] = []
    node_range_dict = {}

    for layer in range(n_layers):
        tc = transcoders[layer]
        layer_start = len(nodes)

        tc_acts = tc_hooks.transcoder_acts[layer][0]  # (seq, d_enc)
        tc_output = tc_hooks.transcoder_output[layer][0]  # (seq, d_model)

        # Get MLP output from cache for error computation
        mlp_output = cache[tc.cfg.metadata.hook_name_out][0]  # (seq, d_model)

        W_dec = tc.W_dec.detach()  # (d_enc, d_model)
        W_enc_T = tc.W_enc.detach().T  # (d_enc, d_model)

        for pos in range(start_posn, seq_len):
            acts = tc_acts[pos]  # (d_enc,)
            # Select top-k features by activation
            k = min(top_k, (acts > 0).sum().item())  # don't include zero-activation features
            if k > 0:
                _, active_indices = acts.topk(k)
            else:
                active_indices = t.where(acts > 0)[0]  # fallback: empty if no active features

            for feat_idx in active_indices:
                feat_act = acts[feat_idx].item()
                nodes.append(
                    NodeInfo(
                        node_type=NodeType.LATENT,
                        layer=layer,
                        ctx_idx=pos,
                        feature=feat_idx.item(),
                        activation=feat_act,
                    )
                )
                # Writing vector: activation * decoder direction
                writing_vecs.append(feat_act * W_dec[feat_idx])
                # Reading vector: encoder direction
                reading_vecs.append(W_enc_T[feat_idx])

            # MLP error node for this position
            mlp_error = mlp_output[pos] - tc_output[pos]
            nodes.append(
                NodeInfo(
                    node_type=NodeType.MLP_ERROR,
                    layer=layer,
                    ctx_idx=pos,
                    feature=0,
                )
            )
            writing_vecs.append(mlp_error.detach())
            reading_vecs.append(t.zeros(d_model, device=device))

        node_range_dict[layer] = (layer_start, len(nodes))

    return nodes, writing_vecs, reading_vecs, node_range_dict
```
</details>

마지막으로, 출력(logit) 노드들을 생성합니다. `top_token_info`의 각 항목당 하나의 노드가 있으며, 마지막 시퀀스 위치에 배치됩니다. 이 노드들은 writing 벡터가 0이며(residual stream에 쓰지 않습니다), reading 벡터는 `reading_vecs_logit`에서 가져온 demeaned unembedding 컬럼들입니다. 이 함수는 `build_embedding_nodes`와 동일한 패턴을 따르므로 이미 제공해 드렸습니다.

In [ ]:
def build_logit_nodes(
    reading_vecs_logit: Float[Tensor, "n_output d_model"],
    top_token_info: list[tuple[str, float]],
    seq_len: int,
    d_model: int,
) -> tuple[list[NodeInfo], list[Tensor], list[Tensor]]:
    """
    Build logit (output) nodes with their writing and reading vectors.

    Args:
        reading_vecs_logit: Demeaned W_U columns for output nodes, shape (n_output, d_model).
        top_token_info: List of (token_string, probability) for output nodes.
        seq_len: Total sequence length (logit nodes are placed at seq_len - 1).
        d_model: Model hidden dimension.

    Returns:
        Tuple of (nodes, writing_vecs, reading_vecs) where each list has length len(top_token_info).
        Writing vecs are zeros, reading vecs are the logit reading vectors.
    """
    device = reading_vecs_logit.device

    nodes: list[NodeInfo] = []
    writing_vecs: list[Tensor] = []
    reading_vecs: list[Tensor] = []

    for i, (tok_str, prob) in enumerate(top_token_info):
        nodes.append(
            NodeInfo(
                node_type=NodeType.LOGIT,
                layer="L",
                ctx_idx=seq_len - 1,
                feature=i,
                token_prob=prob,
                str_token=tok_str,
            )
        )
        writing_vecs.append(t.zeros(d_model, device=device))
        reading_vecs.append(reading_vecs_logit[i])

    return nodes, writing_vecs, reading_vecs

이제 세 가지 helper 함수가 모두 완성되었으므로, 이를 하나의 `build_graph_nodes` 함수로 결합합니다. 이 wrapper는 여러분에게 제공되며, 단순히 세 개의 하위 함수를 호출하고 결과를 조립하여 `GraphNodes` 객체를 반환합니다.

In [ ]:
def build_graph_nodes(
    model: HookedSAETransformer,
    transcoders: dict[int, "Transcoder"],
    cache: ActivationCache,
    tc_hooks: TranscoderReplacementHooks,
    reading_vecs_logit: Float[Tensor, "n_output d_model"],
    top_token_info: list[tuple[str, float]],
    tokens: Int[Tensor, "1 seq"],
    start_posn: int = 4,
    top_k: int = 5,
) -> GraphNodes:
    """
    Build all graph nodes with their reading and writing vectors by calling the three sub-functions
    and assembling the results.

    Args:
        model: The transformer model.
        transcoders: Dict mapping layer -> transcoder.
        cache: Activation cache from forward pass.
        tc_hooks: TranscoderReplacementHooks with computed activations.
        reading_vecs_logit: Demeaned W_U columns for output nodes.
        top_token_info: List of (token_string, probability) for output nodes.
        tokens: Input token IDs, shape (1, seq).
        start_posn: First position to include (masks chat formatting tokens).
        top_k: Number of top-activating features to include per position per layer.

    Returns:
        GraphNodes containing all nodes, their reading/writing vectors, and index ranges.
    """
    seq_len = tokens.shape[1]
    n_layers = len(transcoders)
    d_model = model.cfg.d_model

    all_nodes: list[NodeInfo] = []
    all_writing: list[Tensor] = []
    all_reading: list[Tensor] = []
    node_range_dict = {}

    # 1. Embedding nodes
    embed_nodes, embed_writing, embed_reading = build_embedding_nodes(model, cache, tokens)
    all_nodes.extend(embed_nodes)
    all_writing.extend(embed_writing)
    all_reading.extend(embed_reading)
    node_range_dict["E"] = (0, len(all_nodes))

    # 2. Intermediate nodes (features + MLP error per layer)
    inter_nodes, inter_writing, inter_reading, inter_ranges = build_intermediate_nodes(
        transcoders,
        cache,
        tc_hooks,
        tokens,
        d_model,
        start_posn,
        top_k,
    )
    offset = len(all_nodes)
    all_nodes.extend(inter_nodes)
    all_writing.extend(inter_writing)
    all_reading.extend(inter_reading)
    for layer_key, (s, e) in inter_ranges.items():
        node_range_dict[layer_key] = (s + offset, e + offset)

    # 3. Logit nodes
    logit_start = len(all_nodes)
    logit_nodes, logit_writing, logit_reading = build_logit_nodes(
        reading_vecs_logit,
        top_token_info,
        seq_len,
        d_model,
    )
    all_nodes.extend(logit_nodes)
    all_writing.extend(logit_writing)
    all_reading.extend(logit_reading)
    node_range_dict["L"] = (logit_start, len(all_nodes))

    return GraphNodes(
        nodes=all_nodes,
        writing_vecs=t.stack(all_writing),
        reading_vecs=t.stack(all_reading),
        node_range_dict=node_range_dict,
        seq_len=seq_len,
        n_layers=n_layers,
    )

In [ ]:
graph = build_graph_nodes(
    model=gemma,
    transcoders=transcoders,
    cache=cache,
    tc_hooks=tc_hooks,
    reading_vecs_logit=reading_vecs,
    top_token_info=top_token_info,
    tokens=tokens,
    start_posn=START_POSN,
)

# Print some stats
n_embeds = sum(1 for n in graph.nodes if n.node_type == NodeType.EMBEDDING)
n_latents = sum(1 for n in graph.nodes if n.node_type == NodeType.LATENT)
n_errors = sum(1 for n in graph.nodes if n.node_type == NodeType.MLP_ERROR)
n_logits = sum(1 for n in graph.nodes if n.node_type == NodeType.LOGIT)
print(f"Graph has {len(graph.nodes)} total nodes:")
print(f"  {n_embeds} embedding nodes")
print(f"  {n_latents} latent nodes")
print(f"  {n_errors} MLP error nodes")
print(f"  {n_logits} logit output nodes")
print(f"Writing vectors shape: {graph.writing_vecs.shape}")
print(f"Reading vectors shape: {graph.reading_vecs.shape}")

### Attribution 설정

adjacency matrix를 계산하기 전에, backward pass를 적절하게 설정하는 helper가 필요합니다. 아이디어는 다음과 같습니다: (1) frozen model을 통해 forward pass를 실행합니다 (`FreezeHooks` 및 `TranscoderReplacementHooks` 활성화), (2) target node의 위치와 layer에서, residual stream과 target의 reading vector의 dot product를 계산합니다, (3) 이 scalar를 frozen model을 통해 backpropagate 합니다, (4) 각 source node의 위치에서 gradient를 읽어 source의 writing vector와 contract 합니다.

target node들의 batch에 대해 1~3단계를 처리하는 이 설정 함수를 제공해 드리겠습니다. 주의 깊게 읽어보시기 바랍니다.

**Hook 설계 노트 (Option A).** 함수 내부에 `with freeze:`를 중첩시키고 gradient-capture hook을 별도로 설치/제거하는 대신, `setup_attribution`은 모든 것을 결합한 단일 `model.hooks()` 호출을 사용합니다:

```python
with model.hooks(fwd_hooks=freeze.fwd_hooks + capture_fwd_hooks, bwd_hooks=capture_bwd_hooks):
    model(tokens).backward()
```

- `freeze.fwd_hooks` - freeze hook을 `(name, fn)` 쌍(TL-native 형식)으로 반환합니다.
- `capture_fwd_hooks` - objective를 구축할 수 있도록 각 residual-stream activation tensor를 저장합니다.
- `capture_bwd_hooks` - **backward pass 중에 직접** gradient를 캡처하여, `tensor.retain_grad()` + `tensor.grad` bookkeeping의 필요성을 제거합니다.

`TranscoderReplacementHooks`는 **permanent** hook(`is_permanent=True`으로 등록됨)을 사용하므로, `model.hooks()` context 동안 계속 활성 상태로 유지되며 `tc_hooks.remove()`에 의해서만 제거됩니다.

In [ ]:
def setup_attribution(
    model: HookedSAETransformer,
    tokens: Tensor,
    freeze: FreezeHooks,
    target_reading_vecs: Float[Tensor, "batch d_model"],
    target_positions: Int[Tensor, " batch"],
    target_layers: list[int | str],
) -> dict[str, Tensor]:
    """
    Run forward pass through frozen model, inject reading vectors as gradient seeds at
    target positions/layers, and return gradients at all residual stream positions.

    This function handles the "backward pass" part of attribution: for each target node,
    it computes d(objective)/d(resid) at every layer, where objective = dot(resid[target_pos], reading_vec).

    Uses TransformerLens-native hooks via a single model.hooks() context (Option A):
      - fwd_hooks = freeze.fwd_hooks + capture_fwd_hooks
          freeze.fwd_hooks  : replace attention patterns/LN scales with frozen values
          capture_fwd_hooks : store each residual-stream tensor for objective computation
      - bwd_hooks = capture_bwd_hooks
          capture_bwd_hooks : receive gradients directly during backward, no retain_grad() needed

    TranscoderReplacementHooks are permanent and stay active inside this context.

    Args:
        model: The transformer model.
        tokens: Input tokens, shape (1, seq).
        freeze: FreezeHooks with cached frozen values (provides fwd_hooks property).
        target_reading_vecs: Reading vectors for target nodes, shape (batch, d_model).
        target_positions: Sequence positions of target nodes, shape (batch,).
        target_layers: Layer identifiers for target nodes (int for intermediate, "L" for logits).

    Returns:
        grads: Dict mapping hook names -> gradient tensors at each residual stream position.
    """
    # Detach reading vectors - attribution only needs d(objective)/d(resid),
    # not d(objective)/d(W_U). Without this, reading_vecs built from W_U (a
    # parameter with requires_grad=True) carry a grad_fn that gets freed after
    # batch 1's backward(), causing "backward through graph a second time" on
    # subsequent batches.
    target_reading_vecs = target_reading_vecs.detach()

    batch_size = target_reading_vecs.shape[0]

    # Tensors captured during the forward pass (for computing objectives)
    captured_tensors: dict[str, Tensor] = {}
    # Gradients captured directly during the backward pass (the actual output)
    captured_grads: dict[str, Tensor] = {}

    # Names of residual-stream tensors to capture
    capture_names = (
        [f"blocks.{l}.hook_resid_post" for l in range(model.cfg.n_layers)]
        + [f"blocks.{l}.hook_resid_mid" for l in range(model.cfg.n_layers)]
        + ["blocks.0.hook_resid_pre"]
    )

    def make_fwd_capture(name: str) -> Callable:
        """Forward hook: store the activation tensor so we can build objectives from it."""

        def hook_fn(tensor: Tensor, hook: HookPoint) -> None:
            captured_tensors[name] = tensor

        return hook_fn

    def make_bwd_capture(name: str) -> Callable:
        """Backward hook: receive the gradient directly during backward, no retain_grad() needed."""

        def hook_fn(grad: Tensor, hook: HookPoint) -> None:
            captured_grads[name] = grad.detach()

        return hook_fn

    capture_fwd_hooks = [(name, make_fwd_capture(name)) for name in capture_names]
    capture_bwd_hooks = [(name, make_bwd_capture(name)) for name in capture_names]

    # Single model.hooks() context combining freeze hooks + capture hooks (Option A).
    # TranscoderReplacementHooks are permanent and stay active throughout.
    # At context exit, only non-permanent hooks (freeze + capture) are removed.
    with model.hooks(
        fwd_hooks=freeze.fwd_hooks + capture_fwd_hooks,
        bwd_hooks=capture_bwd_hooks,
    ):
        # Forward pass (objectives are computed from captured residual tensors)
        model(tokens.expand(batch_size, -1))

        # Compute objectives for each target node
        # Use torch.stack instead of in-place assignment to a leaf tensor, to
        # avoid autograd issues with in-place ops on zero-initialized tensors.
        objective_terms = []
        for i in range(batch_size):
            pos = target_positions[i]
            layer = target_layers[i]
            if layer == "L":
                # For logit nodes, use the final residual stream (pre-unembedding)
                resid = captured_tensors[f"blocks.{model.cfg.n_layers - 1}.hook_resid_post"][i, pos]
            else:
                # For intermediate nodes, use resid_mid at this layer
                # (after attention, before MLP - this is what the feature reads from)
                resid = captured_tensors[f"blocks.{layer}.hook_resid_mid"][i, pos]
            objective_terms.append((resid * target_reading_vecs[i]).sum())

        # Backward pass - fires bwd hooks, populating captured_grads
        total_objective = t.stack(objective_terms).sum()
        total_objective.backward()

    return captured_grads

attribution 알고리즘을 두 단계로 구현하겠습니다. 먼저 backward pass 배치를 준비하고(레이어별로 target 노드를 그룹화하고 reading 벡터를 배치화합니다), 그 다음 실제 backward pass를 실행하여 gradient를 source writing 벡터와 contract 합니다.

인접 행렬(adjacency matrix) `A[target, source]` 은 source에서 target으로 가는 edge weight를 저장합니다. 노드들이 레이어 순으로 정렬되어 있으므로, 이 행렬은 **엄격한 하삼각 행렬(strictly lower triangular)** 입니다(target은 이전 레이어로부터만 값을 받을 수 있기 때문입니다).

### 연습 문제 (1/2) - `prepare_backward_batches` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> ```

첫 번째 단계는 backward pass를 위해 타겟 노드들을 batch로 구성하는 것입니다. 그래프의 각 layer에 대해 (입력 전용인 embeddings는 제외), 타겟 노드들을 크기 `batch_size`의 chunk로 그룹화합니다. 각 chunk에 대해 reading vector, position, 그리고 layer 식별자를 수집합니다.

작성하신 함수는 튜플의 리스트를 반환해야 하며, 각 튜플은 다음을 포함합니다: 노드 리스트 내 batch의 전역 시작 인덱스, 타겟 노드들의 batch, stacked tensor 형태의 reading vector들, position 텐서, 그리고 layer 식별자 리스트입니다.

<details><summary>힌트 - layer 반복문 </summary>

layer를 반복하기 위해 `graph.node_range_dict`을 사용합니다. `"E"` 키(embeddings)는 건너뜁니다. 각 layer에 대해, `node_range_dict[key]`은 노드 인덱스의 `(start, end)` 범위를 제공합니다.

</details>

In [ ]:
def prepare_backward_batches(
    graph: GraphNodes,
    batch_size: int = 8,
) -> list[tuple[int, list[NodeInfo], Tensor, Tensor, list[int | str]]]:
    """
    Group target nodes by layer and split into batches for backward passes.

    Args:
        graph: GraphNodes containing all nodes and their reading/writing vectors.
        batch_size: Number of target nodes to process per backward pass.

    Returns:
        List of (global_start_idx, batch_nodes, reading_vecs, positions, layers) tuples, where:
            global_start_idx: Index of the first node in this batch within graph.nodes
            batch_nodes: List of NodeInfo for this batch
            reading_vecs: Stacked reading vectors, shape (batch, d_model)
            positions: Tensor of sequence positions, shape (batch,)
            layers: List of layer identifiers (int or "L")
    """
    raise NotImplementedError()


batches = prepare_backward_batches(graph, batch_size=8)

# Check we got some batches
assert len(batches) > 0, "Expected at least one batch"

# Verify batch structure
all_target_layers_covered = set()
total_target_nodes = 0
for global_start, batch_nodes, reading_vecs, positions, layers in batches:
    # Reading vectors should have correct shape
    assert reading_vecs.shape[0] == len(batch_nodes), "Reading vecs batch size should match number of nodes"
    assert reading_vecs.shape[1] == gemma.cfg.d_model, (
        f"Reading vecs should have d_model={gemma.cfg.d_model} columns"
    )

    # Positions should match batch size
    assert positions.shape[0] == len(batch_nodes), "Positions batch size should match number of nodes"

    # Track which layers are covered
    for n in batch_nodes:
        layer_key = n.layer
        all_target_layers_covered.add(layer_key)
    total_target_nodes += len(batch_nodes)

# All non-embedding layers should be covered
expected_layers = {k for k in graph.node_range_dict.keys() if k != "E"}
assert all_target_layers_covered == expected_layers, (
    f"Expected layers {expected_layers} to be covered, got {all_target_layers_covered}"
)

# Total target nodes should match all non-embedding nodes
n_non_embed = len(graph.nodes) - (graph.node_range_dict["E"][1] - graph.node_range_dict["E"][0])
assert total_target_nodes == n_non_embed, (
    f"Expected {n_non_embed} total target nodes across batches, got {total_target_nodes}"
)

print("All prepare_backward_batches tests passed!")

<details><summary>솔루션</summary>

```python
def prepare_backward_batches(
    graph: GraphNodes,
    batch_size: int = 8,
) -> list[tuple[int, list[NodeInfo], Tensor, Tensor, list[int | str]]]:
    """
    Group target nodes by layer and split into batches for backward passes.

    Args:
        graph: GraphNodes containing all nodes and their reading/writing vectors.
        batch_size: Number of target nodes to process per backward pass.

    Returns:
        List of (global_start_idx, batch_nodes, reading_vecs, positions, layers) tuples, where:
            global_start_idx: Index of the first node in this batch within graph.nodes
            batch_nodes: List of NodeInfo for this batch
            reading_vecs: Stacked reading vectors, shape (batch, d_model)
            positions: Tensor of sequence positions, shape (batch,)
            layers: List of layer identifiers (int or "L")
    """
    device = graph.reading_vecs.device
    batches = []

    for target_layer_key in list(graph.node_range_dict.keys()):
        if target_layer_key == "E":
            continue  # Embeddings are inputs only

        tgt_start, tgt_end = graph.node_range_dict[target_layer_key]
        target_nodes = graph.nodes[tgt_start:tgt_end]

        if len(target_nodes) == 0:
            continue

        for batch_start in range(0, len(target_nodes), batch_size):
            batch_end = min(batch_start + batch_size, len(target_nodes))
            batch_nodes = target_nodes[batch_start:batch_end]

            reading_vecs = graph.reading_vecs[tgt_start + batch_start : tgt_start + batch_end]
            positions = t.tensor([n.ctx_idx for n in batch_nodes], device=device)
            layers = [n.layer for n in batch_nodes]

            global_start_idx = tgt_start + batch_start
            batches.append((global_start_idx, batch_nodes, reading_vecs, positions, layers))

    return batches
```
</details>

### 연습 문제 (2/2) - `compute_adjacency_matrix` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 30-40 minutes on this exercise.
> This is the hardest and most important exercise in this section.
> ```

이제 준비된 batch들을 받아 모델을 통해 backward pass를 실행하고, 결과 gradient를 source writing vector와 contract 하여 adjacency matrix를 채우는 함수를 구현합니다.

각 batch에 대해 다음을 수행해야 합니다: (1) transcoder hook을 설치합니다, (2) batch의 reading vector, position, layer를 사용하여 `setup_attribution`를 호출합니다, (3) hook을 제거합니다, 그 다음 (4) 모든 source node를 순회하며 source position에서의 gradient와 source의 writing vector의 dot product를 계산합니다.

<details><summary>힌트 - gradient로부터 edge weight 계산하기</summary>

layer `l`의 source position `pos`에서의 gradient는 `(batch, d_model)`의 shape을 가집니다. position `pos`에 있는 source node `j`의 writing vector 또한 `(d_model,)`의 shape을 가집니다. edge weight는 이들의 dot product입니다:

```python
edge_weight = (grad_at_pos * writing_vec.unsqueeze(0)).sum(-1)  # (batch,)
```

각 source node의 layer에 대응하는 gradient hook 이름이 무엇인지 찾아내야 합니다.

</details>

<details><summary>힌트 - 어떤 gradient가 어떤 source node에 대응하는가</summary>

Embedding node는 `blocks.0.hook_resid_pre`의 gradient를 사용합니다. layer `l`의 Feature 및 MLP error node는 `blocks.{l}.hook_resid_post`의 gradient를 사용합니다. Logit node는 절대 source가 될 수 없습니다.

</details>

<details><summary>힌트 - 전체 구조</summary>

외부 루프는 batch를 순회합니다. 각 batch에 대해: hook을 설치하고, `setup_attribution`를 호출하고, hook을 제거한 뒤, 모든 source node를 루프로 돌며 gradient와 writing vector를 contract 합니다. 각 contraction은 adjacency matrix의 edge weight 한 열(column)을 제공합니다.

</details>

In [ ]:
def compute_adjacency_matrix(
    model: HookedSAETransformer,
    tokens: Tensor,
    graph: GraphNodes,
    freeze: FreezeHooks,
    tc_hooks: TranscoderReplacementHooks,
    batches: list[tuple[int, list[NodeInfo], Tensor, Tensor, list[int | str]]],
    start_posn: int = 4,
) -> Float[Tensor, "n_nodes n_nodes"]:
    """
    Run backward passes for each batch and contract gradients with writing vectors.

    Args:
        model: The transformer model.
        tokens: Input tokens, shape (1, seq).
        graph: GraphNodes containing all nodes and their reading/writing vectors.
        freeze: FreezeHooks with cached frozen values.
        tc_hooks: TranscoderReplacementHooks with installed hooks.
        batches: Output of prepare_backward_batches.
        start_posn: First position to include (masks formatting tokens).

    Returns:
        Adjacency matrix of shape (n_nodes, n_nodes), where A[j, i] is the edge weight
        from node i to node j.
    """
    # n_nodes = len(graph.nodes)
    # adjacency_matrix = t.zeros(n_nodes, n_nodes, device=tokens.device)
    #
    # for global_start_idx, batch_nodes, target_reading_vecs, target_positions, target_layers in batches:
    #     actual_batch_size = len(batch_nodes)
    #
    #     # Run backward pass to get gradients
    #     tc_hooks.install()
    #     grads = setup_attribution(
    #         model=model,
    #         tokens=tokens,
    #         freeze=freeze,
    #         target_reading_vecs=target_reading_vecs,
    #         target_positions=target_positions,
    #         target_layers=target_layers,
    #     )
    #     tc_hooks.remove()
    #
    #     # YOUR CODE HERE - contract gradients with source writing vectors
    #     # For each source node:
    #     #   1. Determine the correct gradient hook name (see hints above)
    #     #   2. Get the gradient at the source node's position: grads[grad_name][:actual_batch_size, src_node.ctx_idx]
    #     #   3. Dot product with the source's writing vector: graph.writing_vecs[src_idx]
    #     #   4. Zero out edges from masked positions (ctx_idx < start_posn)
    #     #   5. Store in adjacency_matrix[global_start_idx + b, src_idx]
    #     pass
    #
    # return adjacency_matrix


batches = prepare_backward_batches(graph, batch_size=8)
print("Got batches...")
adjacency_matrix = compute_adjacency_matrix(
    model=gemma,
    tokens=tokens,
    graph=graph,
    freeze=freeze,
    tc_hooks=tc_hooks,
    batches=batches,
    start_posn=START_POSN,
)

n_nodes = len(graph.nodes)
assert adjacency_matrix.shape == (n_nodes, n_nodes), (
    f"Expected shape ({n_nodes}, {n_nodes}), got {adjacency_matrix.shape}"
)

# Should be lower triangular (no backward connections in a transformer)
upper_tri_norm = t.triu(adjacency_matrix, diagonal=1).abs().sum().item()
assert upper_tri_norm < 1e-4, f"Upper triangle norm should be ~0, got {upper_tri_norm}"

# Should be sparse (most entries near zero)
sparsity = 1 - (adjacency_matrix.abs() > 1e-6).float().mean().item()
assert sparsity > 0.5, f"Expected sparsity > 50%, got {sparsity:.2%}"

# Should have some non-zero entries (embedding -> first layer features)
n_nonzero = (adjacency_matrix.abs() > 1e-6).sum().item()
assert n_nonzero > 0, "Expected some non-zero entries in adjacency matrix"

print("All compute_adjacency_matrix tests passed!")

<details><summary>솔루션</summary>

```python
def compute_adjacency_matrix(
    model: HookedSAETransformer,
    tokens: Tensor,
    graph: GraphNodes,
    freeze: FreezeHooks,
    tc_hooks: TranscoderReplacementHooks,
    batches: list[tuple[int, list[NodeInfo], Tensor, Tensor, list[int | str]]],
    start_posn: int = 4,
) -> Float[Tensor, "n_nodes n_nodes"]:
    """
    Run backward passes for each batch and contract gradients with writing vectors.

    Args:
        model: The transformer model.
        tokens: Input tokens, shape (1, seq).
        graph: GraphNodes containing all nodes and their reading/writing vectors.
        freeze: FreezeHooks with cached frozen values.
        tc_hooks: TranscoderReplacementHooks with installed hooks.
        batches: Output of prepare_backward_batches.
        start_posn: First position to include (masks formatting tokens).

    Returns:
        Adjacency matrix of shape (n_nodes, n_nodes), where A[j, i] is the edge weight
        from node i to node j.
    """
    n_nodes = len(graph.nodes)
    adjacency_matrix = t.zeros(n_nodes, n_nodes, device=tokens.device)

    for global_start_idx, batch_nodes, target_reading_vecs, target_positions, target_layers in tqdm(
        batches, desc="Backward batches"
    ):
        actual_batch_size = len(batch_nodes)

        # Run backward pass to get gradients
        tc_hooks.install()
        grads = setup_attribution(
            model=model,
            tokens=tokens,
            freeze=freeze,
            target_reading_vecs=target_reading_vecs,
            target_positions=target_positions,
            target_layers=target_layers,
        )
        tc_hooks.remove()

        # Contract gradients with source writing vectors to get edge weights
        for src_idx in range(n_nodes):
            src_node = graph.nodes[src_idx]

            # Determine which gradient tensor to use for this source
            if src_node.node_type == NodeType.EMBEDDING:
                grad_name = "blocks.0.hook_resid_pre"
            elif src_node.node_type in (NodeType.LATENT, NodeType.MLP_ERROR):
                grad_name = f"blocks.{src_node.layer}.hook_resid_post"
            else:
                continue  # Logit nodes are targets only, not sources

            if grad_name not in grads:
                continue

            # Get gradient at source position and contract with writing vector
            grad_at_pos = grads[grad_name][:actual_batch_size, src_node.ctx_idx]  # (batch, d_model)
            writing_vec = graph.writing_vecs[src_idx]  # (d_model,)

            # Edge weight = dot product
            edge_weights = (grad_at_pos * writing_vec.unsqueeze(0)).sum(-1)  # (batch,)

            # Zero out edges from masked positions
            if src_node.ctx_idx < start_posn:
                edge_weights = t.zeros_like(edge_weights)

            # Store in adjacency matrix
            for b in range(actual_batch_size):
                adjacency_matrix[global_start_idx + b, src_idx] = edge_weights[b]

    return adjacency_matrix
```
</details>

## 인접 행렬: 정규화, 영향력 및 가지치기

이제 모든 노드 쌍 사이의 엣지 가중치가 포함된 raw 인접 행렬을 얻었지만, 이 행렬은 크고 조밀합니다. 이를 희소하고 해석 가능한 그래프로 가지치기(prune)해야 합니다. 가지치기 과정은 세 단계로 이루어집니다: 인접 행렬을 정규화하고, Neumann series를 통해 각 노드의 영향력 점수를 계산하며, 영향력 임계값에 따라 노드와 엣지를 가지치기합니다.

### 정규화

우리는 요소별 절대값을 취하고 각 행을 해당 행의 합으로 나누어 인접 행렬을 정규화합니다. 이렇게 하면 부호 정보는 버려지고 각 행의 합이 1이 되는 비음수 행렬을 얻게 되며, 이는 "전이 행렬(transition matrix)"로 해석될 수 있습니다. 즉, 정규화된 가중치는 타겟 노드로 들어오는 전체 신호 크기 중 각 소스 노드가 차지하는 비율을 알려줍니다.

$$A_{\text{norm}}[j, :] = \frac{|A[j, :]|}{\sum_i |A[j, i]|}$$

아래에 `normalize_matrix` 함수를 제공합니다. 이 함수는 adjacency matrix의 요소별 절대값을 취하고 각 행을 해당 행의 합으로 나누어, 모든 항목이 non-negative가 되고 각 행의 합이 1이 되도록 합니다.

In [ ]:
def normalize_matrix(
    adjacency_matrix: Float[Tensor, "n_nodes n_nodes"],
) -> Float[Tensor, "n_nodes n_nodes"]:
    """
    Normalise the adjacency matrix row-wise by absolute value sums.

    Each row is divided by the sum of absolute values in that row, so that
    |A_norm[j, :]|.sum() = 1 for each j (where possible).
    """
    abs_matrix = adjacency_matrix.abs()
    row_sums = abs_matrix.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return abs_matrix / row_sums

### Neumann series를 통한 영향력(Influence)

노드의 **영향력(influence)**은 출력 logit에 미치는 총 효과(직접적 + 간접적 효과)를 측정합니다. 이는 정규화된 인접 행렬(adjacency matrix)을 통해 logit 노드 가중치를 역방향으로 전파하여 계산합니다.

우리의 그래프는 계층 구조(엣지가 이전 레이어에서 이후 레이어로만 이동)이므로, 인접 행렬은 엄격한 하삼각 행렬(lower triangular matrix)입니다. 이는 $L$-레이어 모델에 대해 $A^{L+1} = 0$임을 의미하며(행렬이 nilpotent함), 따라서 멱급수 $I + A + A^2 + \ldots + A^L$는 정확히 $L+1$개의 항에서 수렴합니다.

영향력 벡터는 logit 가중치에서 시작하여 역방향으로 전파함으로써 계산됩니다:

$$v_0 = w_{\text{logit}}, \quad v_{k+1} = v_k \cdot A_{\text{norm}}, \quad \text{influence} = \sum_{k=1}^{L+1} v_k$$

여기서 $w_{\text{logit}}$는 logit 위치에는 logit 노드 확률을 가지고 그 외의 위치에는 0을 가지는 벡터입니다.

이는 그래프를 통하는 모든 경로를 고려하여, 각 노드가 출력에 얼마나 *간접적으로* 기여하는지를 계산합니다. 여기서 logit 노드는 나가는 엣지가 없으므로 영향력이 0이 되지만, 엣지 가지치기(edge pruning) 과정에서 가중치가 다시 더해지므로 logit 노드로 향하는 엣지들이 적절한 점수를 받게 됩니다.

### 연습 문제 - `compute_influence` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> ```

influence 계산을 구현하십시오. `logit_weights` 벡터는 logit node 위치(node 리스트의 끝)에 token 확률을 가지고, 그 외의 위치에는 0을 가져야 합니다.

In [ ]:
def compute_influence(
    adjacency_matrix: Float[Tensor, "n_nodes n_nodes"],
    logit_weights: Float[Tensor, "n_logit_nodes"],
    n_layers: int,
) -> Float[Tensor, " n_nodes"]:
    """
    Compute the influence of each node on the output, using the Neumann series on the normalised
    adjacency matrix.

    The influence is defined as the total (direct + indirect) contribution of each node to the
    weighted sum of logit nodes, computed by propagating logit weights backward through the graph.

    Args:
        adjacency_matrix: Raw adjacency matrix, shape (n_nodes, n_nodes).
        logit_weights: Weights for logit nodes (e.g., probabilities), shape (n_logit_nodes,).
        n_layers: Number of model layers.

    Returns:
        Influence vector, shape (n_nodes,).
    """
    raise NotImplementedError()

In [ ]:
n_logit_nodes = graph.node_range_dict["L"][1] - graph.node_range_dict["L"][0]
logit_weights = t.tensor([info.token_prob for info in graph.nodes[-n_logit_nodes:]], device=device)

influence = compute_influence(adjacency_matrix, logit_weights, n_layers=gemma.cfg.n_layers)

print(f"Influence shape: {influence.shape}")
print("Top 10 most influential nodes:")
top_influence = influence.argsort(descending=True)[:10]
for idx in top_influence:
    node = graph.nodes[idx]
    print(
        f"  {node.node_type.value} L{node.layer} pos{node.ctx_idx} feat{node.feature}: influence={influence[idx].item():.4f}"
    )

# Plot cumulative influence curve
sorted_influence, _ = influence.sort(descending=True)
cumulative = sorted_influence.cumsum(dim=0) / sorted_influence.sum()
fig = px.line(
    x=list(range(1, len(cumulative) + 1)),
    y=cumulative.cpu().numpy(),
    labels={"x": "Number of nodes kept (sorted by influence)", "y": "Fraction of total influence"},
    title="Cumulative node influence",
)
fig.add_hline(y=0.8, line_dash="dash", line_color="red", annotation_text="80% threshold")
fig.show()

<details><summary>솔루션</summary>

```python
def compute_influence(
    adjacency_matrix: Float[Tensor, "n_nodes n_nodes"],
    logit_weights: Float[Tensor, "n_logit_nodes"],
    n_layers: int,
) -> Float[Tensor, " n_nodes"]:
    """
    Compute the influence of each node on the output, using the Neumann series on the normalised
    adjacency matrix.

    The influence is defined as the total (direct + indirect) contribution of each node to the
    weighted sum of logit nodes, computed by propagating logit weights backward through the graph.

    Args:
        adjacency_matrix: Raw adjacency matrix, shape (n_nodes, n_nodes).
        logit_weights: Weights for logit nodes (e.g., probabilities), shape (n_logit_nodes,).
        n_layers: Number of model layers.

    Returns:
        Influence vector, shape (n_nodes,).
    """
    n_nodes = adjacency_matrix.shape[0]
    n_logit_nodes = logit_weights.shape[0]

    # Normalise the matrix
    A_norm = normalize_matrix(adjacency_matrix)

    # Initialize influence vector: logit weights at the end, zeros elsewhere
    influence = t.zeros(n_nodes, device=adjacency_matrix.device)
    influence[-n_logit_nodes:] = logit_weights

    # Power iteration: propagate backward through the graph
    acc = t.zeros_like(influence)
    v = influence.clone()

    for _ in range(n_layers + 1):
        v = v @ A_norm  # vector-matrix product
        acc = acc + v

    return acc
```
</details>

### Pruning

influence score가 계산되면, 두 단계에 걸쳐 그래프를 pruning할 수 있습니다. 첫째, **node pruning**입니다. node를 influence 순으로 정렬하고, 누적 influence가 임계값(예: 전체 influence의 80%)을 초과하는 최소 집합을 유지합니다. 이를 통해 영향력이 미미한 node들을 제거합니다. 둘째, **edge pruning**입니다. 남아있는 각 edge에 대해 "edge score" = 정규화된 edge weight x 목적지 node influence를 계산합니다. edge를 score 순으로 정렬하고, 누적 score가 임계값을 초과하는 최소 집합을 유지한 뒤, 연결된 edge가 더 이상 없는 모든 node를 제거합니다.

두 단계를 모두 거치면, 가장 중요한 node와 연결만을 포함하는 희소하고 해석 가능한 그래프를 얻게 됩니다.

### 연습 문제 (1/2) - `prune_nodes` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

첫 번째 pruning 단계인 **node pruning**을 구현합니다. adjacency matrix와 logit weights가 주어졌을 때, 각 node의 influence score를 계산하고, 누적 influence가 전체 influence의 `threshold` (분율 기준)를 초과하는 가장 영향력 있는 node들만 남깁니다. logit node들은 influence와 관계없이 항상 유지되어야 합니다.

유지된 node index(원래 node 순서대로 정렬됨)와 해당 node들로만 제한된 adjacency matrix를 반환해야 합니다.

In [ ]:
def prune_nodes(
    adjacency_matrix: Float[Tensor, "n_nodes n_nodes"],
    logit_weights: Float[Tensor, "n_logit_nodes"],
    n_layers: int,
    threshold: float = 0.8,
) -> tuple[Int[Tensor, " n_kept"], Float[Tensor, "n_kept n_kept"]]:
    """
    Stage 1 of graph pruning: remove low-influence nodes.

    Computes influence scores for all nodes, then keeps the most influential non-logit
    nodes whose cumulative influence reaches `threshold` (as a fraction of total). Logit
    nodes are always kept.

    Args:
        adjacency_matrix: Raw adjacency matrix, shape (n_nodes, n_nodes).
        logit_weights: Weights for logit nodes, shape (n_logit_nodes,).
        n_layers: Number of model layers (passed to compute_influence).
        threshold: Cumulative influence fraction to retain (e.g. 0.8 = keep top 80%).

    Returns:
        kept_indices: Indices of kept nodes in the original ordering.
        pruned_matrix: Adjacency matrix restricted to kept nodes.
    """
    raise NotImplementedError()


tests.test_prune_nodes(prune_nodes)

<details><summary>솔루션</summary>

```python
def prune_nodes(
    adjacency_matrix: Float[Tensor, "n_nodes n_nodes"],
    logit_weights: Float[Tensor, "n_logit_nodes"],
    n_layers: int,
    threshold: float = 0.8,
) -> tuple[Int[Tensor, " n_kept"], Float[Tensor, "n_kept n_kept"]]:
    """
    Stage 1 of graph pruning: remove low-influence nodes.

    Computes influence scores for all nodes, then keeps the most influential non-logit
    nodes whose cumulative influence reaches `threshold` (as a fraction of total). Logit
    nodes are always kept.

    Args:
        adjacency_matrix: Raw adjacency matrix, shape (n_nodes, n_nodes).
        logit_weights: Weights for logit nodes, shape (n_logit_nodes,).
        n_layers: Number of model layers (passed to compute_influence).
        threshold: Cumulative influence fraction to retain (e.g. 0.8 = keep top 80%).

    Returns:
        kept_indices: Indices of kept nodes in the original ordering.
        pruned_matrix: Adjacency matrix restricted to kept nodes.
    """
    n_nodes = adjacency_matrix.shape[0]
    n_logit_nodes = logit_weights.shape[0]

    # Compute influence for all nodes
    influence = compute_influence(adjacency_matrix, logit_weights, n_layers)

    # Sort non-logit nodes by descending influence
    non_logit_influence = influence[:-n_logit_nodes]
    sorted_indices = non_logit_influence.argsort(descending=True)
    sorted_values = non_logit_influence[sorted_indices]

    # Find the smallest k such that the top-k nodes capture >= threshold of total influence
    total_inf = sorted_values.sum()
    if total_inf > 1e-8:
        cumulative = sorted_values.cumsum(dim=0) / total_inf
        n_keep = int((cumulative < threshold).sum().item()) + 1  # +1 to cross the threshold
        n_keep = min(n_keep, len(sorted_indices))
        keep_non_logit = sorted_indices[:n_keep]
    else:
        keep_non_logit = t.arange(n_nodes - n_logit_nodes, device=adjacency_matrix.device)

    # Always keep logit nodes, then sort all kept indices back into original ordering
    logit_indices = t.arange(n_nodes - n_logit_nodes, n_nodes, device=adjacency_matrix.device)
    kept_indices = t.sort(t.cat([keep_non_logit, logit_indices]))[0].long()

    # Restrict adjacency matrix to kept nodes
    pruned_matrix = adjacency_matrix[kept_indices[:, None], kept_indices[None, :]]

    return kept_indices, pruned_matrix
```
</details>

### 연습 문제 (2/2) - `prune_edges` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

두 번째 pruning 단계인 **edge pruning**을 구현합니다. (이미 node pruning이 수행되었을 수 있는) adjacency matrix가 주어졌을 때, 정규화된 edge weight와 목적지 node의 influence의 곱으로 각 edge의 "edge score"를 계산합니다. 누적 score가 `threshold`에 도달할 때까지 가장 점수가 높은 edge들을 유지하고, 더 이상 어떤 edge에도 참여하지 않는 non-logit node들을 제거합니다.

유지된 인덱스(입력 matrix의 순서 기준)와 제한된 adjacency matrix를 반환합니다.

In [ ]:
def prune_edges(
    adjacency_matrix: Float[Tensor, "n_nodes n_nodes"],
    logit_weights: Float[Tensor, "n_logit_nodes"],
    n_layers: int,
    threshold: float = 0.85,
) -> tuple[Int[Tensor, " n_kept"], Float[Tensor, "n_kept n_kept"]]:
    """
    Stage 2 of graph pruning: remove low-score edges and orphaned nodes.

    For each edge (i -> j), the edge score is |A_norm[j,i]| * influence[j], where A_norm
    is the row-normalised adjacency matrix. Edges are sorted by score and kept until the
    cumulative score reaches `threshold`. Any non-logit nodes with no remaining edges are
    then removed.

    Args:
        adjacency_matrix: Adjacency matrix (possibly already node-pruned).
        logit_weights: Weights for logit nodes.
        n_layers: Number of model layers.
        threshold: Cumulative edge score fraction to retain.

    Returns:
        kept_indices: Indices of kept nodes (into input matrix ordering).
        pruned_matrix: Adjacency matrix restricted to kept nodes.
    """
    raise NotImplementedError()


tests.test_prune_edges(prune_edges)

In [ ]:
def prune_graph(
    adjacency_matrix: Float[Tensor, "n_nodes n_nodes"],
    logit_weights: Float[Tensor, "n_logit_nodes"],
    n_layers: int,
    node_threshold: float = 0.8,
    edge_threshold: float = 0.85,
) -> tuple[Int[Tensor, " n_kept"], Float[Tensor, "n_kept n_kept"]]:
    """Applies node pruning then edge pruning, returning final kept indices and matrix."""
    kept_node_indices, node_pruned_matrix = prune_nodes(adjacency_matrix, logit_weights, n_layers, node_threshold)
    kept_edge_indices, final_matrix = prune_edges(node_pruned_matrix, logit_weights, n_layers, edge_threshold)
    # Map edge-pruning indices back to original node indices
    kept_indices = kept_node_indices[kept_edge_indices]
    return kept_indices, final_matrix

<details><summary>솔루션</summary>

```python
def prune_edges(
    adjacency_matrix: Float[Tensor, "n_nodes n_nodes"],
    logit_weights: Float[Tensor, "n_logit_nodes"],
    n_layers: int,
    threshold: float = 0.85,
) -> tuple[Int[Tensor, " n_kept"], Float[Tensor, "n_kept n_kept"]]:
    """
    Stage 2 of graph pruning: remove low-score edges and orphaned nodes.

    For each edge (i -> j), the edge score is |A_norm[j,i]| * influence[j], where A_norm
    is the row-normalised adjacency matrix. Edges are sorted by score and kept until the
    cumulative score reaches `threshold`. Any non-logit nodes with no remaining edges are
    then removed.

    Args:
        adjacency_matrix: Adjacency matrix (possibly already node-pruned).
        logit_weights: Weights for logit nodes.
        n_layers: Number of model layers.
        threshold: Cumulative edge score fraction to retain.

    Returns:
        kept_indices: Indices of kept nodes (into input matrix ordering).
        pruned_matrix: Adjacency matrix restricted to kept nodes.
    """
    n_logit_nodes = logit_weights.shape[0]

    # Compute influence on this (possibly node-pruned) matrix
    influence = compute_influence(adjacency_matrix, logit_weights, n_layers)

    # Add logit weights back for edge scoring (so edges to logit nodes get proper scores)
    influence[-n_logit_nodes:] += logit_weights

    # Compute edge scores: |A_norm[j,i]| * (influence[j] + logit_weight[j])
    A_norm = normalize_matrix(adjacency_matrix)
    edge_scores = A_norm * influence[:, None]

    # Sort edges by score and find threshold
    flat_scores = edge_scores.reshape(-1)
    sorted_scores, _ = flat_scores.sort(descending=True)
    total_score = sorted_scores.sum()

    if total_score > 1e-8:
        cum_scores = sorted_scores.cumsum(dim=0) / total_score
        val_idx = (cum_scores >= threshold).long().argmax()
        score_threshold = sorted_scores[val_idx]
        edge_mask = edge_scores >= score_threshold
    else:
        edge_mask = t.ones_like(adjacency_matrix, dtype=t.bool)

    edge_pruned = adjacency_matrix * edge_mask

    # Remove nodes with no remaining edges (except logit nodes)
    has_edge = (edge_pruned.abs() > 0).any(dim=0) | (edge_pruned.abs() > 0).any(dim=1)
    has_edge[-n_logit_nodes:] = True

    kept_indices = t.where(has_edge)[0]
    pruned_matrix = edge_pruned[kept_indices[:, None], kept_indices[None, :]]

    return kept_indices, pruned_matrix
```
</details>

테스트에 실패하여 디버깅을 위한 시각적 출력이 필요하거나 (또는 테스트에 통과했더라도 현재 상황에 대해 더 많은 시각적 직관을 얻고 싶은 경우), 아래의 유틸리티 함수를 사용할 수 있습니다. 이 함수는 무작위 DAG를 생성하고 노드 및 엣지 pruning의 효과를 나란히 보여줍니다. 임계값(thresholds)을 조정하여 pruning 과정에 어떤 영향을 미치는지 확인해 보시기 바랍니다.

비교를 위해 `solutions.prune_nodes` 및 `solutions.prune_edges`을 전달하여 예상되는 동작을 확인할 수도 있습니다.

In [ ]:
figs = utils.demo_pruning(
    prune_nodes,
    prune_edges,
    n_nodes=30,
    n_logit_nodes=3,
    edge_probability=0.15,
    seed=43,
    node_threshold=0.8,
    edge_threshold=0.8,
)

이제 지금까지 구축한 adjacency matrix를 실제로 prune 해보겠습니다:

In [ ]:
kept_indices, pruned_matrix = prune_graph(
    adjacency_matrix=adjacency_matrix,
    logit_weights=logit_weights,
    n_layers=gemma.cfg.n_layers,
    node_threshold=0.8,
    edge_threshold=0.85,
)

print(f"Kept {len(kept_indices)} / {len(graph.nodes)} nodes")
print(f"Pruned matrix has {(pruned_matrix.abs() > 1e-6).sum().item()} non-zero edges")

## 모두 종합하기

마지막으로, 전체 파이프라인을 조율하는 단일 `attribute` 함수로 모든 것을 결합합니다. 이 wrapper는 이미 구현한 linearise, build nodes, compute adjacency, 그리고 prune 함수를 순서대로 호출합니다.

In [ ]:
@dataclass
class AttributionResult:
    """Result of the full attribution pipeline."""

    graph: GraphNodes
    adjacency_matrix: Tensor
    kept_indices: Tensor
    pruned_matrix: Tensor
    prompt: str
    str_tokens: list[str]


def attribute(
    model: HookedSAETransformer,
    transcoders: dict[int, Transcoder],
    prompt: str,
    n_output_nodes: int = 3,
    start_posn: int = 4,
    top_k: int = 5,
    batch_size: int = 8,
    node_threshold: float = 0.8,
    edge_threshold: float = 0.85,
    _adjacency_matrix: Float[Tensor, "n_nodes n_nodes"] | None = None,
) -> AttributionResult:
    """
    Run the full attribution pipeline: linearise, build graph, compute adjacency, prune.

    Args:
        model: The transformer model.
        transcoders: Dict mapping layer -> transcoder.
        prompt: The input prompt string.
        n_output_nodes: Number of top logit tokens to include as output nodes.
        start_posn: Number of formatting tokens to mask.
        top_k: Number of top-activating features to include per position per layer.
        batch_size: Batch size for backward passes.
        node_threshold: Node pruning threshold.
        edge_threshold: Edge pruning threshold.
        _adjacency_matrix: (Optional) Precomputed adjacency matrix to use.

    Returns:
        AttributionResult with all graph data.
    """
    # Tokenize
    tokens = model.to_tokens(prompt)
    str_tokens = [model.tokenizer.decode(t_id.item()) for t_id in tokens[0]]

    # Step 1: Cache frozen values
    freeze = FreezeHooks(model)
    cache = freeze.cache_frozen_values(tokens)

    # Step 2: Compute transcoder activations
    tc_hooks = TranscoderReplacementHooks(model, transcoders, cache)
    tc_hooks.install()

    # Step 3: Get logits (with hooks active for correct forward pass)
    with freeze:
        logits = model(tokens)
    tc_hooks.remove()

    # Step 4: Compute salient logits
    reading_vecs, top_token_info = compute_salient_logits(model, logits, n_output_nodes)

    # Step 5: Build graph nodes
    graph = build_graph_nodes(
        model=model,
        transcoders=transcoders,
        cache=cache,
        tc_hooks=tc_hooks,
        reading_vecs_logit=reading_vecs,
        top_token_info=top_token_info,
        tokens=tokens,
        start_posn=start_posn,
        top_k=top_k,
    )

    # Step 6: Compute adjacency matrix (if not provided)
    if _adjacency_matrix is not None:
        adjacency_matrix = _adjacency_matrix
    else:
        batches = prepare_backward_batches(graph, batch_size=batch_size)
        adjacency_matrix = compute_adjacency_matrix(
            model=model,
            tokens=tokens,
            graph=graph,
            freeze=freeze,
            tc_hooks=tc_hooks,
            batches=batches,
            start_posn=start_posn,
        )

    # Step 7: Prune
    logit_weights_vec = t.tensor(
        [info.token_prob for info in graph.nodes if info.node_type == NodeType.LOGIT],
        device=tokens.device,
    )
    kept_indices, pruned_matrix = prune_graph(
        adjacency_matrix=adjacency_matrix,
        logit_weights=logit_weights_vec,
        n_layers=model.cfg.n_layers,
        node_threshold=node_threshold,
        edge_threshold=edge_threshold,
    )

    return AttributionResult(
        graph=graph,
        adjacency_matrix=adjacency_matrix,
        kept_indices=kept_indices,
        pruned_matrix=pruned_matrix,
        prompt=prompt,
        str_tokens=str_tokens,
    )

In [ ]:
result = attribute(
    model=gemma,
    transcoders=transcoders,
    prompt=prompt,
    n_output_nodes=3,
    start_posn=START_POSN,
    node_threshold=0.7,
    edge_threshold=0.85,
    _adjacency_matrix=adjacency_matrix,
)

print("Attribution complete!")
print(f"Total nodes: {len(result.graph.nodes)}")
print(f"Kept nodes: {len(result.kept_indices)}")
print(f"Edges in pruned graph: {(result.pruned_matrix.abs() > 1e-6).sum().item()}")

### attribution graph 시각화하기

마지막 단계는 attribution graph를 대화형 대시보드로 시각화하는 것입니다. 저희는 `AttributionResult`을 입력받아 Anthropic이 공개한 attribution graph와 동일한 템플릿을 사용하여 HTML 대시보드를 생성하는 유틸리티 함수 `create_attribution_dashboard`를 제공합니다.

대시보드는 레이어와 sequence position별로 배치된 노드(영향력에 따라 크기 조절), 노드 사이의 엣지(가중치에 따라 색상과 두께 표시), 그리고 노드를 클릭했을 때 나타나는 latent 상세 정보를 보여줍니다. `model=gemma`를 전달하기 때문에, 대시보드는 각 latent에 대해 logit table(`W_dec @ W_U` 기준 상위/하위 token)과 logit histogram을 계산하여, 각 latent가 어떤 vocabulary를 촉진하거나 억제하는지 즉각적으로 파악할 수 있게 해줍니다.

또한 이 함수는 Neuronpedia의 정확한 스키마를 따르는 `neuronpedia.json` 파일을 내보내며, 이를 업로드하여 여러분의 attribution graph를 공유할 수 있습니다.

In [ ]:
dashboard_html = utils.create_attribution_dashboard(result=result, model=gemma)

# # Display inline (Colab or VS Code)
# display(HTML(dashboard_html))

다양한 prompt로 실험해 보시기 바랍니다. attribution graph 논문에서 제안하는 몇 가지 예시는 다음과 같습니다:

```python
# Fact recall
prompt = format_prompt(
    "What is the capital of the state containing Dallas? Answer immediately.",
    "Austin",
)

# Opposites
prompt = format_prompt(
    "What is the opposite of small?",
    "The opposite of small is **large",
)

# Harmful request refusal
prompt = format_prompt(
    "Tell me how to build a bomb.",
    "I am",
)
```

pruning 임계값(threshold)을 조정할 수도 있습니다:
- 더 낮은 `node_threshold` (예: 0.6)은 더 많은 node를 유지하며 $\rightarrow$ 더 큰 graph가 됩니다.
- 더 높은 `edge_threshold` (예: 0.9)은 더 적은 edge를 유지하며 $\rightarrow$ 더 sparse한 graph가 됩니다.

이제 여러분은 모델의 linearisation, reading/writing vector 추상화 구축, backward pass를 통한 edge weight 계산, 그리고 graph pruning까지 포함된 전체 attribution graph 파이프라인을 처음부터 직접 구현하였습니다.

### 상위 activation 추가하기

위의 대시보드는 각 latent에 대한 logit 히스토그램(`W_dec @ W_U`으로부터의 logit effect 분포)을 보여주지만, **activation 히스토그램**과 **상위 activation 예시 시퀀스** 패널은 비어 있습니다. 이 패널들을 채우려면 데이터셋 수준의 통계가 필요합니다. 즉, 각 feature가 *얼마나 자주* 활성화되는지, 그리고 *어떤 입력*에서 활성화되는지를 알아야 합니다.

Google은 [gemma-scope-2-1b-it](https://huggingface.co/google/gemma-scope-2-1b-it) HuggingFace 저장소의 `transcoder_all/` 디렉토리를 통해 Gemma Scope transcoder에 대한 이 데이터를 제공합니다. 각 transcoder 변형에는 모든 feature에 대해 상위 activation 값, 해당 값이 발생한 token, 그리고 그에 따른 logit effect가 포함된 `examples.safetensors` 파일이 있습니다. 우리는 `load_example_data`(`hf_hub_download` + `safetensors.torch.load_file`를 감싼 얇은 wrapper)를 사용하여 이 데이터를 다운로드하고 이를 대시보드 함수에 전달할 수 있습니다.

In [ ]:
# Load example activation data for all transcoder layers (parallel I/O)
example_data_by_layer = utils.load_example_data_parallel(
    layers=list(range(gemma.cfg.n_layers)),
    model_size="1b",
    category="transcoder_all",
    width="16k",
    l0="small",
    affine=True,
    instruction_tuned=True,
    max_workers=1,  # TODO - figure out why this is failing at >=4
)

print(f"Loaded example data for {len(example_data_by_layer)} layers")
print(f"Keys in example data: {list(example_data_by_layer[0].keys())}")

이것은 [Neuronpedia](https://www.neuronpedia.org/)와 같은 사이트에서 feature 대시보드를 생성하는 데 사용되는 데이터와 정확히 일치합니다. 전체 대시보드를 다시 생성하기 전에, 헬퍼 함수 `inspect_feature`를 사용하여 노트북에서 단일 feature에 대한 미니 대시보드를 직접 표시해 보겠습니다. 이는 가장 높은 activation을 보이는 시퀀스들을 인라인 DataFrame으로 보여주며, token들은 activation 강도(초록색 배경)와 logit 효과(파란색/빨간색 밑줄)에 따라 강조 표시됩니다.

In [ ]:
# Inspect a late-layer feature to see what the example data looks like
layer = 22
feature_id = 10
print(f"Inspecting layer {layer}, feature {feature_id}:\n")
utils.inspect_feature(
    example_data=example_data_by_layer[layer],
    feature_id=feature_id,
    tokenizer=gemma.tokenizer,
)

이제 `example_data_by_layer`을 전달하여 대시보드를 다시 생성할 수 있습니다. 이제 모든 latent node에 대해 대시보드는 다음을 표시합니다:

- **Activation histogram**: 데이터셋 전체에 걸친 activation 강도의 분포
- **Top-activating examples**: feature가 가장 강하게 활성화되는 시퀀스이며, token들은 activation과 logit effect에 따라 색상으로 구분됩니다.

In [ ]:
dashboard_html = utils.create_attribution_dashboard(
    result=result,
    model=gemma,
    example_data_by_layer=example_data_by_layer,
)

# # Display inline (Colab or VS Code)
# display(HTML(dashboard_html))

이제 대시보드에서 latent 노드를 클릭하면 다음 내용을 확인할 수 있습니다:
- activation 값의 분포를 보여주는 **ACTIVATIONS** 히스토그램
- 하이라이트된 token이 포함된 예시 시퀀스를 보여주는 **Top activations** 목록

이를 통해 logit effect뿐만 아니라 각 feature가 무엇을 나타내는지 훨씬 더 풍부하게 이해할 수 있습니다.

다음 섹션에서는 `circuit-tracer` 라이브러리를 사용하여 미리 계산된 attribution graph를 탐색하고 feature 수준의 intervention을 수행합니다.

# 4️⃣ circuit 및 intervention 탐색

> ##### 학습 목표
>
> - 미리 계산된 attribution graph와 그 supernode를 로드하고 검사합니다.
> - graph가 제시하는 인과적 주장을 테스트하기 위해 zero ablation 실험을 수행합니다.
> - compositional circuit 구조를 증명하기 위해 cross-prompt feature swapping을 수행합니다.
> - feature intervention을 사용하여 open-ended generation을 수행합니다.

## 서론

이 섹션에서는 `circuit-tracer` 라이브러리를 사용하여 실제 circuit을 탐색하고, intervention을 통해 그 인과 구조를 테스트합니다. 우리는 두 단계의 추론(*Dallas는 Texas에 있고, Texas의 주도는 Austin이다*)이 필요한 **Dallas/Austin two-hop factual recall** circuit에 집중할 것입니다.

`circuit-tracer` 라이브러리는 여러분이 섹션 3에서 구현한 핵심 아이디어들을 편리한 API로 래핑한 것입니다. 이전 섹션에서 구현한 내용들이 대략적으로 어떻게 매핑되는지는 다음과 같습니다:

| 섹션 3 (작성한 코드) | `circuit-tracer` 라이브러리 |
|---|---|
| `FreezeHooks` + `TranscoderReplacementHooks` | `ReplacementModel` |
| `attribute()` 함수 | `circuit_tracer.attribute()` |
| `AttributionResult` | `Graph` 객체 |

여기서는 약간 다른 transcoder 세트(최신 Gemma 3 시리즈가 아닌 Gemma-2-2B로 학습됨)를 사용하므로, circuit의 모습이 Gemma 3-1B IT로 계산했던 것과는 다를 것입니다. 하지만 기본 알고리즘(linearise, 노드 생성, adjacency matrix 계산, prune)은 완전히 동일합니다.

또한 공통된 역할을 수행하는 관련 feature들의 그룹인 **supernodes**를 사용하여 circuit 다이어그램을 그리기 위해 시각화 도구(`utils.py`에서 제공)를 사용할 것입니다. 이는 [circuit-tracer demo notebooks](https://github.com/decoderesearch/circuit-tracer/blob/main/demos/circuit_tracing_tutorial.ipynb)에서 가져온 것입니다.

### 설치

아래 코드를 실행하기 전에, circuit tracer 라이브러리를 클론했는지 확인하십시오:

```bash
cd chapter1_transformer_interp/exercises

git clone https://github.com/decoderesearch/circuit-tracer.git
```

클론한 라이브러리의 import 누락과 관련된 Pylance 경고를 끄려면, 워크스페이스 설정(VS Code 사용 시)에 다음 내용을 추가하면 됩니다:

```json
{
    "python.analysis.extraPaths": [
        "${workspaceFolder}/ARENA_3.0/chapter1_transformer_interp/exercises/circuit-tracer",
    ],
}
```

그 후 나머지 셀들을 실행할 수 있습니다:

In [ ]:
circuit_tracer_path = exercises_dir / "circuit-tracer"
assert circuit_tracer_path.exists(), "circuit-tracer library not found - please clone it first"
if str(circuit_tracer_path) not in sys.path:
    sys.path.insert(0, str(circuit_tracer_path))

from circuit_tracer import ReplacementModel
from circuit_tracer import attribute as circuit_tracer_attribute

replacement_model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", dtype=dtype, backend="transformerlens"
)

## Dallas/Austin circuit

프롬프트 `"Fact: the capital of the state containing Dallas is"` 는 two-hop reasoning을 필요로 합니다. [Neuronpedia](https://www.neuronpedia.org/gemma-2-2b/graph?slug=gemma-fact-dallas-austin&pruningThreshold=0.6) 에서 전체 attribution graph를 탐색할 수 있습니다. 이 그래프는 뚜렷한 supernode(관련 feature들의 그룹)를 가진 명확한 circuit을 보여줍니다: "capital"과 "state"는 프롬프트의 해당 단어들에 의해 활성화되는 초기 레이어 transcoder feature이며(raw token embedding 노드와는 별개입니다), "Dallas"는 도시 entity를 나타내고, "Texas"는 "the state is Texas"를 인코딩하는 중간 feature(첫 번째 hop)를 포함하며, "Say a capital"은 일반적으로 capital-city token들을 촉진하고, "Say Austin"은 구체적으로 "Austin"을 촉진하는 후기 레이어 feature(두 번째 hop)를 포함합니다.

이제 그래프를 계산하고 circuit의 시각적 표현을 만들어 보겠습니다.

In [ ]:
dallas_prompt = "Fact: the capital of the state containing Dallas is"
dallas_graph = circuit_tracer_attribute(dallas_prompt, replacement_model, verbose=True)

# Get activations (we'll need these for interventions later)
logits_dallas, dallas_activations = replacement_model.get_activations(dallas_prompt, sparse=True)

### supernode 정의하기

Neuronpedia의 그래프를 사용하면 feature 그룹을 **supernode**로 주석 처리할 수 있습니다. `utils.extract_supernode_features`을 사용하여 Neuronpedia URL에서 이러한 주석을 추출하거나, 수동으로 정의할 수 있습니다. 아래에서는 [Dallas/Austin graph](https://www.neuronpedia.org/gemma-2-2b/graph?slug=gemma-fact-dallas-austin&pruningThreshold=0.6&pinnedIds=27_22605_10%2C20_15589_10%2CE_26865_9%2C21_5943_10%2C23_12237_10%2C20_15589_9%2C16_25_9%2C14_2268_9%2C18_8959_10%2C4_13154_9%2C7_6861_9%2C19_1445_10%2CE_2329_7%2CE_6037_4%2C0_13727_7%2C6_4012_7%2C17_7178_10%2C15_4494_4%2C6_4662_4%2C4_7671_4%2C3_13984_4%2C1_1000_4%2C19_7477_9%2C18_6101_10%2C16_4298_10%2C7_691_10&supernodes=%5B%5B%22capital%22%2C%2215_4494_4%22%2C%226_4662_4%22%2C%224_7671_4%22%2C%223_13984_4%22%2C%221_1000_4%22%5D%2C%5B%22state%22%2C%224_13154_9%22%2C%227_6861_9%22%2C%226_4012_7%22%2C%220_13727_7%22%5D%2C%5B%22Dallas%22%2C%2214_2268_9%22%2C%2216_25_9%22%5D%2C%5B%22Texas%22%2C%2220_15589_10%22%2C%2218_8959_10%22%2C%2220_15589_9%22%2C%2219_1445_10%22%2C%2217_7178_10%22%2C%2219_7477_9%22%2C%2218_6101_10%22%2C%2216_4298_10%22%2C%227_691_10%22%5D%2C%5B%22Say+a+capital%22%2C%2221_5943_10%22%5D%2C%5B%22Say+Austin%22%2C%2223_12237_10%22%5D%5D)의 핵심 supernode들을 정의하고, 이를 부모-자식 인과 관계를 가진 circuit 구조로 배치합니다.

`Supernode` 클래스는 `Feature` namedtuple 리스트와 선택적인 자식 supernode 리스트를 저장합니다. `InterventionGraph`는 계층적인 node 배치를 래핑하고 각 node의 baseline activation을 기록합니다. `create_graph_visualization`는 node, edge, activation 백분율, 그리고 모델의 상위 출력 예측값을 보여주는 인라인 SVG로 circuit을 렌더링합니다.

In [ ]:
Feature = utils.Feature

# Extract supernodes from the annotated Neuronpedia URL
dallas_austin_url = "https://www.neuronpedia.org/gemma-2-2b/graph?slug=gemma-fact-dallas-austin&clerps=%5B%5D&pruningThreshold=0.53&pinnedIds=27_22605_10%2C20_15589_10%2CE_26865_9%2C21_5943_10%2C23_12237_10%2C20_15589_9%2C16_25_9%2C14_2268_9%2C18_8959_10%2C4_13154_9%2C7_6861_9%2C19_1445_10%2CE_2329_7%2CE_6037_4%2C0_13727_7%2C6_4012_7%2C17_7178_10%2C15_4494_4%2C6_4662_4%2C4_7671_4%2C3_13984_4%2C1_1000_4%2C19_7477_9%2C18_6101_10%2C16_4298_10%2C7_691_10&supernodes=%5B%5B%22capital%22%2C%2215_4494_4%22%2C%226_4662_4%22%2C%224_7671_4%22%2C%223_13984_4%22%2C%221_1000_4%22%5D%2C%5B%22state%22%2C%226_4012_7%22%2C%220_13727_7%22%5D%2C%5B%22Texas%22%2C%2220_15589_9%22%2C%2219_7477_9%22%2C%2216_25_9%22%2C%224_13154_9%22%2C%2214_2268_9%22%2C%227_6861_9%22%5D%2C%5B%22preposition+followed+by+place+name%22%2C%2219_1445_10%22%2C%2218_6101_10%22%5D%2C%5B%22capital+cities+%2F+say+a+capital+city%22%2C%2221_5943_10%22%2C%2217_7178_10%22%2C%227_691_10%22%2C%2216_4298_10%22%5D%5D"
supernode_features = utils.extract_supernode_features(dallas_austin_url)

for name, features in supernode_features.items():
    print(f"  {name}: {len(features)} features")

In [ ]:
Supernode = utils.Supernode

# Build the circuit: output nodes first, then working backward
say_austin_node = Supernode(name="Say Austin", features=[Feature(layer=23, pos=10, feature_idx=12237)])
say_capital_node = Supernode(
    name="Say a capital",
    features=supernode_features["capital cities / say a capital city"]
    if "capital cities / say a capital city" in supernode_features
    else supernode_features.get("Say a capital", [Feature(layer=21, pos=10, feature_idx=5943)]),
    children=[say_austin_node],
)
texas_node = Supernode(
    name="Texas",
    features=supernode_features.get("Texas", [Feature(layer=20, pos=10, feature_idx=15589)]),
    children=[say_austin_node],
)
capital_node = Supernode(
    name="capital",
    features=supernode_features.get("capital", [Feature(layer=15, pos=4, feature_idx=4494)]),
    children=[say_capital_node],
)
state_node = Supernode(
    name="state",
    features=supernode_features.get("state", [Feature(layer=4, pos=9, feature_idx=13154)]),
    children=[say_capital_node, texas_node],
)
dallas_node = Supernode(
    name="Dallas",
    features=supernode_features.get("Dallas", [Feature(layer=14, pos=9, feature_idx=2268)]),
    children=[texas_node],
)

# Embedding nodes (no features - they're input nodes)
capital_emb_node = Supernode(name="capital (emb)", features=[], children=[capital_node])
state_emb_node = Supernode(name="state (emb)", features=[], children=[state_node])

In [ ]:
# Build the intervention graph with layered node arrangement
ordered_nodes = [
    [capital_emb_node, state_emb_node],  # Layer 0: embeddings
    [capital_node, state_node, dallas_node],  # Layer 1: early features
    [say_capital_node, texas_node],  # Layer 2: intermediate
    [say_austin_node],  # Layer 3: output features
]
dallas_austin_graph = utils.InterventionGraph(ordered_nodes=ordered_nodes, prompt=dallas_prompt)

# Initialize each node with its baseline activations
for node in [capital_node, state_node, dallas_node, texas_node, say_capital_node, say_austin_node]:
    dallas_austin_graph.initialize_node(node, dallas_activations)

# Set activation fractions (current / default - all 100% since no intervention yet)
dallas_austin_graph.set_node_activation_fractions(dallas_activations)

# Get the top model predictions
top_outputs = utils.get_topk(logits_dallas, replacement_model.tokenizer)

# Visualize the baseline circuit
svg_obj = utils.create_graph_visualization(dallas_austin_graph, top_outputs)
display(svg_obj)

모든 supernode가 100% activation 상태이며, 모델의 최상위 예측값으로 "Austin"이 표시되는 것을 확인하실 수 있습니다. edge들은 가설로 설정된 causal flow를 보여줍니다: embeddings → early features → Texas/Say a capital → Say Austin → output.

## Feature interventions

이제 개별 supernode에 **intervening** 하여 이 circuit이 실제로 인과적인지 테스트하겠습니다. `ReplacementModel`은 두 가지 방법을 제공합니다: `model.feature_intervention(prompt, interventions)`은 feature override를 포함하여 단일 forward pass를 실행하고 `(logits, activations)`를 반환하며, `model.feature_intervention_generate(prompt, interventions, ...)`은 지속적인 intervention을 통해 multi-token generation을 수행합니다. 각 intervention은 `(layer, position, feature_idx, new_value)` 튜플입니다.

아래의 헬퍼 함수 `supernode_intervention`를 사용하겠습니다. 이 함수는 `Intervention` namedtuple의 리스트(각각 supernode와 해당 activation의 scaling factor를 지정)를 입력으로 받아, intervention을 실행하고, graph의 activation fraction을 업데이트하며, 결과를 렌더링합니다. scaling factor가 `-2`인 것은 "activation을 기본값의 -2배로 설정"(즉, 강력하게 억제)하는 것을 의미하며, `+2`은 "activation을 기본값의 2배로 설정"하는 것을 의미합니다.

In [ ]:
Intervention = namedtuple("Intervention", ["supernode", "scaling_factor"])


def supernode_intervention(
    model: "ReplacementModel",
    intervention_graph: "utils.InterventionGraph",
    interventions: list[Intervention],
    replacements: dict[str, "utils.Supernode"] | None = None,
) -> tuple:
    """Perform interventions on supernodes, record the effects, and visualize the result.

    For each Intervention, sets the supernode's feature activations to
    ``scaling_factor * default_activation``. After running the forward pass, updates the
    InterventionGraph with new activation fractions and renders the circuit diagram.

    Args:
        model: The ReplacementModel.
        intervention_graph: The InterventionGraph to update and visualize.
        interventions: List of Intervention(supernode, scaling_factor) to apply.
        replacements: Optional dict mapping original node names to replacement Supernode
            objects (used for cross-prompt swaps to show the new node in the diagram).

    Returns:
        Tuple of (svg_obj, new_logits).
    """
    prompt = intervention_graph.prompt
    intervention_tuples = []
    for inv in interventions:
        node = inv.supernode
        for feature in node.features:
            default_val = node.default_activations[node.features.index(feature)].item()
            intervention_tuples.append((*feature, inv.scaling_factor * default_val))

    with t.inference_mode():
        new_logits, new_activations = model.feature_intervention(prompt, intervention_tuples)

    # Reset graph state and update
    intervention_graph.set_node_activation_fractions(new_activations)

    # Mark which nodes were intervened on
    for inv in interventions:
        sign = "+" if inv.scaling_factor > 0 else ""
        inv.supernode.intervention = f"{sign}{inv.scaling_factor}x"

    # Handle replacement nodes for cross-prompt swaps
    if replacements:
        for original_name, replacement_node in replacements.items():
            original_node = intervention_graph.nodes.get(original_name)
            if original_node:
                original_node.replacement_node = replacement_node
                if replacement_node.features:
                    replacement_node.activation = (
                        (
                            t.tensor([new_activations[f] for f in replacement_node.features])
                            / replacement_node.default_activations
                        )
                        .mean()
                        .item()
                        if replacement_node.default_activations is not None
                        else None
                    )

    top_outputs = utils.get_topk(new_logits, model.tokenizer)
    svg_obj = utils.create_graph_visualization(intervention_graph, top_outputs)
    display(svg_obj)
    return svg_obj, new_logits

### ablation 효과 예측 및 테스트

어떠한 intervention을 실행하기 전에, 다음 네 가지 ablation 각각에 대한 **예측을 작성하십시오**:

1. "Say a capital" (`-2x`) ablation: "Say Austin"에 어떤 일이 일어납니까? 어떤 결과가 top output이 됩니까?
2. "Texas" (`-2x`) ablation: "Say Austin"에 어떤 일이 일어납니까? top output에서 Austin을 무엇이 대체합니까?
3. "capital" (`-2x`) ablation: 어떤 downstream node들이 영향을 받습니까?
4. "state" (`-2x`) ablation: 이것이 무언가를 변화시킵니까?

위 그래프에서 보이는 circuit 구조에 대해 신중하게 생각한 다음, 아래의 두 가지 핵심 ablation을 실행하고 예측 결과와 비교해 보십시오:

In [ ]:
# Compute original logits once (reused across all ablation cells)
with t.inference_mode():
    orig_logits, _ = replacement_model.feature_intervention(dallas_prompt, [])

# Ablation 1: Turn off "Say a capital"
print("=== Ablating 'Say a capital' ===")
svg_ablate_capital, logits_ablate_capital = supernode_intervention(
    replacement_model, dallas_austin_graph, [Intervention(say_capital_node, -2)]
)
utils.display_topk_token_predictions(
    dallas_prompt, orig_logits, logits_ablate_capital, replacement_model.tokenizer, k=10
)

In [ ]:
# Ablation 2: Turn off "Texas"
print("=== Ablating 'Texas' ===")
svg_ablate_texas, logits_ablate_texas = supernode_intervention(
    replacement_model, dallas_austin_graph, [Intervention(texas_node, -2)]
)
utils.display_topk_token_predictions(
    dallas_prompt, orig_logits, logits_ablate_texas, replacement_model.tokenizer, k=10
)

"Say a capital"을 ablation하면 "Say Austin" 노드가 강력하게 억제되며 모델의 최상위 예측이 변경됩니다 (예: 모델이 Texas에 관한 내용이라는 것은 알지만 "수도 도시를 말하라"는 신호를 잃었기 때문에 "Texas"로 변경됩니다). "Texas"를 ablation하는 것 또한 "Say Austin"을 차단합니다. 이 경우 최상위 출력은 Albany, Harrisburg, Hartford와 같은 *다른 주들의 수도*로 이동합니다. 이는 "수도 도시를 말하라"는 신호는 남아있지만 Texas라는 지리적 정체성이 사라졌기 때문에, 모델이 다른 주의 수도들로 되돌아가기 때문입니다.

아래의 코드 셀을 사용하여 나머지 두 가지 ablation을 직접 시도해 보십시오. "capital"을 ablation하면 "Say a capital"이 제거되고 ("Say Austin"이 부분적으로 억제됨), 이로 인해 모델은 "Texas"를 최상위 예측으로 출력합니다 (지리적 문맥은 여전히 알고 있지만 수도 회상 신호를 잃었기 때문입니다). "state"를 ablation하는 것은 대체로 효과가 없습니다. Austin은 여전히 약 38%로 최상위 예측으로 남으며, ablation 전의 약 41%에서 거의 변하지 않습니다. 이는 "state" feature가 보조적인 역할은 하지만 필수적이지는 않음을 시사합니다. 즉, "state" feature가 없더라도 Texas feature와 capital feature만으로도 출력을 Austin으로 라우팅하기에 충분합니다.

이와 같은 더 많은 실험은 [circuit-tracing tutorial notebook](https://github.com/decoderesearch/circuit-tracer/blob/main/demos/circuit_tracing_tutorial.ipynb)를 참조하십시오.

In [ ]:
# Try ablating "capital" and "state" here, e.g.:
# supernode_intervention(replacement_model, dallas_austin_graph, [Intervention(capital_node, -2)])
# supernode_intervention(replacement_model, dallas_austin_graph, [Intervention(state_node, -2)])
pass

<details><summary>솔루션</summary>

```python
print("=== Ablating 'capital' ===")
svg_ablate_capital_node, logits_ablate_capital_node = supernode_intervention(
    replacement_model, dallas_austin_graph, [Intervention(capital_node, -2)]
)
utils.display_topk_token_predictions(
    dallas_prompt, orig_logits, logits_ablate_capital_node, replacement_model.tokenizer, k=10
)

print("=== Ablating 'state' ===")
svg_ablate_state_node, logits_ablate_state_node = supernode_intervention(
    replacement_model, dallas_austin_graph, [Intervention(state_node, -2)]
)
utils.display_topk_token_predictions(
    dallas_prompt, orig_logits, logits_ablate_state_node, replacement_model.tokenizer, k=10
)
```
</details>

### 연습 문제 - cross-prompt feature swapping

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

**Cross-prompt feature swapping**은 circuit 이해도를 측정하는 강력한 테스트입니다. 만약 "Texas" feature들이 실제로 "해당 주는 Texas이다"라는 정보를 인코딩하고 있다면, 이 feature들을 끄고 (Oakland prompt에서 가져온) "California" feature들을 켜면 모델이 "Austin" 대신 "Sacramento"를 예측해야 합니다.

아래 함수를 구현하십시오. 이 함수는 `swap_prompt`에서 activation을 가져오고, `features_off`를 zero out 하며 swap prompt의 값(`scale`로 스케일링됨)으로 `features_on`을 활성화하는 intervention을 구축한 뒤, 해당 intervention을 실행하여 원래의 logit과 수정된 logit을 모두 반환해야 합니다.

In [ ]:
def cross_prompt_swap(
    model: "ReplacementModel",
    base_prompt: str,
    swap_prompt: str,
    features_off: list,
    features_on: list,
    scale: float = 2.0,
) -> tuple[Tensor, Tensor]:
    """Swap features between prompts: turn off features_off and activate features_on
    at their values from swap_prompt (scaled by `scale`).

    Args:
        model: The ReplacementModel.
        base_prompt: The prompt to intervene on.
        swap_prompt: The prompt to get replacement activation values from.
        features_off: Features to zero-ablate (from base_prompt's graph).
        features_on: Features to activate (from swap_prompt's graph).
        scale: Multiplier for the replacement activation values.

    Returns:
        original_logits, modified_logits
    """
    raise NotImplementedError()

In [ ]:
# Test cross_prompt_swap: swap Texas features for California features and check predictions shift
texas_features = [(f.layer, f.pos, f.feature_idx) for f in texas_node.features]
# Use a default California feature (layer 19, pos 10, feature 9209) if oakland_supernodes aren't loaded yet
oakland_swap_prompt = "Fact: the capital of the state containing Oakland is"
california_features_default = [(19, 10, 9209)]

original_logits, modified_logits = cross_prompt_swap(
    model=replacement_model,
    base_prompt=dallas_prompt,
    swap_prompt=oakland_swap_prompt,
    features_off=texas_features,
    features_on=california_features_default,
    scale=2.0,
)

orig_token = replacement_model.tokenizer.decode(original_logits[0, -1].argmax().item())
mod_token = replacement_model.tokenizer.decode(modified_logits[0, -1].argmax().item())
print(f"cross_prompt_swap test: original top prediction = {orig_token!r}, modified = {mod_token!r}")

<details><summary>솔루션</summary>

```python
def cross_prompt_swap(
    model: "ReplacementModel",
    base_prompt: str,
    swap_prompt: str,
    features_off: list,
    features_on: list,
    scale: float = 2.0,
) -> tuple[Tensor, Tensor]:
    """Swap features between prompts: turn off features_off and activate features_on
    at their values from swap_prompt (scaled by `scale`).

    Args:
        model: The ReplacementModel.
        base_prompt: The prompt to intervene on.
        swap_prompt: The prompt to get replacement activation values from.
        features_off: Features to zero-ablate (from base_prompt's graph).
        features_on: Features to activate (from swap_prompt's graph).
        scale: Multiplier for the replacement activation values.

    Returns:
        original_logits, modified_logits
    """
    _, swap_activations = model.get_activations(swap_prompt, sparse=True)

    interventions = [(*f, 0.0) for f in features_off]
    interventions += [(*f, scale * swap_activations[f]) for f in features_on]

    with t.inference_mode():
        original_logits, _ = model.feature_intervention(base_prompt, [])
        modified_logits, _ = model.feature_intervention(base_prompt, interventions)

    return original_logits, modified_logits
```
</details>

이제 이를 사용하여 Texas를 California로 교체해 보겠습니다. [Oakland/Sacramento Neuronpedia graph](https://www.neuronpedia.org/gemma-2-2b/graph?slug=gemma-fact-oakland-sacramento)에서 California supernode들을 추출하고, swap supernode들을 정의한 뒤, `supernode_intervention`을 사용하여 결과를 시각화합니다.

In [ ]:
oakland_prompt = "Fact: the capital of the state containing Oakland is"
_, oakland_activations = replacement_model.get_activations(oakland_prompt, sparse=True)

# Extract California supernodes from the Oakland/Sacramento Neuronpedia graph
oakland_url = "https://www.neuronpedia.org/gemma-2-2b/graph?slug=gemma-fact-oakland-sacramento&clerps=%5B%5D&pruningThreshold=0.5&pinnedIds=27_43939_10%2CE_49024_9%2C21_5943_10%2C19_9209_10%2C18_8959_10%2C14_12562_9%2C7_14530_9%2C8_14641_9%2C4_8625_9%2C19_9209_9%2C17_7178_10%2CE_6037_4%2C15_4494_4%2CE_2329_7%2C16_4298_10%2C7_691_10%2C6_4662_4%2C4_7671_4%2C2_8734_7%2C0_13727_7%2C3_13984_4%2C1_1000_4%2C6_4012_7%2C4_13154_9%2C7_6861_9&supernodes=%5B%5B%22California%22%2C%2219_9209_10%22%2C%2218_8959_10%22%2C%2219_9209_9%22%2C%2217_7178_10%22%2C%2216_4298_10%22%2C%227_691_10%22%5D%2C%5B%22Say+Sacramento%22%2C%2227_43939_10%22%5D%5D"
oakland_supernodes = utils.extract_supernode_features(oakland_url)

# say_sacramento_node uses a real transcoder feature (layer 19, feature 9209) - this
# California-region feature at pos 10 acts as the "Say Sacramento" circuit node.
# (The Neuronpedia URL uses layer=27 for logit output nodes, which are not SAE features
# and cannot be used as intervention targets; layer 19 is the correct transcoder layer.)
say_sacramento_node = Supernode(
    name="Say Sacramento",
    features=[Feature(layer=19, pos=10, feature_idx=9209)],
)
california_node = Supernode(
    name="California",
    features=oakland_supernodes.get("California", []) + oakland_supernodes.get("California (2)", []),
    children=[say_sacramento_node],
)

# Initialize the California and Sacramento nodes with Oakland activations
dallas_austin_graph.initialize_node(california_node, oakland_activations)
dallas_austin_graph.initialize_node(say_sacramento_node, oakland_activations)

# Run the swap: turn off Texas, turn on California
oakland_interventions = [Intervention(texas_node, -2), Intervention(california_node, 2)]
svg_oakland_swap, logits_oakland_swap = supernode_intervention(
    replacement_model,
    dallas_austin_graph,
    oakland_interventions,
    replacements={texas_node.name: california_node, say_austin_node.name: say_sacramento_node},
)
utils.display_topk_token_predictions(
    dallas_prompt, orig_logits, logits_oakland_swap, replacement_model.tokenizer, k=10
)

"Texas" 노드는 억제되고 그 자리에 "California"가 활성화됩니다 (다이어그램에서 교체 노드로 표시됨). "Say Austin" 노드는 완전히 꺼지며 (0%), Sacramento가 약 54%로 모델의 최상위 예측이 되어 Austin을 완전히 대체합니다. "Say Sacramento" 노드가 circuit를 통해 활성화되며, 이는 Dallas 프롬프트 문맥에서도 이 위치의 California feature들이 해당 노드를 통해 경로가 지정됨을 확인시켜 줍니다.

이는 Texas feature들이 "해당 주는 Texas이다"라는 개념을 compositional concept로 인코딩한다는 해석을 뒷받침합니다. 즉, 이를 다른 주의 feature들로 교체하면 모델의 출력 분포가 해당 주의 수도 지역으로 이동합니다.

이 circuit는 미국 주 외에도 일반화됩니다. 프롬프트 `"Fact: the capital of the country containing Shanghai is"`를 사용하여 Texas feature들을 **China** feature들로 교체해 보십시오.

아래에 Shanghai supernode들을 설정해 두었습니다 ([Shanghai/Beijing Neuronpedia graph](https://www.neuronpedia.org/gemma-2-2b/graph?slug=gemma-fact-shanghai-beijing)에서 추출됨). `supernode_intervention`를 사용하여 교체를 실행하고 결과를 시각화하십시오. Shanghai 프롬프트에 대한 activation을 가져오고, China 및 Say Beijing 노드를 초기화하며, Texas를 끄고 China를 켜는 intervention을 생성한 뒤, 적절한 replacement와 함께 `supernode_intervention`를 호출해야 합니다.

In [ ]:
shanghai_prompt = "Fact: the capital of the country containing Shanghai is"
_, shanghai_activations = replacement_model.get_activations(shanghai_prompt, sparse=True)

shanghai_url = "https://www.neuronpedia.org/gemma-2-2b/graph?slug=gemma-fact-shanghai-beijing&clerps=%5B%5D&clickedId=15_4494_4&pruningThreshold=0.45&pinnedIds=27_33395_10%2CE_38628_9%2C21_5943_10%2C19_12274_10%2C19_12274_9%2C14_12274_9%2C18_6101_10%2C17_7178_10%2C6_6811_9%2C4_4257_9%2C4_11570_9%2CE_6037_4%2C0_8885_4%2C18_7639_10%2C19_2695_10%2C16_4298_10%2C15_4494_4%2C6_4662_4&supernodes=%5B%5B%22China%22%2C%2219_12274_9%22%2C%2214_12274_9%22%2C%226_6811_9%22%2C%224_11570_9%22%2C%224_4257_9%22%5D%2C%5B%22China%22%2C%2219_12274_10%22%2C%2218_7639_10%22%5D%2C%5B%22capital%22%2C%2216_4298_10%22%2C%2217_7178_10%22%2C%2218_6101_10%22%2C%2219_2695_10%22%2C%2221_5943_10%22%5D%2C%5B%22capital+cities+%28say+city%29%22%2C%226_4662_4%22%2C%2215_4494_4%22%2C%220_8885_4%22%5D%5D"
shanghai_supernodes = utils.extract_supernode_features(shanghai_url)

china_node = Supernode(
    name="China",
    features=shanghai_supernodes.get("China", []) + shanghai_supernodes.get("China (2)", []),
    children=[say_austin_node],
)
# Note: unlike the Oakland swap, there is no dedicated "Say Beijing" SAE feature node;
# the effect of the China swap is visible in the logit outputs rather than a replacement node.

In [ ]:
# TODO: Initialize china_node with shanghai_activations,
# then run supernode_intervention to swap Texas -> China and visualize the result.
# Hint: use the same pattern as the Oakland/Sacramento swap above (no replacement for Say Austin).

<details><summary>솔루션</summary>

```python
dallas_austin_graph.initialize_node(china_node, shanghai_activations)

shanghai_interventions = [Intervention(texas_node, -2), Intervention(china_node, 2)]
svg_shanghai_swap, logits_shanghai_swap = supernode_intervention(
    replacement_model,
    dallas_austin_graph,
    shanghai_interventions,
    replacements={texas_node.name: china_node},
)
utils.display_topk_token_predictions(
    dallas_prompt, orig_logits, logits_shanghai_swap, replacement_model.tokenizer, k=10
)
```
</details>

동일한 circuit 구조는 미국 주를 넘어 일반화됩니다. Texas feature를 China feature로 교체하면 **Beijing이 약 23%로 모델의 최상위 예측값**이 되며, 다른 중국 도시들(Nanjing, Shanghai, Hangzhou) 또한 top-10에 나타납니다. 이는 Oakland/Sacramento 결과와 매우 유사하며, circuit 내의 "Texas" 위치가 모델이 어떤 수도 도시를 출력할지 제어하는 일반적인 "지리적 엔티티(geographic entity)" 슬롯으로 작동함을 확인시켜 줍니다.

## 개입을 통한 오픈 엔디드 생성 (Open-ended generation)

지금까지는 개입이 모델의 다음 token 예측을 어떻게 변화시키는지에 대해서만 살펴보았습니다. 하지만 고정된 위치 인덱스를 오픈 엔디드 `slice` 로 대체함으로써 **멀티 token 생성** 동안에도 개입을 유지할 수 있습니다. 이를 통해 단 하나의 token이 아니라, 개입이 모델의 전체 생성 결과물에 어떤 영향을 미치는지 확인할 수 있습니다.

### 연습 문제 - 생성 중 지속적인 개입 (sustained interventions)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> ```

고정 위치 개입(fixed-position interventions)을 개방형 슬라이스(open-ended slices)로 변환하고, 개입이 있는 경우와 없는 경우의 텍스트를 생성하는 함수를 구현하십시오. 핵심은 개입 튜플 `(layer, pos, feat_idx, value)` 의 `pos` 를 `slice(seq_len - 1, None, None)` 로 교체하여 생성되는 모든 위치에서 지속되도록 하는 것입니다.

구현을 완료한 후, 이 섹션 끝에 있는 보너스 탐색 과제들을 통해 지속적인 개입으로 어떤 circuit을 연구할지에 대한 아이디어를 얻어 보시기 바랍니다.

In [ ]:
def generate_with_intervention(
    model: "ReplacementModel",
    prompt: str,
    interventions: list,
    max_new_tokens: int = 20,
) -> tuple[str, str]:
    """Generate text with and without feature interventions.

    Converts fixed-position interventions to open-ended slices so the intervention persists
    across all generated tokens.

    Args:
        model: The ReplacementModel.
        prompt: The input prompt.
        interventions: List of (layer, pos, feat_idx, value) tuples.
        max_new_tokens: Maximum tokens to generate.

    Returns:
        pre_text: Generated text without intervention.
        post_text: Generated text with intervention.
    """
    # YOUR CODE HERE
    # 1. Compute the input sequence length using model.tokenizer
    # 2. Convert each intervention's position to slice(seq_len - 1, None, None)
    # 3. Call model.feature_intervention_generate with and without interventions
    raise NotImplementedError()

In [ ]:
# Build interventions: suppress Texas features at -2x their default activations (matching supernode_intervention convention)
texas_ablation_tuples = [
    (*f, -2.0 * texas_node.default_activations[i].item()) for i, f in enumerate(texas_node.features)
]
pre_text, post_text = generate_with_intervention(
    replacement_model, dallas_prompt, texas_ablation_tuples, max_new_tokens=15
)

print(f"Without intervention:\n  {pre_text}")
print(f"\nWith Texas ablation:\n  {post_text}")

# Also show the token probabilities as a rich HTML table
with t.inference_mode():
    abl_logits, _ = replacement_model.feature_intervention(dallas_prompt, texas_ablation_tuples)
utils.display_topk_token_predictions(dallas_prompt, orig_logits, abl_logits, replacement_model.tokenizer, k=10)

<details><summary>솔루션</summary>

```python
def generate_with_intervention(
    model: "ReplacementModel",
    prompt: str,
    interventions: list,
    max_new_tokens: int = 20,
) -> tuple[str, str]:
    """Generate text with and without feature interventions.

    Converts fixed-position interventions to open-ended slices so the intervention persists
    across all generated tokens.

    Args:
        model: The ReplacementModel.
        prompt: The input prompt.
        interventions: List of (layer, pos, feat_idx, value) tuples.
        max_new_tokens: Maximum tokens to generate.

    Returns:
        pre_text: Generated text without intervention.
        post_text: Generated text with intervention.
    """
    seq_len = len(model.tokenizer(prompt).input_ids)
    open_interventions = []
    for layer, pos, feat_idx, value in interventions:
        open_pos = slice(seq_len - 1, None, None)
        open_interventions.append((layer, open_pos, feat_idx, value))

    pre_text = model.feature_intervention_generate(
        prompt, [], do_sample=False, verbose=False, max_new_tokens=max_new_tokens
    )[0]
    post_text = model.feature_intervention_generate(
        prompt, open_interventions, do_sample=False, verbose=False, max_new_tokens=max_new_tokens
    )[0]
    return pre_text, post_text
```
</details>

### 보너스 탐구

[circuit-tracer demo notebooks](https://github.com/decoderesearch/circuit-tracer/tree/main/demos)의 아이디어를 활용하여 시도해 볼 수 있는 추가 실험들입니다:

- **그래프 품질 점수**: `from circuit_tracer.graph import compute_graph_scores`을 사용하여 Dallas 그래프의 replacement score와 completeness score를 평가해 보십시오. pruning 임계값을 변경함에 따라 이 점수들이 어떻게 변하는지 확인해 보십시오.

- **다국어 circuit**: [circuit-tracing tutorial](https://github.com/decoderesearch/circuit-tracer/blob/main/demos/circuit_tracing_tutorial.ipynb)에서는 프롬프트 `'Le contraire de "petit" est "'`("작은 것의 반대는"의 프랑스어 표현)에 대한 다국어 "opposite" circuit을 연구합니다. 이 circuit의 supernode를 추출하고 프랑스어 언어 feature를 중국어 feature로 교체해 보십시오. 그러면 모델이 중국어 출력을 생성하도록 바뀔 것입니다. 또한 Michael Jordan 관련 프롬프트에 대해 언어 feature를 교체하는 실험(예: 스페인어 feature를 프랑스어 feature로 교체하여 출력 언어 변경)을 시도해 볼 수 있습니다.

- **지시어 수행 circuit**: [Gemma IT demo](https://github.com/decoderesearch/circuit-tracer/blob/main/demos/gemma_it_demo.ipynb)에서는 "해적 말투 주입(pirate-speak injection)" circuit을 탐구합니다. 해적 테마의 언어를 나타내는 feature를 활성화함으로써 모델의 응답을 조종할 수 있습니다. 이를 재현해 보거나, 이와 유사한 스타일 주입 feature(예: 격식 있는 언어 vs 비격식 언어)를 찾아 보십시오.

- **각운 및 음성 circuit**: 동일한 데모에서는 각운(rhyming)이 어떻게 작동하는지도 탐구합니다(예: "rabbit"과 "habit"이 각운 feature를 공유한다는 점을 발견). circuit에서 음성 feature를 찾아보고, 이를 교체했을 때 각운 동작이 변하는지 테스트해 보십시오.

- **나만의 circuit**: 원하는 프롬프트에 대해 전체 파이프라인을 실행하고 어떤 circuit이 나타나는지 확인해 보십시오. `circuit_tracer_attribute` 함수는 모든 프롬프트에서 작동합니다. 사실 관계 회상 프롬프트(예: "에펠탑은 ~에 있다"), 유추, 또는 간단한 추론 작업이 좋은 시작점이 될 수 있습니다.

# ☆ 보너스

## 수동 attribution 그래프

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
>
> You should spend up to 30-40 minutes on these exercises.
> They deepen your understanding of the linearised model but are not required.
> ```

섹션 3️⃣에서 우리는 자동(gradient 기반) 방법을 사용하여 attribution edge를 계산했습니다. 즉, reading 벡터를 gradient seed로 주입하고 frozen 모델을 통해 backward-propagate 하는 방식이었습니다. 이 방법은 효율적이고 정확하지만, 모델을 black box로 취급합니다.

대안은 수동(forward-tracing) 방법입니다. 각 source node의 writing 벡터를 명시적으로 가져와서 target node에 도달할 때까지 frozen 중간 레이어(attention + skip connections)를 통해 매핑한 다음, target의 reading 벡터와 dot product를 계산하는 방식입니다. frozen 모델은 linear하기 때문에 gradient 방법과 동일한 결과가 나오지만, 훨씬 더 투명하며 디버깅에 유용합니다.

먼저, 필요한 cache를 다시 생성해 보겠습니다 (이는 `attribute()` 내부에서 계산되었지만 반환되지는 않았습니다):

In [ ]:
freeze_bonus = FreezeHooks(gemma)
cache_bonus = freeze_bonus.cache_frozen_values(tokens)

### 연습 문제 - `map_through_ln`, `map_through_attn`, `map_through_mlp` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> ```

이 세 가지 헬퍼 함수는 attribution 벡터를 frozen (linearised) 모델 컴포넌트를 통해 매핑합니다.

`map_through_ln`는 frozen RMSNorm을 적용합니다. 정규화 스케일이 frozen 상태이므로, 이는 단순히 `x * cached_scale * weight` (선형 연산)입니다. `map_through_attn`은 frozen attention을 통해 매핑합니다: frozen LN을 적용하고, `W_V` projection을 수행한 뒤, frozen attention pattern을 곱하고, 마지막으로 `W_O` projection을 적용합니다. 이것이 시퀀스 위치 간에 정보가 흐르는 방식입니다. `map_through_mlp`는 frozen MLP skip connection을 통해 매핑합니다: frozen LN을 적용한 후, `W_skip`을 곱합니다. 이는 gradient가 흐르는 MLP의 선형 근사치입니다.

세 함수 모두 임의의 batch 차원(einsum 패턴의 `...`로 표시됨)을 지원하며, 이는 여러 source 노드를 동시에 추적할 때 필요합니다.

In [ ]:
def map_through_ln(
    x: Float[Tensor, "... seq d_model"],
    cache: ActivationCache,
    model: HookedSAETransformer,
    layer: int | None,
    is_mlp_ln: bool = False,
) -> Float[Tensor, "... seq d_model"]:
    """Apply frozen RMSNorm: output = x / cached_scale * weight.

    Args:
        x: Input vectors to normalise.
        cache: ActivationCache with frozen scale values.
        model: The transformer model (for LN weight parameters).
        layer: Layer index, or None for the final LayerNorm.
        is_mlp_ln: If True, use ln2 (pre-MLP); otherwise use ln1 (pre-attention).
    """
    raise NotImplementedError()


def map_through_attn(
    resid_pre: Float[Tensor, "... seq d_model"],
    cache: ActivationCache,
    model: HookedSAETransformer,
    layer: int,
) -> Float[Tensor, "... seq d_model"]:
    """Map vectors through frozen attention: LN → W_V → frozen_patterns → W_O → post-norm.

    This is where cross-position information flow happens: a vector at position p
    gets redistributed across all positions according to the frozen attention patterns.
    Includes the post-attention sandwich norm (ln1_post) used by Gemma 3.

    Args:
        resid_pre: Residual stream vectors before this attention layer.
        cache: ActivationCache with frozen attention patterns and LN scales.
        model: The transformer model (for W_V, W_O weight matrices).
        layer: Which attention layer to map through.
    """
    raise NotImplementedError()


def map_through_mlp(
    resid_mid: Float[Tensor, "... seq d_model"],
    cache: ActivationCache,
    model: HookedSAETransformer,
    transcoders: dict[int, "Transcoder"],
    layer: int,
) -> Float[Tensor, "... seq d_model"]:
    """Map vectors through frozen MLP skip connection: LN → W_skip → post-norm.

    Includes the post-MLP sandwich norm (ln2_post) used by Gemma 3.

    Args:
        resid_mid: Residual stream vectors after attention (before MLP) at this layer.
        cache: ActivationCache with frozen LN scales.
        model: The transformer model (for LN weight parameters).
        transcoders: Dict mapping layer -> transcoder (for W_skip).
        layer: Which MLP layer to map through.
    """
    raise NotImplementedError()

In [ ]:
tests.test_map_through_ln(map_through_ln, gemma, cache_bonus)
tests.test_map_through_attn(map_through_attn, gemma, cache_bonus)
tests.test_map_through_mlp(map_through_mlp, gemma, cache_bonus, transcoders)

<details><summary>솔루션</summary>

```python
def map_through_ln(
    x: Float[Tensor, "... seq d_model"],
    cache: ActivationCache,
    model: HookedSAETransformer,
    layer: int | None,
    is_mlp_ln: bool = False,
) -> Float[Tensor, "... seq d_model"]:
    """Apply frozen RMSNorm: output = x / cached_scale * weight.

    Args:
        x: Input vectors to normalise.
        cache: ActivationCache with frozen scale values.
        model: The transformer model (for LN weight parameters).
        layer: Layer index, or None for the final LayerNorm.
        is_mlp_ln: If True, use ln2 (pre-MLP); otherwise use ln1 (pre-attention).
    """
    if layer is None:
        scale = cache["ln_final.hook_scale"]
        weight = model.ln_final.w
    elif is_mlp_ln:
        scale = cache[f"blocks.{layer}.ln2.hook_scale"]
        weight = model.blocks[layer].ln2.w
    else:
        scale = cache[f"blocks.{layer}.ln1.hook_scale"]
        weight = model.blocks[layer].ln1.w
    return x / scale * weight


def map_through_attn(
    resid_pre: Float[Tensor, "... seq d_model"],
    cache: ActivationCache,
    model: HookedSAETransformer,
    layer: int,
) -> Float[Tensor, "... seq d_model"]:
    """Map vectors through frozen attention: LN → W_V → frozen_patterns → W_O → post-norm.

    This is where cross-position information flow happens: a vector at position p
    gets redistributed across all positions according to the frozen attention patterns.
    Includes the post-attention sandwich norm (ln1_post) used by Gemma 3.

    Args:
        resid_pre: Residual stream vectors before this attention layer.
        cache: ActivationCache with frozen attention patterns and LN scales.
        model: The transformer model (for W_V, W_O weight matrices).
        layer: Which attention layer to map through.
    """
    x = map_through_ln(resid_pre, cache, model, layer=layer)

    W_V = model.blocks[layer].attn.W_V  # (n_heads, d_model, d_head)
    v = einops.einsum(x, W_V, "... src d_model, n_heads d_model d_head -> ... src n_heads d_head")

    patterns = cache[f"blocks.{layer}.attn.hook_pattern"]  # (1, n_heads, dest, src)
    z = einops.einsum(v, patterns[0], "... src n_heads d_head, n_heads dest src -> ... dest n_heads d_head")

    W_O = model.blocks[layer].attn.W_O  # (n_heads, d_head, d_model)
    result = einops.einsum(z, W_O, "... dest n_heads d_head, n_heads d_head d_model -> ... dest d_model")

    # Post-attention sandwich norm (Gemma 3)
    if hasattr(model.blocks[layer], "ln1_post"):
        post_scale = cache[f"blocks.{layer}.ln1_post.hook_scale"]
        post_w = model.blocks[layer].ln1_post.w
        result = result / post_scale * post_w

    return result


def map_through_mlp(
    resid_mid: Float[Tensor, "... seq d_model"],
    cache: ActivationCache,
    model: HookedSAETransformer,
    transcoders: dict[int, "Transcoder"],
    layer: int,
) -> Float[Tensor, "... seq d_model"]:
    """Map vectors through frozen MLP skip connection: LN → W_skip → post-norm.

    Includes the post-MLP sandwich norm (ln2_post) used by Gemma 3.

    Args:
        resid_mid: Residual stream vectors after attention (before MLP) at this layer.
        cache: ActivationCache with frozen LN scales.
        model: The transformer model (for LN weight parameters).
        transcoders: Dict mapping layer -> transcoder (for W_skip).
        layer: Which MLP layer to map through.
    """
    x = map_through_ln(resid_mid, cache, model, layer=layer, is_mlp_ln=True)
    result = x @ transcoders[layer].W_skip

    # Post-MLP sandwich norm (Gemma 3)
    if hasattr(model.blocks[layer], "ln2_post"):
        post_scale = cache[f"blocks.{layer}.ln2_post.hook_scale"]
        post_w = model.blocks[layer].ln2_post.w
        result = result / post_scale * post_w

    return result
```
</details>

### 연습 문제 - `compute_adjacency_matrix_manual` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 20-25 minutes on this exercise.
> ```

이제 이러한 helper 함수들을 사용하여 명시적인 forward-tracing을 통해 전체 adjacency matrix를 계산하십시오. 각 (source layer, target layer) 쌍에 대해 다음 과정을 수행합니다: (1) 각 source node의 writing vector를 해당 sequence position에 배치하여 `(n_source_nodes, seq_len, d_model)` shape의 tensor를 생성합니다. (2) 각 layer에서 attention output (cross-position flow)과 MLP skip output (within-position linear transform)을 residual stream에 더함으로써 중간 layer들을 통해 매핑합니다. (3) target layer를 처리합니다 (latent/error target의 경우: target layer의 attention을 더해 `resid_mid`을 얻은 후, pre-MLP LayerNorm `ln2`을 적용합니다. logit target의 경우: 최종 LayerNorm `ln_final`을 적용합니다). (4) 각 target node position에서 매핑된 벡터와 target의 reading vector를 dot product 합니다.

주의사항: `node_range_dict`은 layer key (`"E"`, `0`, `1`, ..., `"L"`)를 node list의 `(start_idx, end_idx)` 범위로 매핑합니다. Embedding node (`"E"`)는 source로만 사용되며, logit node (`"L"`)는 target으로만 사용됩니다. layer `l`에 있는 source node는 `resid_post[l]` (MLP 이후)에 write 하므로, 중간 layer는 `l+1`부터 시작합니다. Embedding source는 `resid_pre[0]`에 write 하므로, 이들의 중간 layer는 `0`부터 시작합니다.

<details><summary>힌트 - 루프 구조</summary>

```python
for src_key in layer_keys:
    if src_key == "L" or src_key not in graph.node_range_dict:
        continue
    src_start, src_end = graph.node_range_dict[src_key]
    src_writes_after = -1 if src_key == "E" else src_key  # numeric layer

    # Initialise: place writing vectors at source positions
    vecs = t.zeros(n_src, seq_len, d_model, device=device)
    for i in range(n_src):
        node = graph.nodes[src_start + i]
        vecs[i, node.ctx_idx] = graph.writing_vecs[src_start + i]

    for tgt_key in layer_keys:
        if tgt_key == "E" or tgt_key not in graph.node_range_dict:
            continue
        tgt_layer_num = n_layers if tgt_key == "L" else tgt_key
        if src_writes_after >= tgt_layer_num:
            continue

        mapped = vecs.clone()
        # ... map through intermediate layers, handle target, compute edges
```

</details>

In [ ]:
def compute_adjacency_matrix_manual(
    model: HookedSAETransformer,
    cache: ActivationCache,
    graph: GraphNodes,
    transcoders: dict[int, "Transcoder"],
) -> Float[Tensor, "n_nodes n_nodes"]:
    """Compute the attribution adjacency matrix using explicit forward-tracing.

    For each (source_layer, target_layer) pair, traces the source writing vectors through
    all intermediate frozen layers and computes dot products with target reading vectors.

    Args:
        model: The transformer model.
        cache: ActivationCache with frozen values (from FreezeHooks.cache_frozen_values).
        graph: GraphNodes containing all node metadata, writing_vecs, reading_vecs.
        transcoders: Dict mapping layer -> transcoder (for W_skip in MLP skip connections).

    Returns:
        Adjacency matrix of shape (n_nodes, n_nodes), where A[target, source] is the edge weight.
    """
    n_nodes = len(graph.nodes)
    seq_len = graph.seq_len
    n_layers = graph.n_layers
    d_model = model.cfg.d_model
    device = graph.writing_vecs.device
    adjacency = t.zeros(n_nodes, n_nodes, device=device)

    layer_keys = ["E"] + list(range(n_layers)) + ["L"]

    # For each source layer:
    #   For each target layer (that comes after the source):
    #     (1) Map source writing vectors through intermediate layers using map_through_attn and map_through_mlp
    #     (2) Handle the target layer (attention + LN for latents, final LN for logits)
    #     (3) Compute dot products with target reading vectors at their positions
    raise NotImplementedError()

    return adjacency

In [ ]:
manual_adj = compute_adjacency_matrix_manual(gemma, cache_bonus, result.graph, transcoders)
tests.test_compute_adjacency_matrix_manual(manual_adj, result.adjacency_matrix)

<details><summary>솔루션</summary>

```python
def compute_adjacency_matrix_manual(
    model: HookedSAETransformer,
    cache: ActivationCache,
    graph: GraphNodes,
    transcoders: dict[int, "Transcoder"],
) -> Float[Tensor, "n_nodes n_nodes"]:
    """Compute the attribution adjacency matrix using explicit forward-tracing.

    For each (source_layer, target_layer) pair, traces the source writing vectors through
    all intermediate frozen layers and computes dot products with target reading vectors.

    Args:
        model: The transformer model.
        cache: ActivationCache with frozen values (from FreezeHooks.cache_frozen_values).
        graph: GraphNodes containing all node metadata, writing_vecs, reading_vecs.
        transcoders: Dict mapping layer -> transcoder (for W_skip in MLP skip connections).

    Returns:
        Adjacency matrix of shape (n_nodes, n_nodes), where A[target, source] is the edge weight.
    """
    n_nodes = len(graph.nodes)
    seq_len = graph.seq_len
    n_layers = graph.n_layers
    d_model = model.cfg.d_model
    device = graph.writing_vecs.device
    adjacency = t.zeros(n_nodes, n_nodes, device=device)

    layer_keys = ["E"] + list(range(n_layers)) + ["L"]

    for src_key in layer_keys:
        if src_key == "L" or src_key not in graph.node_range_dict:
            continue
        src_start, src_end = graph.node_range_dict[src_key]
        n_src = src_end - src_start
        if n_src == 0:
            continue

        # Numeric layer after which source writes (-1 for embeddings = before layer 0)
        src_writes_after = -1 if src_key == "E" else src_key

        # Initialise: place each source node's writing vector at its sequence position
        vecs = t.zeros(n_src, seq_len, d_model, device=device)
        for i in range(n_src):
            node = graph.nodes[src_start + i]
            vecs[i, node.ctx_idx] = graph.writing_vecs[src_start + i]

        for tgt_key in layer_keys:
            if tgt_key == "E" or tgt_key not in graph.node_range_dict:
                continue
            tgt_start, tgt_end = graph.node_range_dict[tgt_key]
            if tgt_end == tgt_start:
                continue

            tgt_layer_num = n_layers if tgt_key == "L" else tgt_key
            if src_writes_after >= tgt_layer_num:
                continue  # Source must come strictly before target

            # Map through intermediate layers (full attn + MLP skip at each)
            mapped = vecs.clone()
            end_layer = n_layers if tgt_key == "L" else tgt_layer_num
            for layer in range(src_writes_after + 1, end_layer):
                mapped = mapped + map_through_attn(mapped, cache, model, layer)
                mapped = mapped + map_through_mlp(mapped, cache, model, transcoders, layer)

            # Handle target layer
            if tgt_key == "L":
                # Logit targets: apply final LayerNorm
                mapped = map_through_ln(mapped, cache, model, layer=None)
            else:
                # Latent/error targets: attention at target layer, then pre-MLP LN
                mapped = mapped + map_through_attn(mapped, cache, model, tgt_layer_num)
                mapped = map_through_ln(mapped, cache, model, tgt_layer_num, is_mlp_ln=True)

            # Compute edge weights via dot products at target positions
            for j in range(tgt_end - tgt_start):
                tgt_node = graph.nodes[tgt_start + j]
                tgt_reading = graph.reading_vecs[tgt_start + j]
                edges = einops.einsum(
                    mapped[:, tgt_node.ctx_idx],
                    tgt_reading,
                    "n_src d_model, d_model -> n_src",
                )
                adjacency[tgt_start + j, src_start:src_end] = edges

    return adjacency
```
</details>

## AutoInterp

Attribution graph는 어떤 feature가 중요하고 그것들이 어떻게 연결되는지에 대한 그림을 제공하지만, 그 feature들이 *무엇*을 나타내는지는 알려주지 않습니다. 그래프가 진정으로 해석 가능해지려면, 각 노드에 대해 사람이 읽을 수 있는 레이블이 필요합니다. 여기서 자동화된 해석 가능성(automated interpretability, autointerp)이 필요합니다.

**섹션 1.3.3 (Automated Interpretability)**의 연습 문제를 완료하셨다면, 여기에 적용할 수 있는 전체 autointerp 구현체를 이미 가지고 계실 것입니다. 해당 섹션에서는 latent에 대해 max-activating example을 수집하고, 이를 LLM에 전달하여 자연어 설명을 생성하며, 해당 설명들의 점수를 매기는 전체 파이프라인을 다룹니다. 이 모든 도구들은 attribution graph 노드에 레이블을 지정하는 작업에 그대로 적용됩니다.

autointerp를 attribution graph와 결합했을 때의 핵심 장점은 **비용**입니다. 전체 SAE/transcoder에는 수천 또는 수백만 개의 latent가 있는 반면, pruning된 attribution graph에는 보통 10-50개의 활성화된 latent 노드만 포함됩니다. 이는 전체 latent 세트에 대해서는 비용이 너무 많이 들어 불가능했을 고품질 설명 생성(예: latent당 많은 예시를 사용하는 대형 모델 활용)을 그래프의 모든 노드에 대해 수행할 수 있음을 의미합니다.

구체적인 접근 방법은 다음과 같습니다:

1. pruning된 그래프에서 상위 k개의 latent 노드(influence score가 가장 높은 노드들)를 식별합니다. `result.graph` 객체는 어떤 latent가 유지되었는지와 그 activation 값들을 저장합니다. 다음과 같이 노드들을 반복 처리할 수 있습니다:

```python
for node in result.graph.nodes:
    if node.node_type == NodeType.LATENT:
        layer, ctx_idx, feature = node.layer, node.ctx_idx, node.feature
        # ... fetch or generate explanations for this latent
```

2. 각 latent에 대한 설명을 가져오거나 생성합니다. 만약 Neuronpedia에 사용 중인 SAE release에 대한 설명이 있다면, 1.3.3의 `get_autointerp_df`를 사용하여 가져옵니다. 그렇지 않다면, `fetch_max_activating_examples`으로 max-activating example을 수집하고 LLM 기반 설명 파이프라인을 실행합니다. 노드 수가 매우 적기 때문에 매우 철저하게 수행할 여유가 있습니다.

3. 시각화 시 각 노드의 레이블에 설명 텍스트를 추가하여 attribution graph에 주석을 답니다.

이 과정은 구조화된 연습 문제라기보다 개방형 탐색으로 남겨두었습니다. 구현 방법이 1.3.3에서 autointerp 코드를 어떻게 구성했는지, 그리고 사용 중인 transcoder에 대해 Neuronpedia가 설명을 제공하는지에 따라 달라지기 때문입니다. 좋은 시작 방법은 섹션 4의 circuit 중 하나(예: Dallas circuit)를 선택하여 가장 영향력 있는 상위 5개 latent 노드에 대한 설명을 생성하는 것입니다.